# Synaptic Bouton Analysis Pipeline


This notebook walks through the post-processing workflow used to characterize glutamatergic parallel fiber boutons.
Each chapter tracks a distinct analytical theme, from data curation to mutant comparisons and stability assays.


## Chapter A – Data Foundations

Chapter A assembles the datasets, cleans fluorescence traces, and prepares pooled tables that will feed every later analysis.


### A Prelude – Toolkit Orientation

The opening steps prepare the computational environment and shared constants that support every subsequent analysis task in this notebook.


### A.1 Library Imports

This cell assembles the analytical toolbox needed for the bouton study. Core scientific libraries such as NumPy, pandas, SciPy, and scikit-learn support numerical modeling, clustering, and statistical testing, while Matplotlib and Seaborn provide the visualization backbone. By centralizing these imports we ensure every downstream analysis stage—trace preprocessing, dimensionality reduction, and classification—can access the same well-defined computational environment.


In [ ]:
## Standard library imports. PCA, Clustering, Ellipse/boundary, Alpha-shape, k-NN boundary, Stability/plasticity

# Standard library imports
import json
import os
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

# Scientific computing and data analysis
import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage, set_link_color_palette
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Data visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from matplotlib.colors import ListedColormap, to_hex
from matplotlib.cm import Set1
from matplotlib.patches import Ellipse
from matplotlib.widgets import Button

# Geometric and statistical analysis tools
import alphashape
from shapely.geometry import MultiPolygon, Point, Polygon as ShapelyPolygon
from shapely.affinity import scale as shp_scale
from shapely.prepared import prep
from statannotations.Annotator import Annotator

### A.2 Data Source Configuration

Here we define the file system layout for the complete dataset, including the Excel workbooks that store PCA features, per-trial failure metrics, and target cell annotations. Establishing these paths ensures that subsequent routines retrieve raw traces and metadata consistently, keeping the analysis reproducible regardless of where the notebook is executed.


In [ ]:
## File paths and directory structure. Failures/reliability, PCA, Clustering, Extracellular Ca²⁺, Stability/plasticity, Temporal traces

# File paths and directory structure
BASE_DIR            = Path('E:\Seafile\Stability_After_temp_t_delete_later')
PPR_FILENAME        = 'summary.xlsx'           # PCA features file
PPR_TRIALS_FILENAME = 'summary_trials.xlsx'  # Per-trial failure data
TARGET_MAP_FILENAME = 'Target_WT_pooled.xlsx'  # Bouton target identity mapping
OUTPUT_DIR          = BASE_DIR / 'output'

# Data filtering and analysis parameters
EXCEPTIONAL_CONDITIONS = ['Stability_After_05', 'Stability_Before_05', 'Theo_1_5Ca', 'Theo_4Ca', 'WT_Theo', 'Theo_1_5_50Hz', 'Theo_4_50Hz', 'Theo_2_5_50Hz']  # Conditions to exclude from PCA and clustering
PCA_DROP_COLS          = [f'AMP{i}' for i in range(3, 11)] + ['measurement', 'Condition', 'ID', 'Target', '%Fail3']
N_CLUSTERS             = 5                          # Number of clusters for analysis

# Trace processing parameters
STIM_SHIFT  = 0.5                        # Stimulus time offset (seconds)
CROP_END    = 2.0                          # Trace duration to keep (seconds)
SAMPLE_RATE = 1000                      # Target sampling rate (Hz)
N_SAMPLES   = int(CROP_END * SAMPLE_RATE) + 1
COMMON_TIME = np.linspace(0, CROP_END, N_SAMPLES)  # Standardized time vector

### A.3 Output Logistics and Helpers

We next organize the raw recordings and metadata that feed the bouton analysis pipeline, ensuring every condition is accounted for before processing.

This block creates the output directory structure and introduces helper utilities for filtering valid trace files. By curating which spreadsheets qualify as bouton recordings, we avoid ingesting metadata or temporary files and maintain a clean provenance for every trace that enters the processing workflow.


In [ ]:
# Create output directory and define helper functions
OUTPUT_DIR.mkdir(exist_ok=True)

def is_bouton_file(file_path: Path) -> bool:
    """Check if Excel file contains bouton trace data (excludes metadata files)"""
    metadata_files = {PPR_FILENAME.lower(), PPR_TRIALS_FILENAME.lower(), TARGET_MAP_FILENAME.lower()}
    return (file_path.suffix.lower() == '.xlsx' and 
            not file_path.name.startswith('~$') and 
            file_path.name.lower() not in metadata_files)

def clean_bouton_id(raw_bouton_id: str) -> str:
    """Remove file suffixes from bouton IDs for consistent matching"""
    return raw_bouton_id.replace('_traces_converted', '')

### A.4 Experimental Inventory

The loop catalogues each experimental condition present in the raw data repository and enumerates the bouton trace files found within. This systematic survey provides immediate feedback on data availability and builds the foundation for condition-specific preprocessing that follows.


In [ ]:
## Discover experimental conditions and load bouton trace files. Ellipse/boundary, Temporal traces

experimental_conditions = [dir_path.name for dir_path in sorted(BASE_DIR.iterdir()) if dir_path.is_dir()]
raw_traces_data         = []

for condition_name in experimental_conditions:
    condition_dir = BASE_DIR / condition_name
    bouton_files  = [file_path for file_path in condition_dir.glob('*.xlsx') if is_bouton_file(file_path)]
    print(f"Processing {condition_name}: {len(bouton_files)} files")
    
    for xlsx_file in sorted(bouton_files):
        bouton_id  = clean_bouton_id(xlsx_file.stem)
        trace_data = pd.read_excel(xlsx_file).apply(pd.to_numeric, errors='coerce')
        
        if trace_data.shape[1] >= 2:  # Need at least time and average columns
            time_column    = trace_data.columns[-1]     # Time is last column
            average_column = trace_data.columns[-2]  # Average trace is second-to-last
            
            raw_traces_data.append({
                'ID': bouton_id,
                'Condition': condition_name,
                'Time': trace_data[time_column].tolist(),
                'Avg': trace_data[average_column].tolist()
            })

RAW_TRACES_DF = pd.DataFrame(raw_traces_data)

### A.5 Feature Matrix Assembly

This cell loads the multi-sheet Excel workbook containing precomputed bouton features and aligns them with the discovered experimental conditions. By harmonizing the feature matrices across conditions, we prepare a coherent dataset that can support pooled analyses as well as condition-specific comparisons.


In [ ]:
## Load PCA features from multi-sheet Excel file. AMP1/strength, PCA, Feature correlations

# Load PCA features from multi-sheet Excel file
pca_features_file = BASE_DIR / PPR_FILENAME
excel_data        = pd.ExcelFile(pca_features_file)

# Only process conditions that have corresponding feature sheets
available_conditions = [cond for cond in experimental_conditions if cond in excel_data.sheet_names]
CONDITIONS           = available_conditions

feature_dataframes = []
for condition_name in CONDITIONS:
    condition_features = pd.read_excel(pca_features_file, sheet_name=condition_name)
    
    # Standardize ID column name (handle various naming conventions)
    id_column_names = ['id', 'bouton', 'bouton_id', 'name']
    for column in condition_features.columns:
        if str(column).strip().lower() in id_column_names:
            condition_features = condition_features.rename(columns={column: 'ID'})
            break
    
    # Clean bouton IDs and add condition label
    condition_features['ID'] = condition_features['ID'].apply(
        lambda x: clean_bouton_id(str(x)) if pd.notnull(x) else x
    )
    condition_features['Condition'] = condition_name
    feature_dataframes.append(condition_features)

excel_data.close()
FEATURES_DATAFRAME = pd.concat(feature_dataframes, ignore_index=True)

# Remove specified amplitude columns from analysis
amplitude_columns_to_drop = [col for col in PCA_DROP_COLS[:8] if col in FEATURES_DATAFRAME.columns]
if amplitude_columns_to_drop:
    FEATURES_DATAFRAME = FEATURES_DATAFRAME.drop(columns=amplitude_columns_to_drop)

### A.6 Target Identity Integration

Target identity information (Purkinje cell, interneuron, or unclassified) is imported and standardized in this step. Cleaning identifiers and storing them as categorical labels allows later projections and clustering analyses to be biologically interpretable, connecting statistical patterns back to synaptic targets.


In [ ]:
## Load bouton target identity mapping (PC/IN/UN classification). Extracellular Ca²⁺, Temporal traces

# Load bouton target identity mapping (PC/IN/UN classification)
target_mapping_file          = BASE_DIR / TARGET_MAP_FILENAME
target_identity_data         = pd.read_excel(target_mapping_file).iloc[:, :2].copy()
target_identity_data.columns = ['ID', 'Target']

# Clean and standardize target data
target_identity_data['ID']     = target_identity_data['ID'].apply(lambda x: clean_bouton_id(str(x).strip()))
target_identity_data['Target'] = target_identity_data['Target'].astype(str).str.strip().str.upper()

# Set invalid targets to 'UN' (undefined)
valid_targets = ['PC', 'IN', 'UN']
target_identity_data['Target'] = target_identity_data['Target'].where(
    target_identity_data['Target'].isin(valid_targets), 'UN'
)

# Merge target identities into feature data
FEATURES_DATAFRAME['ID']     = FEATURES_DATAFRAME['ID'].astype(str).str.strip()
FEATURES_DATAFRAME           = FEATURES_DATAFRAME.merge(target_identity_data, on='ID', how='left')
FEATURES_DATAFRAME['Target'] = FEATURES_DATAFRAME['Target'].fillna('UN')  # Missing targets → undefined

# Data loading summary
print(f"\n=== DATA LOADING SUMMARY ===")
print(f"Conditions: {len(CONDITIONS)} ({', '.join(CONDITIONS)})")
print(f"Traces: {len(RAW_TRACES_DF)} | Features: {len(FEATURES_DATAFRAME)} | Target mappings: {len(target_identity_data)}")
print(f"Target distribution: {dict(FEATURES_DATAFRAME['Target'].value_counts())}")
print(f"✓ Successfully processed {len(raw_traces_data)} bouton files")

### A.7 Photobleaching and Normalization Utilities

Before modeling, we align, pad, and normalize fluorescence traces so that recordings collected with different acquisition schemes can be compared on equal footing.

This section defines the signal-processing toolkit for fluorescence traces. Functions handle photobleaching correction, baseline stabilization, exponential fitting, and smoothing, ensuring that raw optical signals are transformed into comparable, biologically meaningful time courses before higher-level analysis.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def correct_photobleaching(
    fluorescence_signal: np.ndarray,
    time_points: np.ndarray,
    *,
    stim_window       =(1.0, 1.7),
    ref_points        =20,
    return_info: bool = False,
    plot: bool        = False
) -> np.ndarray | tuple[np.ndarray, dict]:
    """
    Photobleaching correction with bi-exponential fitting.
    Always applies correction.
    """
    y = fluorescence_signal.copy()
    t = time_points
    
    valid = ~np.isnan(y)
    if valid.sum() < 10:
        return (y, {}) if return_info else y

    # Baseline is 10th percentile of ALL valid data
    baseline = np.nanpercentile(y[valid], 10)
    
    # Get pre-stim data (all non-NaN before stimulus window)
    stim_start    = stim_window[0] 
    pre_stim_mask = (t < stim_start) & valid
    pre_stim_idx  = np.where(pre_stim_mask)[0]
    
    if len(pre_stim_idx) < 5:
        return (y, {"applied": False}) if return_info else y
        
    # Get end points (last ref_points)
    valid_idx = np.where(valid)[0]
    end_idx   = valid_idx[-ref_points:] if len(valid_idx) >= ref_points else valid_idx[-5:]
    
    # Decide whether to use end points
    pre_median = np.nanmedian(y[pre_stim_idx])
    end_median = np.nanmedian(y[end_idx]) 
    use_end    = end_median < pre_median
    
    # Build fitting data
    if use_end:
        fit_idx = np.concatenate([pre_stim_idx, end_idx])
    else:
        fit_idx = pre_stim_idx
        
    fit_idx   = np.unique(fit_idx)
    t_fit     = t[fit_idx]
    y_fit_raw = y[fit_idx]
    y_fit     = y_fit_raw - baseline
    
    # Bi-exponential decay: A1*exp(k1*t) + A2*exp(k2*t)
    def biexp_decay(t, A1, k1, A2, k2):
        return A1 * np.exp(k1 * t) + A2 * np.exp(k2 * t)
    
    try:
        # Initial guess for bi-exponential
        A_total = np.max(y_fit) if np.max(y_fit) > 0 else 1.0
        A1_guess = A_total * 0.7
        A2_guess = A_total * 0.3
        k1_guess = -0.1  # Fast component
        k2_guess = -0.01  # Slow component
        
        # Fit bi-exponential with constraints
        popt, pcov = curve_fit(
            biexp_decay, 
            t_fit, 
            y_fit,
            p0     =[A1_guess, k1_guess, A2_guess, k2_guess],
            bounds =([0.01, -10, 0.01, -10], [1000, 0.1, 1000, 0.1]),
            maxfev =3000
        )
        A1_fit, k1_fit, A2_fit, k2_fit = popt
        
        # Check fit quality
        y_pred    = biexp_decay(t_fit, A1_fit, k1_fit, A2_fit, k2_fit)
        ss_res    = np.sum((y_fit - y_pred)**2)
        ss_tot    = np.sum((y_fit - np.mean(y_fit))**2)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
    except Exception:
        # Fallback to single exponential
        def exp_decay(t, A, k):
            return A * np.exp(k * t)
            
        A_guess = np.max(y_fit) if np.max(y_fit) > 0 else 1.0
        k_guess = -0.01
        
        try:
            popt, pcov = curve_fit(
                exp_decay, 
                t_fit, 
                y_fit,
                p0     =[A_guess, k_guess],
                bounds =([0.1, -10], [1000, 0.1]),
                maxfev =2000
            )
            A_fit, k_fit = popt
            
            # Convert to bi-exp format for consistency
            A1_fit, k1_fit, A2_fit, k2_fit = A_fit, k_fit, 0, 0
            y_pred                         = exp_decay(t_fit, A_fit, k_fit)
            ss_res                         = np.sum((y_fit - y_pred)**2)
            ss_tot                         = np.sum((y_fit - np.mean(y_fit))**2)
            r_squared                      = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
            
        except Exception:
            return (y, {"applied": False, "error": "Both fits failed"}) if return_info else y
    
    # Always apply correction - subtraction method
    stim_start_time = stim_window[0]
    decay_at_stim   = biexp_decay(stim_start_time, A1_fit, k1_fit, A2_fit, k2_fit)
    decay_curve     = biexp_decay(t, A1_fit, k1_fit, A2_fit, k2_fit)
    
    y_corrected     = y.copy() 

    # Subtraction correction: subtract decay trend, normalized to stimulus start
    y_corrected[valid] = y[valid] - (decay_curve[valid] - decay_at_stim)
    
    # Gain correction (commented out):
    
    if plot:
        fig, ax = plt.subplots(1, 1, figsize=(8, 4))
        
        # Original trace in red (background)
        ax.plot(t, y, 'r-', alpha=0.7, linewidth=0.5, label='Original')
        
        # Corrected trace in black
        ax.plot(t, y_corrected, 'k-', linewidth=0.5, label='Corrected')
        
        # Fit of original trace in blue
        t_plot     = t[valid]
        y_fit_plot = baseline + biexp_decay(t_plot, A1_fit, k1_fit, A2_fit, k2_fit)
        ax.plot(t_plot, y_fit_plot, 'b-', linewidth=1, label=f'Bi-exp fit (R²={r_squared:.3f})')
        
        ax.fill_betweenx(ax.get_ylim(), stim_window[0], stim_window[1], 
                        alpha=0.2, color='yellow')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Fluorescence')
        ax.legend(fontsize=8)
        
        plt.tight_layout()
        plt.show()
    
    info = {
        "applied": True, "A1": A1_fit, "k1": k1_fit, "A2": A2_fit, "k2": k2_fit,
        "baseline": baseline, "use_end": use_end, "r_squared": r_squared, 
        "n_fit_points": len(fit_idx)
    }
    return (y_corrected, info) if return_info else y_corrected

def process_single_trace(trace_data) -> dict:
    """
    Process individual bouton trace: time alignment, resampling, bleaching correction, ΔF/F normalization.
    Handles NaN values throughout the pipeline.
    """
    original_time   = np.array(trace_data['Time'])
    original_signal = np.array(trace_data['Avg'])
    condition_name  = trace_data['Condition']
    
    # Remove any infinite values and replace with NaN
    original_signal = np.where(np.isfinite(original_signal), original_signal, np.nan)
    
    # Apply time shift for specific experimental conditions
    if condition_name in EXCEPTIONAL_CONDITIONS:
        adjusted_time = original_time + STIM_SHIFT
        
        # Pad signal with NaN values if time shift creates gap at beginning
        points_before_start = np.sum(COMMON_TIME < adjusted_time[0])
        if points_before_start > 0:
            padded_signal = np.concatenate([np.full(points_before_start, np.nan), original_signal])
            adjusted_time = np.concatenate([COMMON_TIME[:points_before_start], adjusted_time])
        else:
            padded_signal = original_signal
    else:
        adjusted_time = original_time
        padded_signal = original_signal
    
    # Resample to common time base using linear interpolation
    # Only interpolate where we have valid data
    valid_mask = np.isfinite(padded_signal) & np.isfinite(adjusted_time)
    
    if np.sum(valid_mask) < 2:  # Need at least 2 points for interpolation
        resampled_signal = np.full(N_SAMPLES, np.nan)
    else:
        try:
            interpolator = interp1d(adjusted_time[valid_mask], padded_signal[valid_mask], 
                                  kind='linear', bounds_error=False, fill_value=np.nan)
            resampled_signal = interpolator(COMMON_TIME)
        except ValueError:
            # If interpolation fails, fill with NaN
            resampled_signal = np.full(N_SAMPLES, np.nan)
    
    # Ensure exact length (crop or pad to N_SAMPLES)
    if len(resampled_signal) < N_SAMPLES:
        padding_needed   = N_SAMPLES - len(resampled_signal)
        resampled_signal = np.concatenate([resampled_signal, np.full(padding_needed, np.nan)])
    else:
        resampled_signal = resampled_signal[:N_SAMPLES]
    
    # Apply photobleaching correction (now handles NaN values)
    bleach_corrected_signal = correct_photobleaching(resampled_signal, COMMON_TIME)
    
    # Calculate ΔF/F using pre-stimulus baseline (0-1s)
    baseline_period_mask  = (COMMON_TIME >= 0) & (COMMON_TIME < 1.0)
    baseline_fluorescence = np.nanmean(bleach_corrected_signal[baseline_period_mask])
    
    if not np.isnan(baseline_fluorescence) and baseline_fluorescence != 0:
        delta_f_over_f = (bleach_corrected_signal - baseline_fluorescence) / baseline_fluorescence
    else:
        delta_f_over_f = np.full_like(bleach_corrected_signal, np.nan)
    
    return {
        'ID': trace_data['ID'],
        'Condition': condition_name,
        'Time': COMMON_TIME,
        'Avg': delta_f_over_f
    }

### A.8 Trace Processing Pipeline

With the utilities in place, this code iterates through all boutons, applies bleaching correction, normalizes baselines, and stores metadata describing each trace. The result is a harmonized collection of time series that captures synaptic responses across experimental conditions while preserving the contextual information needed for downstream grouping.


In [ ]:
## Process all bouton traces and organize by condition. Temporal traces

all_processed_traces = []
traces_by_condition  = {}

for condition_name in RAW_TRACES_DF['Condition'].unique():
    condition_traces = RAW_TRACES_DF[RAW_TRACES_DF['Condition'] == condition_name]
    
    condition_all_processed_traces = []
    for _, single_trace in condition_traces.iterrows():
        processed_trace = process_single_trace(single_trace)
        all_processed_traces.append(processed_trace)
        condition_all_processed_traces.append(processed_trace['Avg'])
    
    traces_by_condition[condition_name] = condition_all_processed_traces
    print(f"Processed {len(condition_all_processed_traces)} traces for condition: {condition_name}")

print(f"✓ Processed {len(all_processed_traces)} traces across {len(traces_by_condition)} conditions")

### A.9 Condition-Level Trace Visualization

Processed traces are visualized for every condition to sanity-check preprocessing outcomes. Overlaying individual responses reveals whether normalization, alignment, and denoising preserved the stereotyped waveform dynamics expected for each experimental group.


In [ ]:
## Plot processed traces by condition. Extracellular Ca²⁺, Temporal traces

n_conditions = len(traces_by_condition)
fig, axes    = plt.subplots(2, 5, figsize=(20, 8))
axes         = axes.flatten()

for plot_idx, (condition_name, condition_traces) in enumerate(traces_by_condition.items()):
    if plot_idx >= len(axes):
        break
        
    ax = axes[plot_idx]
    
    # Plot individual traces (transparent gray)
    for single_trace in condition_traces:
        ax.plot(COMMON_TIME, single_trace, alpha=0.2, color='gray', linewidth=0.5)
    
    # Plot condition average (bold black)
    condition_average = np.nanmean(condition_traces, axis=0)
    ax.plot(COMMON_TIME, condition_average, color='black', linewidth=2, label='Average')
    
    # Format subplot
    ax.set_title(f'{condition_name}\n(n={len(condition_traces)})', fontsize=10, pad=10)
    ax.set_xlim(0, CROP_END)
    ax.axhline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)  # Zero line
    ax.axvline(1.0, color='red', linestyle='--', alpha=0.5, linewidth=1)  # Stimulus onset
    
    # Add axis labels for edge subplots
    if plot_idx >= 5:  # Bottom row
        ax.set_xlabel('Time (s)')
    if plot_idx % 5 == 0:  # Leftmost column
        ax.set_ylabel('ΔF/F')

# Remove unused subplots
for empty_idx in range(n_conditions, len(axes)):
    fig.delaxes(axes[empty_idx])

plt.tight_layout()
plt.suptitle('Processed Bouton Traces: Aligned, Resampled, and Normalized by Condition', 
             y=1.02, fontsize=14, fontweight='bold')
plt.show()

# Create final processed traces dataframe
NORM_TRACES_DATAFRAME = pd.DataFrame(all_processed_traces)

print(f"Time range: {COMMON_TIME[0]:.2f} - {COMMON_TIME[-1]:.2f}s ({len(COMMON_TIME)} points)")
print(f"Exceptional conditions (shifted by {STIM_SHIFT}s): {', '.join(EXCEPTIONAL_CONDITIONS)}")

### A.10 Condition Pooling

The final phase of Chapter A consolidates cleaned traces and associated metadata into pooled tables that downstream statistical models can consume.

Here we aggregate related experimental conditions into pooled datasets that reflect meaningful biological groupings. This pooling step increases statistical power for later PCA and clustering stages while retaining the ability to trace results back to the original acquisition cohorts.


In [ ]:
## Pool related experimental conditions for analysis. Extracellular Ca²⁺, Stability/plasticity, Synapsin-II / genotype

CONDITION_POOLS = {
    'WT_pooled': ['WT_Theo', 'WT_Anthime', 'WT_Theo_1scd'],
    'stability_before': ['Stability_Before', 'Stability_Before_05'],
    'stability_after': ['Stability_After', 'Stability_After_05']
}

# Add pooled conditions to features dataframe
existing_conditions = set(FEATURES_DATAFRAME['Condition'].unique())
for pool_name, source_conditions in CONDITION_POOLS.items():
    if pool_name not in existing_conditions:
        available_sources = [c for c in source_conditions if c in existing_conditions]
        if available_sources:
            pooled_data              = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'].isin(available_sources)].copy()
            pooled_data['Condition'] = pool_name
            FEATURES_DATAFRAME       = pd.concat([FEATURES_DATAFRAME, pooled_data], ignore_index=True)

# Extract condition-specific dataframes (keeping original names for downstream compatibility)
PCA_Data_WT_Pooled        = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'WT_pooled'].copy()
PCA_Data_WT_Theo          = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'WT_Theo'].copy()
PCA_Data_WT_Anthime       = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'WT_Anthime'].copy()
PCA_Data_SynII            = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'SynII'].copy()
PCA_Data_WT_Low_Ca        = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'Theo_1_5Ca'].copy()
PCA_Data_WT_High_Ca       = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'Theo_4Ca'].copy()
PCA_Data_Stability_Before = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'stability_before'].copy()
PCA_Data_Stability_After  = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'stability_after'].copy()
PCA_Data_50Hz_1_5_Ca      = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'Theo_1_5_50Hz'].copy()
PCA_Data_50Hz_4_Ca        = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'Theo_4_50Hz'].copy()
PCA_Data_50Hz_2_5_Ca      = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'Theo_2_5_50Hz'].copy()    

print(f"✓ {len(FEATURES_DATAFRAME)} boutons across {len(FEATURES_DATAFRAME['Condition'].unique())} conditions")
print(f"Key datasets: WT_pooled({len(PCA_Data_WT_Pooled)}), SynII({len(PCA_Data_SynII)}), 1.5Ca({len(PCA_Data_WT_Low_Ca)}), 4Ca({len(PCA_Data_WT_High_Ca)})")

The PCA will use only the columns AMP1, AMP2, all PPRs and %Fail1 and 2. All the other columns will be un-selected.

## Chapter B – Baseline Comparisons

Chapter B evaluates whether foundational amplitude and plasticity metrics align across experimental cohorts before dimensionality reduction.


### B Prelude – Baseline Comparability Questions

Chapter B interrogates whether foundational amplitude and plasticity measurements agree across cohorts prior to dimensionality reduction.


#### B Focus – Distribution Diagnostics

We begin by contrasting amplitude, failure, and variability distributions between experimental groups to flag any gross disparities.


### B.1 Distribution Comparison Setup

To test whether key synaptic parameters are comparable across conditions, we assemble a dictionary of metrics—amplitudes, failure rates, and plasticity indices—for visualization. Organizing the data in this way supports consistent histograms and boxplots for each feature.


In [ ]:
# Parameter comparison setup: histograms and boxplots
comparison_datasets = {
    'WT_Theo': PCA_Data_WT_Theo, 
    'WT_Anthime': PCA_Data_WT_Anthime, 
    'SynII': PCA_Data_SynII
}

dataset_colors = ['#1f77b4', '#2ca02c', '#ff7f0e']
color_palette = dict(zip(comparison_datasets.keys(), dataset_colors))

# Parameters to analyze and their reference values
analysis_parameters = ['PPR2/1', 'AMP1', 'AMP2', 'STD_baseline', '%Fail1']
reference_values    = {'PPR2/1': 1.0}  # Theoretical no-facilitation line

# Statistical comparisons to perform
pairwise_comparisons = [("WT_Theo", "SynII"), ("WT_Theo", "WT_Anthime"), ("WT_Anthime", "SynII")]

### B.2 Amplitude and Plasticity Diagnostics

Using the prepared datasets, this cell renders paired histograms and boxplots that contrast amplitude, paired-pulse ratios, and failure fractions across groups. The accompanying statistical annotations help determine whether experimental cohorts can be legitimately compared or require condition-specific treatment.


In [ ]:
# Create parameter comparison plots with statistical analysis (excluding STD_baseline)
parameters_to_plot = [p for p in analysis_parameters if p != 'STD_baseline']
n_params           = len(parameters_to_plot)

fig, axes = plt.subplots(n_params, 2, figsize=(15, 4 * n_params))
# ensure axes is 2D for consistent indexing when n_params == 1
if n_params == 1:
    axes = axes.reshape(1, 2)

statistical_results = []

for param_idx, parameter_name in enumerate(parameters_to_plot):
    # Extract valid data for each dataset
    parameter_data = {}
    for dataset_name, dataset_df in comparison_datasets.items():
        if parameter_name in dataset_df.columns:
            clean_data = dataset_df[parameter_name].dropna()
            if len(clean_data) > 0:
                parameter_data[dataset_name] = clean_data

    # Skip if insufficient data
    if len(parameter_data) < 2 or sum(len(data) for data in parameter_data.values()) < 10:
        statistical_results.append(f"[SKIP] {parameter_name}: insufficient data")
        axes[param_idx, 0].text(0.5, 0.5, f'No data for {parameter_name}',
                               ha='center', va='center', transform=axes[param_idx, 0].transAxes)
        axes[param_idx, 1].axis('off')
        continue

    # Create histogram (left panel)
    ax_histogram         = axes[param_idx, 0]
    all_parameter_values = np.concatenate([data.values for data in parameter_data.values()])
    histogram_bins       = np.linspace(all_parameter_values.min(), all_parameter_values.max(), 21)

    for dataset_name, dataset_values in parameter_data.items():
        transparency = 0.35 if dataset_name == 'SynII' else 0.65
        ax_histogram.hist(dataset_values, bins=histogram_bins, alpha=transparency,
                         label=dataset_name, color=color_palette[dataset_name],
                         density=True, edgecolor='black')

    ax_histogram.set_xlabel(parameter_name)
    ax_histogram.set_ylabel('Probability Density')
    ax_histogram.set_title(f'{parameter_name} Distribution')
    ax_histogram.legend(fontsize=8)

    # Add reference line if specified
    if parameter_name in reference_values:
        ax_histogram.axvline(reference_values[parameter_name], color='gray', linestyle='--', alpha=0.7)

    # Create boxplot with individual points (right panel)
    combined_data_list = []
    for dataset_name, dataset_values in parameter_data.items():
        combined_data_list.append(pd.DataFrame({
            parameter_name: dataset_values,
            'Condition': dataset_name
        }))
    combined_parameter_df = pd.concat(combined_data_list, ignore_index=True)

    ax_boxplot = axes[param_idx, 1]
    sns.boxplot(data=combined_parameter_df, x='Condition', y=parameter_name,
            ax=ax_boxplot, hue='Condition', palette=color_palette, legend=False)
    sns.stripplot(data=combined_parameter_df, x='Condition', y=parameter_name,
                 ax=ax_boxplot, color='black', size=3, alpha=0.6)

    ax_boxplot.set_title(f'{parameter_name} by Condition')
    ax_boxplot.tick_params(axis='x', rotation=30)

    # Add reference line to boxplot
    if parameter_name in reference_values:
        ax_boxplot.axhline(reference_values[parameter_name], color='gray', linestyle='--', alpha=0.7)

    # Statistical annotations and tests
    try:
        # Add pairwise comparison annotations
        stats_annotator = Annotator(ax_boxplot, pairwise_comparisons,
                                   data=combined_parameter_df, x='Condition', y=parameter_name)
        stats_annotator.configure(test='Mann-Whitney', text_format='star', loc='outside',
                                 comparisons_correction='bonferroni', show_test_name=False)
        stats_annotator.apply_and_annotate()

        # Overall group comparison (Kruskal-Wallis)
        group_data           = [parameter_data[name] for name in comparison_datasets.keys() if name in parameter_data]
        kruskal_h, kruskal_p = stats.kruskal(*group_data)
        statistical_results.append(f"{parameter_name}: Kruskal-Wallis H={kruskal_h:.3f}, p={kruskal_p:.4g}")

    except Exception as error:
        statistical_results.append(f"[ERROR] {parameter_name}: {error}")

plt.tight_layout()
plt.show()

# Save results
output_filename_base = OUTPUT_DIR / "parameter_comparison"
plt.savefig(f"{output_filename_base}.pdf", dpi=300, bbox_inches='tight')

# Save statistical summary
with open(f"{output_filename_base}_statistics.txt", "w") as stats_file:
    stats_file.write("Parameter Comparison Statistical Results\n")
    stats_file.write("=" * 50 + "\n\n")
    for result in statistical_results:
        stats_file.write(f"{result}\n")
        print(result)

print(f"✓ Saved analysis to {output_filename_base}")

### B.3 Paired-Pulse Response Profiles

Comparing mean PPR waveforms reveals whether facilitation and depression motifs are conserved or condition-specific before clustering.

Here we overlay the averaged PPR trajectories for each condition to inspect facilitation and depression patterns across ten stimulus pulses. This comparison highlights whether short-term plasticity motifs differ significantly between cohorts before entering dimensionality reduction.


In [ ]:
# PPR profile analysis: compare facilitation patterns across conditions

def extract_ppr_profile(condition_dataframe, max_pulse_number=10):
    """Extract PPR profile with means and standard errors for plotting."""
    # Find available PPR columns (PPR2/1, PPR3/1, etc.)
    ppr_column_names = [f'PPR{pulse_num}/1' for pulse_num in range(2, max_pulse_number+1) 
                        if f'PPR{pulse_num}/1' in condition_dataframe.columns]
    
    # PPR1/1 = 1.0 by definition, then calculate means for other ratios
    ppr_means           = [1.0] + condition_dataframe[ppr_column_names].mean().tolist()
    ppr_standard_errors = [0.0] + condition_dataframe[ppr_column_names].sem().tolist()
    total_pulses        = len(ppr_column_names) + 1
    
    return ppr_means, ppr_standard_errors, total_pulses

# Configure conditions for comparison
condition_configs = [
    ('WT_Theo', PCA_Data_WT_Theo, '#1f77b4', 'o'),        # Blue circles
    ('WT_Anthime', PCA_Data_WT_Anthime, '#2ca02c', 's'),  # Green squares  
    ('WT_pooled', PCA_Data_WT_Pooled, '#d62728', 'D'),    # Red diamonds
    ('SynII', PCA_Data_SynII, '#ff7f0e', '^')             # Orange triangles
]

plt.figure(figsize=(10, 6))

# Plot PPR profiles for each condition
for condition_name, condition_data, plot_color, marker_style in condition_configs:
    if condition_data.empty:
        continue
    
    ppr_means, ppr_errors, num_pulses = extract_ppr_profile(condition_data)
    pulse_numbers                     = list(range(1, num_pulses + 1))
    
    # Plot mean trajectory with error bands
    plt.plot(pulse_numbers, ppr_means, marker=marker_style, 
             label = f'{condition_name} (n={len(condition_data)})', 
             color = plot_color, linewidth=2, markersize=6)
    
    # Add standard error shading
    plt.fill_between(pulse_numbers, 
                     np.array(ppr_means) - np.array(ppr_errors),
                     np.array(ppr_means) + np.array(ppr_errors),
                     alpha=0.2, color=plot_color)

# Format plot
plt.axhline(1.0, color='gray', linestyle='--', alpha=0.7, linewidth=1, 
            label='No facilitation')
plt.xlabel('Pulse Number')
plt.ylabel('PPR (A_n/A_1)')
plt.title('Paired-Pulse Ratio Profiles Across Conditions')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "ppr_profiles_comparison.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics
print("PPR Profile Summary:")
print("-" * 50)
for condition_name, condition_data, _, _ in condition_configs:
    if condition_data.empty:
        continue
    profile_means, _, num_pulses = extract_ppr_profile(condition_data)
    
    summary_text = f"{condition_name:12} (n={len(condition_data):2d}): PPR2/1={profile_means[1]:.3f}"
    if len(profile_means) >= 10:  # Include PPR10/1 if available
        summary_text += f", PPR10/1={profile_means[9]:.3f}"
    print(summary_text)

print(f"✓ Saved PPR profile comparison to {output_file}")

## Chapter C – PCA Preparation

Chapter C standardizes features and constructs a unified PCA model that anchors all subsequent visualizations and statistics.


### C Prelude – Feature Preparation Agenda

With baseline comparability established, Chapter C standardizes feature matrices and builds the shared PCA model used throughout the analysis.


### C.1 Feature Sanitization for PCA

Before dimensionality reduction, we strip away non-numeric identifiers, harmonize column names, and scale each feature to unit variance. This normalization ensures that PCA captures true covariation among synaptic properties rather than artifacts of measurement scale.


In [ ]:
# Prepare datasets for PCA analysis by removing non-feature columns and scaling

def apply_standard_drops(dataframe_dict):
    """Apply standard column drops for PCA analysis to multiple dataframes."""
    return {name: df.drop(columns=[col for col in PCA_DROP_COLS if col in df.columns], errors='ignore') 
            for name, df in dataframe_dict.items() if df is not None and hasattr(df, 'columns')}

# Organize all condition dataframes
condition_dfs = {
    'PCA_Data_WT_Pooled'       : PCA_Data_WT_Pooled, 'PCA_Data_WT_Theo': PCA_Data_WT_Theo, 'PCA_Data_WT_Anthime': PCA_Data_WT_Anthime,
    'PCA_Data_SynII'           : PCA_Data_SynII, 'PCA_Data_WT_Low_Ca': PCA_Data_WT_Low_Ca, 'PCA_Data_WT_High_Ca': PCA_Data_WT_High_Ca,
    'PCA_Data_Stability_Before': PCA_Data_Stability_Before, 'PCA_Data_Stability_After': PCA_Data_Stability_After
}

# Prepare reference dataset for PCA (remove metadata columns)
WT_pooled_for_pca = PCA_Data_WT_Pooled.drop(columns=[col for col in PCA_DROP_COLS if col in PCA_Data_WT_Pooled.columns])

# Fit StandardScaler on WT_pooled reference dataset
scaler                   = StandardScaler()
scaled_data              = {}
scaled_data['WT_pooled'] = scaler.fit_transform(WT_pooled_for_pca)

# Transform all other datasets using same scaling parameters from WT_pooled
datasets_to_scale = {
    'WT_Theo'    : PCA_Data_WT_Theo, 'WT_Anthime': PCA_Data_WT_Anthime, 'SynII': PCA_Data_SynII,
    'WT_1_5Ca'   : PCA_Data_WT_Low_Ca, 'WT_4Ca': PCA_Data_WT_High_Ca,
    'stab_before': PCA_Data_Stability_Before, 'stab_after': PCA_Data_Stability_After,
    '50Hz_1_5Ca' : PCA_Data_50Hz_1_5_Ca, '50Hz_4Ca': PCA_Data_50Hz_4_Ca, '50Hz_2_5Ca': PCA_Data_50Hz_2_5_Ca
}

for dataset_name, dataset_df in datasets_to_scale.items():
    pca_features              = dataset_df.drop(columns=[col for col in PCA_DROP_COLS if col in dataset_df.columns])
    scaled_data[dataset_name] = scaler.transform(pca_features)

print(f"✓ Feature columns for PCA: {list(WT_pooled_for_pca.columns)}")
print(f"✓ Removed {len(PCA_DROP_COLS)} metadata columns from each dataset")  
print(f"✓ Scaled {len(scaled_data)} datasets using WT_pooled reference parameters")

### C.2 Reference PCA Model

The WT pooled dataset defines the PCA axes used throughout the study. Fitting the decomposition here and projecting every condition into the shared space creates a consistent coordinate system for cross-condition comparisons and clustering.


In [ ]:
# Fit PCA on WT_pooled reference dataset and transform all conditions

# Fit PCA model using WT_pooled as reference
pca                   = PCA(n_components=2)
pca_data              = {}
pca_data['WT_pooled'] = pca.fit_transform(scaled_data['WT_pooled'])

# Transform all other datasets using same PCA axes from WT_pooled
for dataset_name in datasets_to_scale.keys():
    pca_data[dataset_name] = pca.transform(scaled_data[dataset_name])

# Display PCA results summary
variance_pc1, variance_pc2 = pca.explained_variance_ratio_
print(f"✓ PCA transformation complete")
print(f"✓ PC1 explains {variance_pc1:.1%} of variance, PC2 explains {variance_pc2:.1%}")
print(f"✓ Total variance explained: {variance_pc1 + variance_pc2:.1%}")
print(f"✓ Transformed {len(pca_data)} datasets using WT_pooled PCA axes")

### C.3 PCA DataFrames

Projected coordinates are packaged into tidy DataFrames alongside metadata so that plotting and statistical routines can access principal components with clear provenance. This structure underpins every visualization and classification step built on top of the PCA embedding.


In [ ]:
# Convert PCA coordinates to DataFrames for analysis and plotting

# Create DataFrames for PCA-transformed coordinates
principal_component_columns = ['PC1', 'PC2']
pca_dfs                     = {}

for dataset_name, pca_coordinates in pca_data.items():
    pca_dfs[f'transformed_{dataset_name}'] = pd.DataFrame(pca_coordinates, columns=principal_component_columns)

# Create PCA components table showing feature contributions to each PC
feature_names = list(WT_pooled_for_pca.columns)
df_components = pd.DataFrame(pca.components_, columns=feature_names, index=['PC1', 'PC2'])

# Display top feature contributors for each principal component
print("PCA Components Analysis (top 3 contributors per PC):")
print("-" * 50)
for pc_name in ['PC1', 'PC2']:
    top_contributing_features = df_components.loc[pc_name].abs().nlargest(3)
    feature_contributions     = [f'{feature_name}({contribution:.3f})' 
                           for feature_name, contribution in top_contributing_features.items()]
    print(f"  {pc_name}: {', '.join(feature_contributions)}")

print(f"\n✓ Created {len(pca_dfs)} PCA coordinate DataFrames")
print(f"✓ PCA components table shape: {df_components.shape}")

### C.4 PCA–Feature Correlation Mapping

By concatenating PCA coordinates with the original feature measurements, we compute correlation coefficients that reveal which biophysical parameters drive each principal component. Identifying the strongest contributors translates abstract PCA axes back into mechanistic synaptic descriptors.


In [ ]:
# Combine PCA coordinates with original features for correlation analysis

# Create combined dataset: PCA coordinates + original features
combined_pca_FEATURES_DATAFRAME = pd.concat([
    pca_dfs['transformed_WT_pooled'].reset_index(drop=True),
    PCA_Data_WT_Pooled.reset_index(drop=True)
], axis=1)

# Calculate correlation matrix between principal components and original features
full_correlation_matrix = combined_pca_FEATURES_DATAFRAME.corr(numeric_only=True)
pc_feature_correlations = full_correlation_matrix.iloc[:2, 2:]  # Extract PC1,PC2 vs features

# Display strongest correlations for interpretability
print("Strongest PC-Feature Correlations:")
print("-" * 40)
for pc_name in ['PC1', 'PC2']:
    strongest_correlations = pc_feature_correlations.loc[pc_name].abs().nlargest(3)
    correlation_strings    = [f'{feature_name}({correlation_value:.3f})' 
                          for feature_name, correlation_value in strongest_correlations.items()]
    print(f"  {pc_name}: {', '.join(correlation_strings)}")

# Store comprehensive PCA results for downstream analysis
PCA_RESULTS = {
    'pca_model'         : pca,                           # Fitted PCA transformer
    'scaler'            : scaler,                        # Fitted StandardScaler
    'pca_dataframes'    : pca_dfs,                       # PCA coordinates for all datasets
    'components'        : df_components,                 # Feature contributions to PCs
    'correlations'      : pc_feature_correlations,       # PC-feature correlation matrix
    'explained_variance': pca.explained_variance_ratio_  # Variance explained by each PC
}

print(f"\n✓ Correlation analysis complete")
print(f"✓ PCA results stored in PCA_RESULTS dictionary with {len(PCA_RESULTS)} components")

## Chapter D – Clustering and Release Phenotypes

Chapter D leverages the PCA embedding to interrogate hierarchical clusters, relate them to biological targets, and describe release properties.


### D.1 Cluster Plot Utilities

We import specialized plotting utilities that annotate PCA scatter plots with explained variance and cluster information. These helpers streamline the visualization of complex clustering results throughout the rest of the chapter.


In [ ]:
# Perform hierarchical clustering on PCA-transformed WT_pooled data

# Extract PCA coordinates for clustering
pca_coordinates = pca_data['WT_pooled']  # Shape: (n_samples, 2)

# Perform hierarchical clustering using Ward linkage method
linkage_matrix      = linkage(pca_coordinates, method='ward')
cluster_assignments = fcluster(linkage_matrix, N_CLUSTERS, criterion='maxclust')

# Add cluster labels to original dataframe
PCA_Data_WT_Pooled_clustered               = PCA_Data_WT_Pooled.copy()
PCA_Data_WT_Pooled_clustered['HC_Cluster'] = cluster_assignments

# Visualize clusters in PCA space using plot_pca_nice
pc1_variance = PCA_RESULTS["explained_variance"][0]
pc2_variance = PCA_RESULTS["explained_variance"][1]

# Generate consistent Set1 colors for clusters
set1_colors          = Set1(np.linspace(0, 1, N_CLUSTERS))
cluster_rgba_colors  = [tuple(color) for color in set1_colors]
cluster_hex_colors   = [to_hex(color) for color in cluster_rgba_colors]
cluster_palette      = ListedColormap(cluster_rgba_colors, name='cluster_palette')
cluster_color_lookup = {cid: cluster_rgba_colors[cid - 1] for cid in range(1, N_CLUSTERS + 1)}

def get_cluster_color(cluster_id):
    cluster_id = int(cluster_id)
    return cluster_color_lookup[cluster_id]

def get_cluster_colors(labels):
    return [get_cluster_color(cid) for cid in labels]

def get_cluster_hex_color(cluster_id):
    cluster_id = int(cluster_id)
    return cluster_hex_colors[cluster_id - 1]

# Plot each cluster separately for legend
for cluster_id in range(1, N_CLUSTERS + 1):
    mask = cluster_assignments == cluster_id
    plt.scatter(pca_coordinates[mask, 0], pca_coordinates[mask, 1],
                c=[get_cluster_color(cluster_id)],
                s=50, marker='o', edgecolors='black', linewidths=0.6,
                label=f'Cluster {cluster_id}', alpha=0.9)

pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.xlim(-7, 10)
plt.ylim(-6, 6)
plt.title('Hierarchical Clustering in PCA Space (WT pooled)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Display cluster distribution statistics
print("Hierarchical Clustering Results:")
print("-" * 40)
total_samples = len(cluster_assignments)

for cluster_id in range(1, N_CLUSTERS + 1):
    cluster_size       = np.sum(cluster_assignments == cluster_id)
    cluster_percentage = (cluster_size / total_samples) * 100
    print(f"Cluster {cluster_id}: {cluster_size:2d} samples ({cluster_percentage:4.1f}%)")

print(f"Total: {total_samples} samples distributed across {N_CLUSTERS} clusters")
print("✓ Saved clustering visualization using consistent cluster colors")


### D.2 Hierarchical Dendrogram

Using the PCA coordinates, this code reconstructs the hierarchical clustering tree to visualize how boutons group across linkage distances. The dendrogram exposes nested relationships among boutons that complement the 2D PCA projection.


In [ ]:
# Create dendrogram visualization of hierarchical clustering

# Reuse consistent cluster colors for dendrogram branches
if 'cluster_hex_colors' not in globals():
    dendrogram_colormap = plt.get_cmap('Set1')
    cluster_hex_colors  = [to_hex(dendrogram_colormap(i)) for i in range(N_CLUSTERS)]
set_link_color_palette(cluster_hex_colors)

# Create sample labels for dendrogram leaves
try:
    sample_labels = PCA_Data_WT_Pooled.index.tolist()
except AttributeError:
    sample_labels = [f'Sample_{i+1}' for i in range(len(pca_coordinates))]

# Calculate clustering threshold for specified number of clusters
clustering_threshold = linkage_matrix[-N_CLUSTERS+1, 2]

# Generate dendrogram plot
plt.figure(figsize=(12, 6))
dendrogram(linkage_matrix,
          color_threshold=clustering_threshold,
          labels=sample_labels,
          leaf_rotation=90,
          leaf_font_size=8)

plt.title(f'Hierarchical Clustering Dendrogram (k={N_CLUSTERS})')
plt.xlabel('Samples')
plt.ylabel('Ward Distance')
plt.tight_layout()

# Save dendrogram
output_file = OUTPUT_DIR / "hierarchical_clustering_dendrogram.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Display clustering information
print(f"Clustering threshold for {N_CLUSTERS} clusters: {clustering_threshold:.3f}")
print(f"Dendrogram branches colored by cluster membership")
print(f"✓ Saved dendrogram to {output_file}")


### D.3 PCA Correlation Circle

A correlation circle is generated to display how each feature loads onto the first two principal components. This biplot view clarifies which synaptic properties pull samples along specific PCA axes and aids in interpreting cluster separation.


In [ ]:
# Create PCA correlation circle (biplot) showing feature contributions to principal components

# Extract correlation coefficients between PCs and original features
feature_pc_correlations = PCA_RESULTS['correlations'].T.values  # Shape: (n_features, 2)
scaling_factor          = 1.0
correlation_vectors     = feature_pc_correlations * scaling_factor

# Create correlation circle visualization
fig, ax = plt.subplots(figsize=(8, 8))

# Add reference elements: axes and unit circle
ax.axhline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)
ax.axvline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)
unit_circle = plt.Circle((0, 0), 1, color='black', fill=False, linestyle='-', alpha=0.5)
ax.add_patch(unit_circle)

# Plot correlation vectors for each feature
feature_names = list(WT_pooled_for_pca.columns)
for feature_idx, feature_name in enumerate(feature_names):
    pc1_correlation, pc2_correlation = correlation_vectors[feature_idx, 0], correlation_vectors[feature_idx, 1]
    
    # Draw correlation vector as arrow
    ax.arrow(0, 0, pc1_correlation, pc2_correlation, 
             color='darkred', alpha=0.8, 
             head_width=0.03, head_length=0.05, 
             length_includes_head=True, linewidth=1.5)
    
    # Position feature label outside the arrow tip
    label_x_position = pc1_correlation * 1.1
    label_y_position = pc2_correlation * 1.1
    ax.text(label_x_position, label_y_position, feature_name, 
            ha='center', va='center', fontsize=10, weight='bold', 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

# Format plot with variance information
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
total_variance             = pc1_variance + pc2_variance

ax.set_xlabel(f'PC1 ({pc1_variance:.1%} variance)')
ax.set_ylabel(f'PC2 ({pc2_variance:.1%} variance)')
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_aspect('equal')
ax.set_title(f'PCA Correlation Circle\n({total_variance:.1%} total variance explained)')
ax.grid(True, alpha=0.2)

plt.tight_layout()

# Save correlation circle
output_file = OUTPUT_DIR / "pca_correlation_circle.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Display strongest feature correlations for biological interpretation
print("Strongest Feature-PC Correlations:")
print("=" * 45)
pc_feature_correlations = PCA_RESULTS['correlations']

for pc_name in ['PC1', 'PC2']:
    strongest_features = pc_feature_correlations.loc[pc_name].abs().nlargest(5)
    print(f"\n{pc_name} (strongest contributors):")
    
    for feature_name, correlation_magnitude in strongest_features.items():
        correlation_value     = pc_feature_correlations.loc[pc_name, feature_name]
        correlation_direction = "+" if correlation_value > 0 else "-"
        print(f"  {correlation_direction} {feature_name}: {correlation_magnitude:.3f}")

print(f"\n✓ Saved correlation circle to {output_file}")

### D.4 WT Reference Check

To validate that Anthime's WT dataset aligns with the pooled WT reference, we overlay both cohorts in PCA space. This control confirms that lab-to-lab differences do not distort the shared coordinate system.


In [ ]:
# Compare WT Pooled and WT Anthime datasets in PCA space

# Extract PCA coordinates for comparison datasets
wt_pooled_coordinates  = np.asarray(pca_data['WT_pooled'])
wt_anthime_coordinates = np.asarray(pca_data['WT_Anthime'])

# Create PCA comparison plot
plt.figure(figsize=(8, 6))

# Plot WT pooled data with cluster colors
plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1],
           c=get_cluster_colors(cluster_assignments), alpha=0.4, s=30, label='WT pooled (2.5mM Ca)')


# Plot WT Anthime dataset  
plt.scatter(wt_anthime_coordinates[:, 0], wt_anthime_coordinates[:, 1], 
           marker='d', edgecolors='black', linewidths=0.6, s=50, c='magenta',
           label=f'WT Anthime (n={len(wt_anthime_coordinates)})')

# Format plot
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.xlim(-7, 10)
plt.ylim(-6, 6)
plt.title('PCA Projection: WT Pooled vs WT Anthime')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "pca_wt_pooled_vs_anthime.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ PCA comparison plot saved to {output_file}")
print(f"✓ WT Pooled: {len(wt_pooled_coordinates)} samples")
print(f"✓ WT Anthime: {len(wt_anthime_coordinates)} samples")

### D.5 Release Property Survey

Beyond traces, we examine amplitude and plasticity metrics across clusters to understand how release phenotypes differ among bouton classes.


#### D.5.a Cluster PPR Trajectories

Paired-pulse response curves are assembled per cluster to evaluate how facilitation or depression patterns differ among bouton classes. Linking temporal plasticity to cluster identity refines our interpretation of each group.


In [ ]:
# PPR profiles by hierarchical cluster
ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled_clustered.columns]
x_pulses = list(range(1, len(ppr_cols)+2))
clusters = sorted(PCA_Data_WT_Pooled_clustered['HC_Cluster'].unique())

plt.figure(figsize=(10, 6))
for cluster in clusters:
    cluster_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cluster]
    means        = [1] + cluster_data[ppr_cols].mean().tolist()
    sems         = [0] + cluster_data[ppr_cols].sem().tolist()
    color        = get_cluster_color(cluster)

    plt.plot(x_pulses, means, marker='o', label=f'Cluster {cluster} (n={len(cluster_data)})', color=color, linewidth=2)
    plt.fill_between(x_pulses, np.array(means)-np.array(sems), np.array(means)+np.array(sems), alpha=0.2, color=color)

plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
plt.ylabel('Mean PPR (A_n/A_1)')
plt.xlabel('Pulse Number')
plt.xticks(x_pulses)
plt.title('PPR Profiles by Hierarchical Cluster')
plt.legend(loc='upper right')
plt.tight_layout()

output_file = OUTPUT_DIR / "ppr_profiles_by_cluster.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()


#### D.5.b Cluster Metric Utility

Reusable helper functions are defined to produce boxplots and statistical annotations for any scalar feature across clusters. This modularity supports consistent reporting of amplitude and plasticity differences.


In [ ]:
# Reusable function for cluster boxplot analysis
def cluster_boxplot_analysis(data, column, title, output_prefix):
    'Create boxplot by cluster with statistical analysis.'
    from scipy.stats import kruskal, mannwhitneyu
    from itertools import combinations

    plt.figure(figsize=(8, 5))
    clusters = sorted(data['HC_Cluster'].unique())
    ax = sns.boxplot(x='HC_Cluster', y=column, hue='HC_Cluster', data=data, palette=cluster_hex_colors, legend=False)
    sns.stripplot(x='HC_Cluster', y=column, data=data, color='k', size=3, alpha=0.5, ax=ax)

    if 'PPR' in column:
        plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)

    plt.ylabel(column)
    plt.title(title)

    # Clean styling
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.get_xaxis().set_visible(False)

    # Create legend
    handles = [plt.Line2D([0], [0], color=get_cluster_color(cluster_id), lw=4) for cluster_id in clusters]
    labels = [f'Cluster {cluster_id}' for cluster_id in clusters]
    plt.legend(handles, labels, loc='upper right')

    # Statistical tests
    data_per_cluster = {c: data.loc[data['HC_Cluster'] == c, column].dropna().values for c in clusters}

    try:
        kw_stat, kw_p = kruskal(*data_per_cluster.values())
    except ValueError:
        kw_stat, kw_p = float('nan'), float('nan')

    pairs = list(combinations(clusters, 2))
    results = []
    for a, b in pairs:
        x, y = data_per_cluster[a], data_per_cluster[b]
        try:
            stat, p = mannwhitneyu(x, y, alternative='two-sided')
        except ValueError:
            stat, p = float('nan'), float('nan')
        results.append({
            'C_A': a, 'C_B': b, 'n_A': len(x), 'n_B': len(y),
            'U_stat': stat, 'p_raw': p, 'p_corr': min(p * len(pairs), 1.0) if not np.isnan(p) else np.nan
        })

    results.sort(key=lambda d: d['p_corr'] if not np.isnan(d['p_corr']) else 1)

    # Save results
    stats_file = OUTPUT_DIR / f"{output_prefix}_statistics.txt"
    with open(stats_file, "w") as f:
        f.write(f"{column} Statistical Analysis by Cluster")
        f.write(f"Kruskal-Wallis: H = {kw_stat:.4f}, p = {kw_p:.6g}")
        f.write(f"Bonferroni correction: {len(pairs)}")
        f.write("C_A	C_B	nA	nB	U_stat	p_raw	p_corr")
        for r in results:
            f.write(f"{r['C_A']}	{r['C_B']}	{r['n_A']}	{r['n_B']}	"
                    f"{r['U_stat']:.4f}	{r['p_raw']:.6g}	{r['p_corr']:.6g}")

    plt.tight_layout()
    output_file = OUTPUT_DIR / f"{output_prefix}.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✓ Saved {column} analysis to {output_file} and {stats_file}")


#### D.5.c Cluster-Level PPR2/1 Analysis

Applying the reusable plotting routine, we examine how the second pulse amplitude relative to the first varies across clusters. This metric probes whether early facilitation distinguishes bouton groups discovered by hierarchical clustering.


In [ ]:
# PPR2/1 analysis
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, 'PPR2/1', 'PPR2/1 by Cluster', 'ppr2_1_cluster')

# PPR3/1 analysis
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, 'PPR3/1', 'PPR3/1 by Cluster', 'ppr3_1_cluster')

# AMP1 analysis  
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, 'AMP1', 'AMP1 by Cluster', 'amp1_cluster')

# %Fail1 analysis
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, '%Fail1', '%Fail1 by Cluster', 'fail1_cluster')

## Chapter E - Diveristy along a fiber

Chapter E analyses the axonal organization, examining how bouton classes distribute along individual fibers.


### E.1 Fiber-Level Diversity

Boutons are regrouped by axonal fiber to quantify how many distinct clusters appear along individual fibers. Assessing intra-fiber heterogeneity sheds light on whether structural units host multiple functional bouton classes.


In [ ]:
# Analyze cluster diversity within individual fibers (boutons from same axon)

def extract_fiber_id(bouton_id):
    """Extract fiber ID from bouton ID (first 22 characters)."""
    return str(bouton_id)[:22]

def analyze_fiber_diversity(min_boutons_per_fiber=4):
    """Analyze how clusters are distributed within individual fibers."""
    
    # Extract fiber IDs and count boutons per fiber
    fiber_data             = PCA_Data_WT_Pooled_clustered.copy()
    fiber_data['Fiber_ID'] = fiber_data['ID'].apply(extract_fiber_id)
    
    # Count boutons per fiber
    fiber_bouton_counts = fiber_data.groupby('Fiber_ID').size()
    
    # Filter fibers with sufficient boutons
    valid_fibers  = fiber_bouton_counts[fiber_bouton_counts >= min_boutons_per_fiber].index
    filtered_data = fiber_data[fiber_data['Fiber_ID'].isin(valid_fibers)]
    
    print(f"Fiber analysis: {len(valid_fibers)} fibers with {min_boutons_per_fiber}+ boutons")
    print(f"Total boutons analyzed: {len(filtered_data)}")
    
    return filtered_data, valid_fibers

def calculate_cluster_diversity(filtered_data):
    """Calculate number of different clusters per fiber."""
    fiber_diversity = {}
    
    for fiber_id in filtered_data['Fiber_ID'].unique():
        fiber_boutons        = filtered_data[filtered_data['Fiber_ID'] == fiber_id]
        unique_clusters      = fiber_boutons['HC_Cluster'].nunique()
        total_boutons        = len(fiber_boutons)
        cluster_distribution = fiber_boutons['HC_Cluster'].value_counts(normalize=True)
        
        fiber_diversity[fiber_id] = {
            'num_cluster_types': unique_clusters,
            'total_boutons': total_boutons,
            'cluster_props': cluster_distribution.to_dict()
        }
    
    return fiber_diversity

# Main analysis
filtered_data, valid_fibers = analyze_fiber_diversity(min_boutons_per_fiber=4)
fiber_diversity             = calculate_cluster_diversity(filtered_data)

# Organize data by diversity level
diversity_categories = {'1': [], '2': [], '3': [], '4+': []}
for fiber_id, info in fiber_diversity.items():
    num_types = info['num_cluster_types']
    category  = str(num_types) if num_types <= 3 else '4+'
    diversity_categories[category].append(info)

# Calculate average cluster proportions for each diversity category
diversity_means  = {}
diversity_counts = {}

for category, fiber_list in diversity_categories.items():
    diversity_counts[category] = len(fiber_list)
    
    if len(fiber_list) > 0:
        # Calculate mean proportion for each cluster
        cluster_means = {}
        for cluster_id in range(1, N_CLUSTERS + 1):
            proportions               = [fiber['cluster_props'].get(cluster_id, 0) for fiber in fiber_list]
            cluster_means[cluster_id] = np.mean(proportions)
        diversity_means[category] = cluster_means
    else:
        diversity_means[category] = {i: 0 for i in range(1, N_CLUSTERS + 1)}

# Create stacked bar plot
fig, ax = plt.subplots(figsize=(10, 6))

diversity_order = ['1', '2', '3', '4+']
x_positions     = np.arange(len(diversity_order))
bottom_values   = np.zeros(len(diversity_order))

total_fibers    = len(valid_fibers)

# Plot stacked bars
for cluster_id in range(1, N_CLUSTERS + 1):
    cluster_heights = []
    for category in diversity_order:
        # Height = (fibers in category / total fibers) * 100 * average cluster proportion
        fiber_percentage   = (diversity_counts[category] / total_fibers) * 100
        cluster_proportion = diversity_means[category][cluster_id]
        height             = fiber_percentage * cluster_proportion
        cluster_heights.append(height)
    
    ax.bar(x_positions, cluster_heights, bottom=bottom_values, 
           color=get_cluster_color(cluster_id), label=f'Cluster {cluster_id}', alpha=0.8)
    bottom_values += cluster_heights

# Add fiber count labels
for i, category in enumerate(diversity_order):
    count      = diversity_counts[category]
    percentage = (count / total_fibers) * 100
    ax.text(i, percentage + 1, f'n={count}\n({percentage:.1f}%)', 
            ha='center', va='bottom', fontweight='bold', fontsize=9)

# Format plot
ax.set_xlabel('Number of Cluster Types per Fiber')
ax.set_ylabel('Percentage of Fibers (%)')
ax.set_title(f'Fiber Cluster Diversity (Fibers with 4+ Boutons)\nTotal: {total_fibers} fibers')
ax.set_xticks(x_positions)
ax.set_xticklabels(diversity_order)
ax.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)

plt.tight_layout()

output_file = OUTPUT_DIR / "fiber_cluster_diversity_analysis.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics
print(f"\n=== FIBER CLUSTER DIVERSITY SUMMARY ===")
for category in diversity_order:
    count = diversity_counts[category]
    if count > 0:
        percentage = (count / total_fibers) * 100
        print(f"\nFibers with {category} cluster type(s): {count} ({percentage:.1f}%)")
        
        # Show cluster composition
        for cluster_id in range(1, N_CLUSTERS + 1):
            prop = diversity_means[category][cluster_id]
            if prop > 0.05:  # Only show clusters with >5% average proportion
                print(f"  Cluster {cluster_id}: {prop:.1%} average proportion")

# Identify most diverse fibers
diverse_fibers = [fid for fid, info in fiber_diversity.items() if info['num_cluster_types'] >= 3]
if diverse_fibers:
    print(f"\nMost diverse fibers (3+ cluster types): {len(diverse_fibers)} fibers")
    for fiber_id in diverse_fibers[:5]:  # Show first 5
        info = fiber_diversity[fiber_id]
        clusters = list(info['cluster_props'].keys())
        print(f"  {fiber_id}: {info['num_cluster_types']} clusters ({clusters})")

print(f"\n✓ Saved fiber diversity analysis to {output_file}")


### E.2 Representative Fiber Dynamics

For a selected fiber, raw and smoothed traces are visualized to showcase how preprocessing captures the essential synaptic waveform while reducing noise. This example grounds the fiber-level analysis in concrete data.


In [ ]:
# Create trace lookup from resampled data
resampled_trace_lookup = {}
for _, trace_row in NORM_TRACES_DATAFRAME.iterrows():
    resampled_trace_lookup[trace_row['ID']] = {
        'Time': trace_row['Time'],
        'Avg': trace_row['Avg']
    }

# Analyze traces from a specific fiber (raw vs smoothed)
from scipy.signal import savgol_filter

def analyze_single_fiber(fiber_prefix, window_length=9, poly_order=2):
    """Analyze all boutons from a specific fiber."""
    
    # Find boutons from this fiber
    fiber_boutons = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['ID'].str.startswith(fiber_prefix)]
    
    if len(fiber_boutons) == 0:
        print(f"No boutons found with prefix '{fiber_prefix}'")
        return
    
    print(f"Found {len(fiber_boutons)} boutons from fiber '{fiber_prefix}'")
    
    # Get trace data for these boutons
    fiber_traces = {}
    for _, bouton in fiber_boutons.iterrows():
        bouton_id = bouton['ID']
        if bouton_id in resampled_trace_lookup:
            trace_data = resampled_trace_lookup[bouton_id]['Avg']
            cluster_id = bouton['HC_Cluster']
            fiber_traces[bouton_id] = {
                'raw': trace_data,
                'cluster': cluster_id
            }
    
    if not fiber_traces:
        print(f"No trace data found for fiber '{fiber_prefix}'")
        return
    
    # Apply smoothing
    for bouton_id in fiber_traces:
        raw_trace = fiber_traces[bouton_id]['raw']
        
        # Adjust window length if needed
        win_len = min(window_length, len(raw_trace))
        if win_len % 2 == 0:  # Must be odd
            win_len -= 1
        if win_len < 3:
            smoothed_trace = raw_trace.copy()
        else:
            smoothed_trace = savgol_filter(raw_trace, window_length=win_len, 
                                         polyorder=min(poly_order, win_len-1), mode='interp')
        
        fiber_traces[bouton_id]['smoothed'] = smoothed_trace
    
    return fiber_traces

def plot_fiber_traces(fiber_traces, fiber_prefix):
    """Plot raw vs smoothed traces for a fiber."""
    if not fiber_traces:
        return
    
    n_boutons = len(fiber_traces)
    
    # Create subplot layout (2 columns: raw, smoothed)
    fig, axes = plt.subplots(n_boutons, 2, figsize=(10, min(20, 2*n_boutons)), 
                            sharex=True, sharey=True)
    
    if n_boutons == 1:
        axes = axes.reshape(1, -1)
    
    # Calculate common y-limits
    all_values = []
    for data in fiber_traces.values():
        all_values.extend(data['raw'][np.isfinite(data['raw'])])
        all_values.extend(data['smoothed'][np.isfinite(data['smoothed'])])
    
    if all_values:
        y_min, y_max = np.min(all_values), np.max(all_values)
        y_padding = (y_max - y_min) * 0.05
        y_lims = (y_min - y_padding, y_max + y_padding)
    else:
        y_lims = (-0.5, 0.5)
    
    # Plot each bouton
    for idx, (bouton_id, data) in enumerate(fiber_traces.items()):
        cluster_id = data['cluster']
        color = get_cluster_color(cluster_id)
        
        # Raw trace (left)
        ax_raw = axes[idx, 0]
        ax_raw.plot(COMMON_TIME, data['raw'], color=color, linewidth=1.2)
        ax_raw.axhline(0, color='gray', linestyle='dotted', linewidth=0.7)
        ax_raw.axvline(1.0, color='red', linestyle='--', alpha=0.5)
        ax_raw.set_xlim(0.5, 2.0)
        ax_raw.set_ylim(y_lims)
        ax_raw.set_ylabel('ΔF/F', fontsize=8)
        ax_raw.set_title(f'{bouton_id} (Cluster {cluster_id})\nRaw', fontsize=9, loc='left')
        ax_raw.tick_params(labelsize=7)
        
        # Smoothed trace (right)
        ax_smooth = axes[idx, 1]
        ax_smooth.plot(COMMON_TIME, data['smoothed'], color=color, linewidth=1.2)
        ax_smooth.axhline(0, color='gray', linestyle='dotted', linewidth=0.7)
        ax_smooth.axvline(1.0, color='red', linestyle='--', alpha=0.5)
        ax_smooth.set_xlim(0.5, 2.0)
        ax_smooth.set_ylim(y_lims)
        ax_smooth.set_title('Savitzky-Golay Smoothed', fontsize=9, loc='left')
        ax_smooth.tick_params(labelsize=7)
    
    # Add x-axis labels to bottom row
    axes[-1, 0].set_xlabel('Time (s)', fontsize=8)
    axes[-1, 1].set_xlabel('Time (s)', fontsize=8)
    
    fig.suptitle(f'Single Fiber Analysis: {fiber_prefix} (n={n_boutons})', fontsize=12, fontweight='bold')
    plt.tight_layout()
    
    # Save figure
    safe_prefix = fiber_prefix.replace(':', '_').replace('/', '_')
    output_file = OUTPUT_DIR / f"fiber_traces_{safe_prefix}_raw_vs_smoothed.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return output_file

def plot_fiber_ppr_profiles(fiber_boutons, fiber_prefix):
    """Plot PPR profiles for boutons from the same fiber."""
    
    # Get PPR data
    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in fiber_boutons.columns]
    if not ppr_cols:
        print("No PPR columns found")
        return
    
    pulse_numbers = list(range(1, len(ppr_cols) + 2))  # Include pulse 1
    
    plt.figure(figsize=(8, 5))
    
    # Plot each bouton's PPR profile
    for _, bouton in fiber_boutons.iterrows():
        cluster_id = bouton['HC_Cluster']
        color = get_cluster_color(cluster_id)
        
        # Get PPR values (start with 1.0 for pulse 1)
        ppr_values = [1.0] + bouton[ppr_cols].tolist()
        
        plt.plot(pulse_numbers, ppr_values, color=color, alpha=0.8, linewidth=2, 
                marker='o', markersize=4, label=f'Cluster {cluster_id}')
    
    plt.axhline(1.0, color='gray', linestyle='--', linewidth=1)
    plt.xlabel('Pulse Number')
    plt.ylabel('PPR (A_n/A_1)')
    plt.title(f'PPR Profiles: {fiber_prefix} (n={len(fiber_boutons)})')
    plt.grid(True, alpha=0.3)
    plt.xticks(pulse_numbers)
    
    # Only show legend if multiple clusters
    if fiber_boutons['HC_Cluster'].nunique() > 1:
        plt.legend()
    
    plt.tight_layout()
    
    # Save figure
    safe_prefix = fiber_prefix.replace(':', '_').replace('/', '_')
    output_file = OUTPUT_DIR / f"fiber_ppr_{safe_prefix}_profiles.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return output_file

# Example usage - change the prefix to analyze different fibers
FIBER_PREFIX = "241212_Fibre2_PortionA_"  # Change this to your fiber of interest

# Run analysis
fiber_traces = analyze_single_fiber(FIBER_PREFIX)

if fiber_traces:
    # Plot traces
    trace_output = plot_fiber_traces(fiber_traces, FIBER_PREFIX)
    
    # Plot PPR profiles
    fiber_boutons = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['ID'].str.startswith(FIBER_PREFIX)]
    ppr_output    = plot_fiber_ppr_profiles(fiber_boutons, FIBER_PREFIX)
    
    print(f"✓ Fiber analysis complete:")
    print(f"  Traces: {trace_output}")
    print(f"  PPR profiles: {ppr_output}")
    
    # Summary statistics
    cluster_distribution = fiber_boutons['HC_Cluster'].value_counts().sort_index()
    print(f"\nFiber cluster composition:")
    for cluster_id, count in cluster_distribution.items():
        print(f"  Cluster {cluster_id}: {count} boutons")

else:
    print(f"No analysis possible for fiber '{FIBER_PREFIX}'")
    
    # Show available fiber prefixes
    available_prefixes = PCA_Data_WT_Pooled_clustered['ID'].str[:25].value_counts()
    print("\nAvailable fiber prefixes (showing top 10):")
    for prefix, count in available_prefixes.head(10).items():
        if count >= 3:  # Only show fibers with multiple boutons
            print(f"  '{prefix}': {count} boutons")


## Chapter F - Post-synaptic element identification

Chapter F associates datapoints to their target, either Purkinje, Interneurons or Undefined.

### F.1 Target Projection

By coloring the PCA scatter with Purkinje versus interneuron labels, we inspect whether synaptic target identity explains variance captured by the first components. This biological overlay links statistical clusters back to anatomical classes.


In [ ]:
# Visualize PCA space colored by target cell type (Purkinje Cells vs Interneurons)

# Extract target cell type labels from dataframe
target_cell_types = PCA_Data_WT_Pooled['Target'].values
pc_cell_mask      = target_cell_types == 'PC'  # Purkinje Cells
in_cell_mask      = target_cell_types == 'IN'  # Interneurons
un_cell_mask      = target_cell_types == 'UN'  # Undefined

plt.figure(figsize=(8, 6))

# Plot different target types with distinct markers and colors
if np.any(un_cell_mask):
    plt.scatter(pca_coordinates[un_cell_mask, 0], pca_coordinates[un_cell_mask, 1], 
               c='gray', marker='s', s=60, alpha=0.6,
               label=f'Undefined (n={np.sum(un_cell_mask)})')

if np.any(pc_cell_mask):
    plt.scatter(pca_coordinates[pc_cell_mask, 0], pca_coordinates[pc_cell_mask, 1], 
               c='red', edgecolors='black', linewidths=0.6, marker='o', s=100,
               label=f'Purkinje Cells (n={np.sum(pc_cell_mask)})')

if np.any(in_cell_mask):
    plt.scatter(pca_coordinates[in_cell_mask, 0], pca_coordinates[in_cell_mask, 1], 
               c='blue', edgecolors='black', linewidths=0.6, marker='^', s=100,
               label=f'Interneurons (n={np.sum(in_cell_mask)})')



# Format plot
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.xlim(-7, 10)
plt.ylim(-6, 6)
plt.title('PCA Space by Target Cell Type')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "pca_target_cell_types.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics
print("Target Cell Type Distribution:")
print("-" * 35)
print(f"Purkinje Cells (PC): {np.sum(pc_cell_mask):2d} samples")
print(f"Interneurons (IN):   {np.sum(in_cell_mask):2d} samples") 
print(f"Undefined (UN):      {np.sum(un_cell_mask):2d} samples")
print(f"Total:               {len(target_cell_types):2d} samples")
print(f"✓ Saved target visualization to {output_file}")

In [ ]:
# Scatter plot: WT pooled points with PC (red) and IN (blue) overlays
# Assumes pca_coordinates (np.ndarray), PCA_Data_WT_Pooled (DataFrame) and PCA_RESULTS are available

plt.figure(figsize=(9, 7))

# Plot original PCA
plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1],
            c=get_cluster_colors(cluster_assignments),
            s=50, marker='o', linewidths=0.6,
            label='WT pooled (2.5mM Ca)', alpha=0.4)


# Target masks
targets = PCA_Data_WT_Pooled['Target'].values
pc_mask = targets == 'PC'
in_mask = targets == 'IN'



# Plot Purkinje Cells (PC)
if np.any(pc_mask):
    plt.scatter(pca_coordinates[pc_mask, 0], pca_coordinates[pc_mask, 1],
                c='red', marker='o', s=100, 
                edgecolors='black', linewidths=0.6,
                label=f'Purkinje Cells (PC, n={np.sum(pc_mask)})')

# Plot Interneurons (IN)
if np.any(in_mask):
    plt.scatter(pca_coordinates[in_mask, 0], pca_coordinates[in_mask, 1],
                c='blue', marker='^', s=100, 
                edgecolors='black', linewidths=0.6,
                label=f'Interneurons (IN, n={np.sum(in_mask)})')


# Axis labels with explained variance
pc1_var, pc2_var = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_var:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_var:.1%} variance)')
plt.xlim(-7, 10)
plt.ylim(-6, 6)
plt.title('PCA: WT pooled colored by Target (PC vs IN)')
plt.legend(loc='best', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and show
output_path = OUTPUT_DIR / "pca_pc_in_scatter.pdf"
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved PCA PC/IN scatter to {output_path}")

### F.2 Target-Aligned Trace Summaries

Average traces for Purkinje and interneuron boutons are contrasted here with consistent color assignments. Visualizing response dynamics per target type reveals how physiology underpins spatial separation in PCA space.


In [ ]:
# Extract PC and IN data
pc_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['Target'] == 'PC'].copy()
in_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['Target'] == 'IN'].copy()

# Extract target coordinates and IDs
target_cell_types = PCA_Data_WT_Pooled['Target'].values
in_cell_mask      = target_cell_types == 'IN'
pc_cell_mask      = target_cell_types == 'PC'

in_coordinates = pca_coordinates[in_cell_mask]
in_ids         = PCA_Data_WT_Pooled.loc[in_cell_mask, 'ID'].tolist()

pc_coordinates = pca_coordinates[pc_cell_mask]
pc_ids         = PCA_Data_WT_Pooled.loc[pc_cell_mask, 'ID'].tolist()

# Ellipse tightness control (0.50 = loose, 0.95 = tight, 0.99 = very tight)
ELLIPSE_CONFIDENCE = 0.5

def fit_confidence_ellipse(points, confidence=0.95):
    """Fit confidence ellipse around points and return parameters."""
    from scipy.stats import chi2
    center = points.mean(axis=0)
    cov = np.cov(points.T)
    chi2_val = chi2.ppf(confidence, df=2)
    
    eigenvals, eigenvecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(eigenvecs[1, 0], eigenvecs[0, 0]))
    width, height = 2 * np.sqrt(chi2_val * eigenvals)
    cov_inv = np.linalg.inv(cov)
    
    return center, cov_inv, chi2_val, (width, height, angle)

# Fit ellipse around SynII points
synii_points              = pca_data['SynII']
ellipse_params            = fit_confidence_ellipse(synii_points, ELLIPSE_CONFIDENCE)
center, cov_inv, chi2_val = ellipse_params[:3]

def classify_points_in_ellipse(points, center, cov_inv, chi2_threshold):
    """Return boolean mask for points inside ellipse."""
    inside_mask = []
    for pt in points:
        diff          = pt - center
        mahal_dist_sq = diff @ cov_inv @ diff.T
        inside_mask.append(mahal_dist_sq <= chi2_threshold)
    return np.array(inside_mask)

def compare_traces_inside_outside_improved(target_ids, inside_mask, target_name, target_color):
    """Compare mean traces for inside vs outside ellipse groups with better visibility."""
    inside_ids  = [target_ids[i] for i in range(len(target_ids)) if inside_mask[i]]
    outside_ids = [target_ids[i] for i in range(len(target_ids)) if not inside_mask[i]]
    
    # Get traces for each group
    inside_traces  = [resampled_trace_lookup[bid]['Avg'] for bid in inside_ids if bid in resampled_trace_lookup]
    outside_traces = [resampled_trace_lookup[bid]['Avg'] for bid in outside_ids if bid in resampled_trace_lookup]
    synii_traces   = [resampled_trace_lookup[bid]['Avg'] for bid in PCA_Data_SynII['ID'] if bid in resampled_trace_lookup]
    
    if not (inside_traces and outside_traces and synii_traces):
        print(f"Insufficient trace data for {target_name} comparison")
        return
    
    # Calculate means and SEMs
    inside_mean = np.nanmean(inside_traces, axis=0)
    inside_sem  = np.nanstd(inside_traces, axis=0, ddof=1) / np.sqrt(len(inside_traces))
    
    outside_mean = np.nanmean(outside_traces, axis=0)
    outside_sem  = np.nanstd(outside_traces, axis=0, ddof=1) / np.sqrt(len(outside_traces))
    
    synii_mean   = np.nanmean(synii_traces, axis=0)
    synii_sem    = np.nanstd(synii_traces, axis=0, ddof=1) / np.sqrt(len(synii_traces))
    
    # Create 3-panel figure
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    
    # Panel 1: Inside ellipse with SEM
    inside_color = 'red' if target_name == 'IN' else 'green'
    ax1.plot(COMMON_TIME, inside_mean, color=inside_color, linewidth=1, 
             label=f'{target_name} inside ellipse (n={len(inside_traces)})')
    ax1.fill_between(COMMON_TIME, inside_mean-inside_sem, inside_mean+inside_sem, color=inside_color, alpha=0.3)
    ax1.axhline(0, color='black', linestyle='dotted', linewidth=1)
    ax1.axvline(1.0, color='black', linestyle='--', alpha=0.5)
    ax1.set_xlim(0.5, 2.0)
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('ΔF/F')
    ax1.set_title(f'{target_name} Inside Ellipse')
    ax1.legend()
    
    # Panel 2: Outside ellipse with SEM
    ax2.plot(COMMON_TIME, outside_mean, color='black', linewidth=1,
             label=f'{target_name} outside ellipse (n={len(outside_traces)})')
    ax2.fill_between(COMMON_TIME, outside_mean-outside_sem, outside_mean+outside_sem, color='black', alpha=0.3)
    ax2.axhline(0, color='black', linestyle='dotted', linewidth=1)
    ax2.axvline(1.0, color='black', linestyle='--', alpha=0.5)
    ax2.set_xlim(0.5, 2.0)
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('ΔF/F')
    ax2.set_title(f'{target_name} Outside Ellipse')
    ax2.legend()
    
    # Panel 3: Overlay comparison
    ax3.plot(COMMON_TIME, inside_mean, color=inside_color, linewidth=1, 
             label=f'Inside (n={len(inside_traces)})')
    ax3.plot(COMMON_TIME, outside_mean, color='black', linewidth=1,
             label=f'Outside (n={len(outside_traces)})')
    ax3.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax3.axvline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax3.set_xlim(0.5, 2.0)
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('ΔF/F')
    ax3.set_title(f'{target_name} Inside vs Outside')
    ax3.legend()
    
    # Remove top and right spines, remove grids
    for ax in [ax1, ax2, ax3]:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / f"{target_name.lower()}_inside_outside_ellipse_traces_improved.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ {target_name} ellipse analysis:")
    print(f"  Inside: {len(inside_traces)} traces ({len(inside_traces)/len(target_ids)*100:.1f}%)")
    print(f"  Outside: {len(outside_traces)} traces ({len(outside_traces)/len(target_ids)*100:.1f}%)")
    
    return output_file

# Run improved analysis for both cell types
if len(in_ids) > 0:
    in_inside_mask = classify_points_in_ellipse(in_coordinates, center, cov_inv, chi2_val)
    compare_traces_inside_outside_improved(in_ids, in_inside_mask, 'IN', 'red')

if len(pc_ids) > 0:
    pc_inside_mask = classify_points_in_ellipse(pc_coordinates, center, cov_inv, chi2_val)
    compare_traces_inside_outside_improved(pc_ids, pc_inside_mask, 'PC', 'mediumseagreen')

### F.3 Target Distance Metrics

Beyond visualization, we compute the median PCA distance between Purkinje and interneuron groups. Quantifying this separation gauges how distinctly the two synapse types occupy the reduced-dimensional manifold.


In [ ]:
# Enhanced PCA visualization with median distance between PC and IN groups

# Extract target cell type labels and coordinates
target_cell_types = PCA_Data_WT_Pooled['Target'].values
pc_cell_mask      = target_cell_types == 'PC'
in_cell_mask      = target_cell_types == 'IN'
un_cell_mask      = target_cell_types == 'UN'

# Calculate medians for PC and IN groups
pc_coordinates = pca_coordinates[pc_cell_mask]
in_coordinates = pca_coordinates[in_cell_mask]

pc_median = np.median(pc_coordinates, axis=0) if len(pc_coordinates) > 0 else None
in_median = np.median(in_coordinates, axis=0) if len(in_coordinates) > 0 else None

# Calculate distance between medians
if pc_median is not None and in_median is not None:
    median_distance = np.linalg.norm(pc_median - in_median)
else:
    median_distance = None

plt.figure(figsize=(10, 8))

# Plot individual data points
if np.any(un_cell_mask):
    plt.scatter(pca_coordinates[un_cell_mask, 0], pca_coordinates[un_cell_mask, 1], 
               c='gray', marker='s', s=60, alpha=0.6, edgecolors='black', linewidth=0.5,
               label=f'Undefined (n={np.sum(un_cell_mask)})')

if np.any(pc_cell_mask):
    plt.scatter(pca_coordinates[pc_cell_mask, 0], pca_coordinates[pc_cell_mask, 1], 
               c='red', marker='o', s=80, edgecolors='black', linewidths=0.6,
               label=f'Purkinje Cells (n={np.sum(pc_cell_mask)})')

if np.any(in_cell_mask):
    plt.scatter(pca_coordinates[in_cell_mask, 0], pca_coordinates[in_cell_mask, 1], 
               c='blue', marker='^', s=80, edgecolors='black', linewidths=0.6,
               label=f'Interneurons (n={np.sum(in_cell_mask)})')



# Plot medians and distance line
if pc_median is not None and in_median is not None:
    # Plot median points
    plt.scatter(pc_median[0], pc_median[1], c='darkred', marker='X', s=200, 
               label='PC Median', edgecolors='black', linewidth=2)
    plt.scatter(in_median[0], in_median[1], c='darkblue', marker='X', s=200, 
               label='IN Median', edgecolors='black', linewidth=2)
    
    # Draw line between medians
    plt.plot([pc_median[0], in_median[0]], [pc_median[1], in_median[1]], 
             color='black', linestyle='--', linewidth=2, alpha=0.8, 
             label=f'Median Distance: {median_distance:.3f}')
    
    # Annotate distance
    midpoint_x = (pc_median[0] + in_median[0]) / 2
    midpoint_y = (pc_median[1] + in_median[1]) / 2
    plt.annotate(f'd = {median_distance:.3f}', 
                xy=(midpoint_x, midpoint_y), 
                xytext=(midpoint_x + 0.5, midpoint_y + 0.5),
                fontsize=12, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Format plot
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.xlim(-7, 10)
plt.ylim(-6, 6)
plt.title('PCA Space: PC vs IN with Median Distance Analysis')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "pca_pc_in_median_distance.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics
print("=== PC vs IN SEPARATION ANALYSIS ===")
print(f"Purkinje Cells (PC): {np.sum(pc_cell_mask):2d} samples")
print(f"Interneurons (IN):   {np.sum(in_cell_mask):2d} samples")
print(f"Undefined (UN):      {np.sum(un_cell_mask):2d} samples")



if pc_median is not None and in_median is not None:
    print(f"PC median coordinates:     ({pc_median[0]:.3f}, {pc_median[1]:.3f})")
    print(f"IN median coordinates:     ({in_median[0]:.3f}, {in_median[1]:.3f})")
    print(f"Median-to-median distance: {median_distance:.3f}")
else:
    print("Cannot calculate median distance - insufficient data")

print(f"✓ Saved enhanced PCA plot to {output_file}")

### F.4 Cluster-Averaged Traces

Having mapped PCA space, we now translate cluster assignments back into temporal response motifs to interpret functional signatures.

Mean fluorescence traces are computed for each hierarchical cluster, with side-by-side comparisons for Purkinje and interneuron members. These profiles translate abstract clusters into recognizable temporal response motifs.


In [ ]:
# Generate average trace profiles by hierarchical cluster with PC/IN side-by-side comparison

# Check if required data is available
if 'PCA_Data_WT_Pooled_clustered' not in globals() or 'Target' not in PCA_Data_WT_Pooled_clustered.columns:
    print('[CLUSTER PROFILES][SKIP] Required clustered data or Target column missing.')
else:
    # Extract PC and IN data with cluster assignments
    pc_clustered_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['Target'] == 'PC'].copy()
    in_clustered_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['Target'] == 'IN'].copy()
    
    # Organize traces by cluster number for each target type
    pc_cluster_traces = {}
    in_cluster_traces = {}
    
    # Process PC traces by cluster
    for _, row in pc_clustered_data.iterrows():
        bouton_id = row['ID']
        cluster_id = row['HC_Cluster']
        
        if bouton_id in resampled_trace_lookup:
            trace_values = resampled_trace_lookup[bouton_id]['Avg']
            
            if cluster_id not in pc_cluster_traces:
                pc_cluster_traces[cluster_id] = []
            pc_cluster_traces[cluster_id].append(trace_values)
    
    # Process IN traces by cluster
    for _, row in in_clustered_data.iterrows():
        bouton_id = row['ID']
        cluster_id = row['HC_Cluster']
        
        if bouton_id in resampled_trace_lookup:
            trace_values = resampled_trace_lookup[bouton_id]['Avg']
            
            if cluster_id not in in_cluster_traces:
                in_cluster_traces[cluster_id] = []
            in_cluster_traces[cluster_id].append(trace_values)
    
    # Determine all cluster numbers present
    all_clusters = sorted(set(pc_cluster_traces.keys()).union(set(in_cluster_traces.keys())))
    
    if not all_clusters:
        print('[CLUSTER PROFILES][SKIP] No cluster data found.')
    else:
        # Calculate statistics for each cluster and target type
        def calculate_cluster_stats(trace_list):
            if len(trace_list) > 0:
                trace_matrix = np.column_stack(trace_list)
                mean_trace = np.nanmean(trace_matrix, axis=1)
                sem_trace = np.nanstd(trace_matrix, axis=1, ddof=1) / np.sqrt(trace_matrix.shape[1])
                return mean_trace, sem_trace, len(trace_list)
            return None, None, 0
        
        # Set up side-by-side subplot layout
        n_clusters = len(all_clusters)
        fig, axes = plt.subplots(n_clusters, 2, figsize=(16, 3*n_clusters), sharex=True, sharey=True)
        
        if n_clusters == 1:
            axes = axes.reshape(1, -1)
        
        # Calculate global y-limits for consistent scaling
        all_trace_values = []
        for cluster_id in all_clusters:
            if cluster_id in pc_cluster_traces:
                for trace in pc_cluster_traces[cluster_id]:
                    all_trace_values.extend(trace[np.isfinite(trace)])
            if cluster_id in in_cluster_traces:
                for trace in in_cluster_traces[cluster_id]:
                    all_trace_values.extend(trace[np.isfinite(trace)])
        
        if all_trace_values:
            y_padding = 0.1
            y_min = np.min(all_trace_values) - y_padding
            y_max = np.max(all_trace_values) + y_padding
        else:
            y_min, y_max = -0.5, 0.5
        
        # Plot each cluster row
        for row_idx, cluster_id in enumerate(all_clusters):
            # Left column: PC traces
            ax_pc = axes[row_idx, 0]
            if cluster_id in pc_cluster_traces:
                pc_mean, pc_sem, pc_count = calculate_cluster_stats(pc_cluster_traces[cluster_id])
                
                if pc_mean is not None:
                    valid_indices = np.isfinite(pc_mean) & np.isfinite(pc_sem)
                    
                    if np.any(valid_indices):
                        ax_pc.plot(COMMON_TIME[valid_indices], pc_mean[valid_indices], 
                                  color='red', linewidth=2.5, 
                                  label=f'PC Cluster {cluster_id} (n={pc_count})')
                        ax_pc.fill_between(COMMON_TIME[valid_indices], 
                                          (pc_mean - pc_sem)[valid_indices], 
                                          (pc_mean + pc_sem)[valid_indices], 
                                          color='red', alpha=0.2)
            
            ax_pc.set_title(f'PC Cluster {cluster_id}', fontweight='bold')
            ax_pc.set_ylabel('ΔF/F')
            ax_pc.axhline(0, color='gray', linestyle='dotted', linewidth=1, alpha=0.7)
            ax_pc.axvline(1.0, color='black', linestyle='--', alpha=0.5, linewidth=1)
            ax_pc.grid(True, alpha=0.3)
            ax_pc.set_ylim(y_min, y_max)
            
            # Add sample count annotation
            if cluster_id in pc_cluster_traces:
                ax_pc.text(0.02, 0.95, f'n={len(pc_cluster_traces[cluster_id])}', 
                          transform=ax_pc.transAxes, verticalalignment='top',
                          bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
            else:
                ax_pc.text(0.5, 0.5, 'No PC data', transform=ax_pc.transAxes, 
                          ha='center', va='center', style='italic', color='gray')
            
            # Right column: IN traces
            ax_in = axes[row_idx, 1]
            if cluster_id in in_cluster_traces:
                in_mean, in_sem, in_count = calculate_cluster_stats(in_cluster_traces[cluster_id])
                
                if in_mean is not None:
                    valid_indices = np.isfinite(in_mean) & np.isfinite(in_sem)
                    
                    if np.any(valid_indices):
                        ax_in.plot(COMMON_TIME[valid_indices], in_mean[valid_indices], 
                                  color='blue', linewidth=2.5, 
                                  label=f'IN Cluster {cluster_id} (n={in_count})')
                        ax_in.fill_between(COMMON_TIME[valid_indices], 
                                          (in_mean - in_sem)[valid_indices], 
                                          (in_mean + in_sem)[valid_indices], 
                                          color='blue', alpha=0.2)
            
            ax_in.set_title(f'IN Cluster {cluster_id}', fontweight='bold')
            ax_in.axhline(0, color='gray', linestyle='dotted', linewidth=1, alpha=0.7)
            ax_in.axvline(1.0, color='black', linestyle='--', alpha=0.5, linewidth=1)
            ax_in.grid(True, alpha=0.3)
            ax_in.set_ylim(y_min, y_max)
            
            # Add sample count annotation
            if cluster_id in in_cluster_traces:
                ax_in.text(0.02, 0.95, f'n={len(in_cluster_traces[cluster_id])}', 
                          transform=ax_in.transAxes, verticalalignment='top',
                          bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
            else:
                ax_in.text(0.5, 0.5, 'No IN data', transform=ax_in.transAxes, 
                          ha='center', va='center', style='italic', color='gray')
        
        # Format final plot
        for ax in axes[-1, :]:  # Bottom row x-axis labels
            ax.set_xlabel('Time (s)')
            ax.set_xlim(0, CROP_END)
        
        plt.tight_layout()
        
        # Save plot
        output_file = OUTPUT_DIR / "cluster_profiles_side_by_side.pdf"
        plt.savefig(output_file, dpi=300, bbox_inches='tight')
        plt.show()
        
        # Summary statistics
        print(f"=== CLUSTER PROFILE COMPARISON ===")
        print(f"Clusters analyzed: {', '.join(map(str, all_clusters))}")
        
        for cluster_id in all_clusters:
            pc_count = len(pc_cluster_traces.get(cluster_id, []))
            in_count = len(in_cluster_traces.get(cluster_id, []))
            print(f"Cluster {cluster_id}: PC={pc_count} traces, IN={in_count} traces")
        
        print(f"✓ Saved side-by-side cluster profiles to {output_file}")

### F.5 Cluster Composition Comparison

In [ ]:
# Compare cluster distributions between Purkinje Cells, Interneurons, and overall WT
target_cell_types = PCA_Data_WT_Pooled_clustered['Target'].values
pc_mask = target_cell_types == 'PC'
in_mask = target_cell_types == 'IN'

# Get cluster assignments for each target type
pc_cluster_assignments = cluster_assignments[pc_mask]
in_cluster_assignments = cluster_assignments[in_mask]
wt_cluster_assignments = cluster_assignments  # All WT pooled

# Count clusters for each group
pc_cluster_counts = pd.Series(pc_cluster_assignments).value_counts().sort_index()
in_cluster_counts = pd.Series(in_cluster_assignments).value_counts().sort_index()
wt_cluster_counts = pd.Series(wt_cluster_assignments).value_counts().sort_index()

# Ensure all clusters represented
all_clusters = sorted(range(1, N_CLUSTERS + 1))
pc_complete = pd.Series([pc_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)
in_complete = pd.Series([in_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)
wt_complete = pd.Series([wt_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)

# Calculate percentages
pc_percentages = 100 * pc_complete / len(pc_cluster_assignments)
in_percentages = 100 * in_complete / len(in_cluster_assignments)
wt_percentages = 100 * wt_complete / len(wt_cluster_assignments)

# Stacked bar plot with 3 bars
fig, ax = plt.subplots(figsize=(10, 6))
bar_width = 0.6

def get_text_color(rgb):
    r, g, b = rgb[:3]
    return 'white' if 0.2126*r + 0.7152*g + 0.0722*b < 0.55 else 'black'

bottom_pc = bottom_in = bottom_wt = 0
for cluster_id in all_clusters:
    color = get_cluster_color(cluster_id)
    pc_pct = pc_percentages.iloc[cluster_id - 1]
    in_pct = in_percentages.iloc[cluster_id - 1]
    wt_pct = wt_percentages.iloc[cluster_id - 1]

    # PC bar
    ax.bar(0, pc_pct, bar_width, bottom=bottom_pc, color=color,
           label=f'Cluster {cluster_id}' if cluster_id == 1 else None)
    if pc_pct > 3:
        ax.text(0, bottom_pc + pc_pct/2, f"{pc_pct:.1f}%", ha='center', va='center',
                color=get_text_color(color), fontsize=9, fontweight='bold')
    bottom_pc += pc_pct

    # IN bar
    ax.bar(1, in_pct, bar_width, bottom=bottom_in, color=color)
    if in_pct > 3:
        ax.text(1, bottom_in + in_pct/2, f"{in_pct:.1f}%", ha='center', va='center',
                color=get_text_color(color), fontsize=9, fontweight='bold')
    bottom_in += in_pct
    
    # WT bar
    ax.bar(2, wt_pct, bar_width, bottom=bottom_wt, color=color)
    if wt_pct > 3:
        ax.text(2, bottom_wt + wt_pct/2, f"{wt_pct:.1f}%", ha='center', va='center',
                color=get_text_color(color), fontsize=9, fontweight='bold')
    bottom_wt += wt_pct

# Add sample counts
ax.text(0, 102, f"n={len(pc_cluster_assignments)}", ha='center', va='bottom', fontweight='bold')
ax.text(1, 102, f"n={len(in_cluster_assignments)}", ha='center', va='bottom', fontweight='bold')
ax.text(2, 102, f"n={len(wt_cluster_assignments)}", ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Percentage (%)')
ax.set_title('Cluster Distribution: PC vs IN vs WT Pooled')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Purkinje Cells', 'Interneurons', 'WT Pooled'])
ax.set_ylim(0, 110)

# Legend
handles = [plt.Rectangle((0,0),1,1, color=get_cluster_color(c)) for c in all_clusters]
ax.legend(handles, [f'Cluster {c}' for c in all_clusters],
          bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
output_file = OUTPUT_DIR / "pc_vs_in_vs_wt_cluster_distribution.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved cluster distribution to {output_file}")
print(f"PC distribution: {dict(pc_cluster_counts)}")
print(f"IN distribution: {dict(in_cluster_counts)}")
print(f"WT distribution: {dict(wt_cluster_counts)}")


## Chapter G – SynII Integration and Fiber Organization

Chapter G projects SynII boutons into the WT-defined manifold, contrasts their properties, and examines how bouton classes distribute along axonal fibers.


### G.1 SynII Projection

SynII bouton features are projected into the WT-derived PCA space to assess how the mutant dataset occupies the existing manifold. This shared embedding enables direct comparison between SynII and WT boutons.


In [ ]:
# Project SynII data onto WT-trained PCA space
plt.figure(figsize=(8, 6))

# Plot WT pooled data with cluster colors
plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1],
           c=get_cluster_colors(cluster_assignments), alpha=0.4, s=30, label='WT pooled (2.5mM Ca)')

# Plot SynII projected data
synii_pca_coords = pca_data['SynII']
plt.scatter(synii_pca_coords[:, 0], synii_pca_coords[:, 1],
           marker='*', s=100, c='black', alpha=0.8,
           edgecolors='white', linewidth=1, label=f'SynII KO (n={len(synii_pca_coords)})')

# Format plot
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.xlim(-7, 10)                    
plt.ylim(-6, 6)
plt.title('SynII KO Projection onto WT PCA Space')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "synii_pca_projection.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ SynII projection: {len(synii_pca_coords)} samples projected onto WT PCA space")
print(f"✓ Saved to {output_file}")


### G.2 Cluster Attribution for SynII

Using distances in PCA space, SynII boutons are assigned to the nearest WT-derived clusters. This provides a principled way to translate the WT clustering schema onto the mutant data.


In [ ]:
# Assign SynII samples to WT-derived clusters using PCA space
def assign_clusters_robust(X_existing, labels_existing, X_new, method='centroid'):
    """Assign new samples to existing clusters using specified linkage method."""
    unique_labels = np.unique(labels_existing)
    
    if method == 'centroid':
        centroids = np.vstack([X_existing[labels_existing == lbl].mean(axis=0) for lbl in unique_labels])
        distances = cdist(X_new, centroids)
        return unique_labels[np.argmin(distances, axis=1)]
    
    elif method == 'single':
        dist_matrix = np.empty((X_new.shape[0], len(unique_labels)))
        for j, lbl in enumerate(unique_labels):
            cluster_points = X_existing[labels_existing == lbl]
            dist_matrix[:, j] = np.min(cdist(X_new, cluster_points), axis=1)
        return unique_labels[np.argmin(dist_matrix, axis=1)]
    
    else:
        raise ValueError("method must be 'centroid' or 'single'")

# Assign SynII samples to WT clusters
synii_cluster_assignments = assign_clusters_robust(pca_coordinates, cluster_assignments, 
                                                   pca_data['SynII'], method='single')

print(f"✓ Assigned {len(synii_cluster_assignments)} SynII samples to WT clusters")
print(f"SynII cluster distribution: {dict(pd.Series(synii_cluster_assignments).value_counts().sort_index())}")

### G.3 Cluster Composition Comparison

Cluster membership counts for WT and SynII boutons are contrasted to reveal which bouton phenotypes expand or diminish in the mutant. The resulting bar plots offer a population-level view of SynII remodeling.


In [ ]:
# Compare cluster distributions between WT and SynII
wt_cluster_counts = pd.Series(cluster_assignments).value_counts().sort_index()
synii_cluster_counts = pd.Series(synii_cluster_assignments).value_counts()

# Ensure all clusters represented
all_clusters = sorted(wt_cluster_counts.index)
synii_complete = pd.Series([synii_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)

# Calculate percentages
wt_percentages = 100 * wt_cluster_counts / len(cluster_assignments)
synii_percentages = 100 * synii_complete / len(synii_cluster_assignments)

# Stacked bar plot
fig, ax = plt.subplots(figsize=(8, 6))
bar_width = 0.6

def get_text_color(rgb):
    r, g, b = rgb[:3]
    return 'white' if 0.2126*r + 0.7152*g + 0.0722*b < 0.55 else 'black'

bottom_wt = bottom_synii = 0
for cluster_id in all_clusters:
    color = get_cluster_color(cluster_id)
    wt_pct = wt_percentages.iloc[cluster_id - 1]
    synii_pct = synii_percentages.iloc[cluster_id - 1]

    # WT bar
    ax.bar(0, wt_pct, bar_width, bottom=bottom_wt, color=color,
           label=f'Cluster {cluster_id}' if cluster_id == 1 else None)
    if wt_pct > 3:  # Only label if segment is large enough
        ax.text(0, bottom_wt + wt_pct/2, f"{wt_pct:.1f}%", ha='center', va='center',
                color=get_text_color(color), fontsize=9)
    bottom_wt += wt_pct

    # SynII bar
    ax.bar(1, synii_pct, bar_width, bottom=bottom_synii, color=color, alpha=0.7)
    if synii_pct > 3:
        ax.text(1, bottom_synii + synii_pct/2, f"{synii_pct:.1f}%", ha='center', va='center',
                color=get_text_color(color), fontsize=9)
    bottom_synii += synii_pct

# Add sample counts
ax.text(0, 102, f"n={len(cluster_assignments)}", ha='center', va='bottom', fontweight='bold')
ax.text(1, 102, f"n={len(synii_cluster_assignments)}", ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Percentage (%)')
ax.set_title('Cluster Distribution: WT vs SynII')
ax.set_xticks([0, 1])
ax.set_xticklabels(['WT', 'SynII'])
ax.set_ylim(0, 110)

# Legend
handles = [plt.Rectangle((0,0),1,1, color=get_cluster_color(c)) for c in all_clusters]
ax.legend(handles, [f'Cluster {c}' for c in all_clusters],
          bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
output_file = OUTPUT_DIR / "wt_vs_synii_cluster_distribution.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved cluster distribution to {output_file}")


### G.4 SynII Descriptive Statistics

Summary statistics for SynII amplitudes and failure rates are computed to contextualize the mutant population. Reporting mean and standard deviation for key metrics helps quantify how SynII boutons diverge from WT benchmarks.


In [ ]:
# Calculate mean ± SD for Amp1, Amp2, and %Fail1 in SynII data
print("Mean ± SD for SynII data:")
print(f"AMP1: {PCA_Data_SynII['AMP1'].mean():.3f} ± {PCA_Data_SynII['AMP1'].std():.3f}")
print(f"AMP2: {PCA_Data_SynII['AMP2'].mean():.3f} ± {PCA_Data_SynII['AMP2'].std():.3f}")
print(f"%Fail1: {PCA_Data_SynII['%Fail1'].mean():.3f} ± {PCA_Data_SynII['%Fail1'].std():.3f}")

### G.5 SynII-Enriched Clusters

This analysis identifies clusters disproportionately populated by SynII boutons and extracts the associated traces. Spotlighting these groups isolates the synaptic phenotypes most affected by the SynII mutation.


In [ ]:
# Identify and analyze SynII-enriched clusters
enriched_clusters = []
for cluster_id in all_clusters:
    wt_pct    = wt_percentages[cluster_id]
    synii_pct = synii_percentages[cluster_id]
    if synii_pct > wt_pct:
        enriched_clusters.append(cluster_id)
        print(f"Cluster {cluster_id}: WT={wt_pct:.1f}%, SynII={synii_pct:.1f}% (enriched)")

if enriched_clusters:
    # PPR profile comparison for enriched clusters
    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled_clustered.columns]
    x_pulses = list(range(1, len(ppr_cols)+2))
    
    plt.figure(figsize=(10, 6))
    
    # Plot enriched WT clusters
    for cluster_id in enriched_clusters:
        cluster_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cluster_id]
        means        = [1] + cluster_data[ppr_cols].mean().tolist()
        sems         = [0] + cluster_data[ppr_cols].sem().tolist()
        color        = get_cluster_color(cluster_id)
        
        plt.plot(x_pulses, means, marker='o', label=f'WT Cluster {cluster_id} (n={len(cluster_data)})', 
                color=color, linewidth=2)
        plt.fill_between(x_pulses, np.array(means)-np.array(sems), np.array(means)+np.array(sems), 
                        alpha=0.2, color=color)
    
    # Add SynII profile
    synii_means = [1] + PCA_Data_SynII[ppr_cols].mean().tolist()
    synii_sems  = [0] + PCA_Data_SynII[ppr_cols].sem().tolist()
    plt.plot(x_pulses, synii_means, marker='s', label=f'SynII KO (n={len(PCA_Data_SynII)})', 
            color='red', linewidth=1)
    plt.fill_between(x_pulses, np.array(synii_means)-np.array(synii_sems), 
                    np.array(synii_means)+np.array(synii_sems), alpha=0.2, color='red')
    
    plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
    plt.ylabel('Mean PPR (A_n/A_1)')
    plt.xlabel('Pulse Number')
    plt.xticks(x_pulses)
    plt.title(f'PPR Profiles: SynII-Enriched WT Clusters vs SynII KO')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "synii_enriched_clusters_ppr_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Saved enriched clusters comparison to {output_file}")
else:
    print("No SynII-enriched clusters found")

# SynII summary statistics
print(f"\nSynII Summary Statistics:")
for param in ['AMP1', 'AMP2', '%Fail1']:
    if param in PCA_Data_SynII.columns:
        mean_val = PCA_Data_SynII[param].mean()
        std_val  = PCA_Data_SynII[param].std()
        print(f"{param}: {mean_val:.3f} ± {std_val:.3f}")

### G.6 SynII vs WT Trace Overlays

Mean traces from SynII-enriched clusters are compared against their WT counterparts to visualize how response kinetics shift in the mutant. These overlays tie population statistics back to time-domain dynamics.


In [ ]:
# Compare mean traces between SynII and WT with cluster assignments

# Ensure SynII has cluster assignments
if 'cluster_synII' not in PCA_Data_SynII.columns:
    PCA_Data_SynII['cluster_synII'] = synii_cluster_assignments

# Extract SynII and WT traces from resampled data
synii_traces = [resampled_trace_lookup[bid]['Avg'] for bid in PCA_Data_SynII['ID'] if bid in resampled_trace_lookup]
wt_traces    = []

for _, row in PCA_Data_WT_Pooled_clustered.iterrows():
    bouton_id = row['ID']
    if bouton_id in resampled_trace_lookup:
        wt_traces.append(resampled_trace_lookup[bouton_id]['Avg'])

if len(synii_traces) > 0 and len(wt_traces) > 0:
    # Calculate mean and SEM for each group
    synii_matrix = np.column_stack(synii_traces)
    wt_matrix    = np.column_stack(wt_traces)
    
    synii_mean   = np.nanmean(synii_matrix, axis=1)
    synii_sem    = np.nanstd(synii_matrix, axis=1, ddof=1) / np.sqrt(synii_matrix.shape[1])
    
    wt_mean      = np.nanmean(wt_matrix, axis=1)
    wt_sem       = np.nanstd(wt_matrix, axis=1, ddof=1) / np.sqrt(wt_matrix.shape[1])
    
    # Calculate y-limits for consistent scaling
    all_values = np.concatenate([
        synii_mean - synii_sem, synii_mean + synii_sem,
        wt_mean - wt_sem, wt_mean + wt_sem
    ])
    y_min, y_max = np.nanmin(all_values), np.nanmax(all_values)
    y_padding    = (y_max - y_min) * 0.05
    
    # Create comparison plot with cropped time axis
    fig, (ax_synii, ax_wt) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    
    # SynII panel
    ax_synii.plot(COMMON_TIME, synii_mean, color='red', linewidth=2, 
                  label=f'SynII KO (n={len(synii_traces)})')
    ax_synii.fill_between(COMMON_TIME, synii_mean - synii_sem, synii_mean + synii_sem, 
                          color='red', alpha=0.25)
    
    ax_synii.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax_synii.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Stimulus')
    ax_synii.set_xlim(0.5, 2.0)  # Cropped time axis
    ax_synii.set_ylim(y_min - y_padding, y_max + y_padding)
    ax_synii.set_title('SynII KO - Average Response')
    ax_synii.set_ylabel('ΔF/F')
    ax_synii.set_xlabel('Time (s)')
    ax_synii.legend()
    ax_synii.grid(True, alpha=0.3)
    
    # WT panel
    ax_wt.plot(COMMON_TIME, wt_mean, color='black', linewidth=2, 
               label=f'WT (n={len(wt_traces)})')
    ax_wt.fill_between(COMMON_TIME, wt_mean - wt_sem, wt_mean + wt_sem, 
                       color='black', alpha=0.25)
    ax_wt.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax_wt.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Stimulus')
    ax_wt.set_xlim(0.5, 2.0)  # Cropped time axis
    ax_wt.set_ylim(y_min - y_padding, y_max + y_padding)
    ax_wt.set_title('WT - Average Response')
    ax_wt.set_xlabel('Time (s)')
    ax_wt.legend()
    ax_wt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "synii_vs_wt_mean_traces.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Mean trace comparison: SynII (n={len(synii_traces)}) vs WT (n={len(wt_traces)})")
    print(f"✓ Time axis focused on stimulus response period (0.5-2.0s)")
    
    # Cluster distribution summary
    if 'enriched_clusters' in locals() and enriched_clusters:
        print(f"\nSynII distribution in enriched clusters {enriched_clusters}:")
        for cluster_id in enriched_clusters:
            count   = np.sum(synii_cluster_assignments == cluster_id)
            percent = 100 * count / len(synii_cluster_assignments)
            print(f"  Cluster {cluster_id}: {count} samples ({percent:.1f}%)")
    
    print(f"✓ Saved to {output_file}")
    
else:
    print("No trace data available for comparison")

### G.7 High-Amplitude Bouton VS Syn II

#### G.7.a High-Amplitude Bouton Identification VS SYN II

This cell pinpoints WT boutons whose initial EPSC amplitudes exceed the largest SynII response. Flagging these outliers enables targeted inspection of whether unusually strong WT boutons cluster together or remain dispersed.


In [ ]:
# Extract SynII amplitude data from AMP1-AMP10 fields in FEATURES_DATAFRAME
synii_trace_data = []
synii_max_amplitudes = []

# Loop through SynII boutons in NORM_TRACES_DATAFRAME
for _, trace_row in NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == 'SynII'].iterrows():
    bouton_id = trace_row['ID']
    if bouton_id in resampled_trace_lookup:
        trace_values = resampled_trace_lookup[bouton_id]['Avg']
        synii_trace_data.append((bouton_id, trace_values))
        
        # Find matching row in FEATURES_DATAFRAME
        matching_feature = FEATURES_DATAFRAME[
            (FEATURES_DATAFRAME['ID'] == bouton_id) & 
            (FEATURES_DATAFRAME['Condition'] == 'SynII')
        ]
        
        if not matching_feature.empty:
            # Extract AMP1-AMP10 values
            amp_columns = [f'AMP{i}' for i in range(1, 11)]
            amp_values = []
            for col in amp_columns:
                if col in matching_feature.columns:
                    val = matching_feature[col].iloc[0]
                    if pd.notna(val):
                        amp_values.append(abs(val))
            
            # Find maximum absolute amplitude across all AMP fields
            if len(amp_values) > 0:
                max_amp = np.max(amp_values)
                synii_max_amplitudes.append(max_amp)

# Calculate SynII amplitude threshold (95th percentile)
if len(synii_max_amplitudes) > 0:
    synii_amplitude_threshold = np.percentile(synii_max_amplitudes, 95)
else:
    synii_amplitude_threshold = 0

# Extract WT pooled trace data and identify high-amplitude traces
wt_trace_data = []
wt_high_amplitude_traces = []
wt_regular_traces = []

# Loop through WT boutons in NORM_TRACES_DATAFRAME
for _, trace_row in NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'].isin(['WT_Anthime', 'WT_Theo', 'WT_Theo_1scd'])].iterrows():
    bouton_id = trace_row['ID']
    condition = trace_row['Condition']
    
    if bouton_id in resampled_trace_lookup:
        trace_values = resampled_trace_lookup[bouton_id]['Avg']
        wt_trace_data.append((bouton_id, trace_values))
        
        # Find matching row in FEATURES_DATAFRAME
        matching_feature = FEATURES_DATAFRAME[
            (FEATURES_DATAFRAME['ID'] == bouton_id) & 
            (FEATURES_DATAFRAME['Condition'] == condition)
        ]
        
        if not matching_feature.empty:
            # Extract AMP1-AMP10 values
            amp_columns = [f'AMP{i}' for i in range(1, 11)]
            amp_values = []
            for col in amp_columns:
                if col in matching_feature.columns:
                    val = matching_feature[col].iloc[0]
                    if pd.notna(val):
                        amp_values.append(abs(val))
            
            # Check if any amplitude exceeds SynII threshold
            if len(amp_values) > 0:
                max_amp = np.max(amp_values)
                if max_amp > synii_amplitude_threshold:
                    wt_high_amplitude_traces.append((bouton_id, trace_values, max_amp))
                else:
                    wt_regular_traces.append((bouton_id, trace_values))

# Calculate common y-axis limits for both panels
all_trace_values = []

# Collect all SynII trace values
for _, trace_values in synii_trace_data:
    all_trace_values.extend(trace_values[~np.isnan(trace_values)])

# Collect all WT trace values
for _, trace_values in wt_trace_data:
    all_trace_values.extend(trace_values[~np.isnan(trace_values)])

# Set common y-limits with some padding
y_min = np.min(all_trace_values) * 1.1
y_max = np.max(all_trace_values) * 1.1

# Create the dual-panel plot with clipped x-axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: SynII traces
ax1.set_title(f'SynII Traces (n={len(synii_trace_data)})', fontweight='bold')

synii_traces_for_avg = []
for bouton_id, trace_values in synii_trace_data:
    ax1.plot(COMMON_TIME, trace_values, color='orange', alpha=0.1, linewidth=0.3)
    synii_traces_for_avg.append(trace_values)

# SynII average
if synii_traces_for_avg:
    synii_average = np.nanmean(synii_traces_for_avg, axis=0)
    ax1.plot(COMMON_TIME, synii_average, color='darkorange', linewidth=1, label='SynII Average')

# Threshold lines
ax1.axhline(synii_amplitude_threshold, color='red', linestyle='--', linewidth=2, 
           label=f'Threshold: {synii_amplitude_threshold:.3f}')
ax1.axhline(-synii_amplitude_threshold, color='red', linestyle='--', linewidth=2)

ax1.set_xlabel('Time (s)')
ax1.set_ylabel('ΔF/F')
ax1.set_xlim(0.5, 2.0)
ax1.set_ylim(y_min, y_max)
ax1.legend()
ax1.set_title('SynII Traces (Individual + Average)')

# Panel 2: WT traces with improved readability
ax2.set_title(f'WT Traces: {len(wt_high_amplitude_traces)}/{len(wt_trace_data)} above threshold', fontweight='bold')

# Plot WT traces below threshold with very low alpha
for bouton_id, trace_values in wt_regular_traces:
    ax2.plot(COMMON_TIME, trace_values, color='black', alpha=0.05, linewidth=0.25)

# Plot WT traces above threshold with moderate alpha
for bouton_id, trace_values, max_amp in wt_high_amplitude_traces:
    ax2.plot(COMMON_TIME, trace_values, color='red', alpha=0.25, linewidth=0.3)

# WT average
if wt_trace_data:
    wt_traces_for_avg = [trace_values for _, trace_values in wt_trace_data]
    wt_average = np.nanmean(wt_traces_for_avg, axis=0)
    ax2.plot(COMMON_TIME, wt_average, color='darkblue', linewidth=1, label='WT Average')

# Threshold lines
ax2.axhline(synii_amplitude_threshold, color='red', linestyle='--', linewidth=2, 
           label=f'SynII Threshold')
ax2.axhline(-synii_amplitude_threshold, color='red', linestyle='--', linewidth=2)

# Add legend entries for trace types
ax2.plot([], [], color='black', alpha=0.6, linewidth=2, label=f'Below threshold (n={len(wt_regular_traces)})')
ax2.plot([], [], color='red', alpha=0.7, linewidth=2, label=f'Above threshold (n={len(wt_high_amplitude_traces)})')

ax2.set_xlabel('Time (s)')
ax2.set_ylabel('ΔF/F')
ax2.set_xlim(0.5, 2.0)
ax2.set_ylim(y_min, y_max)
ax2.legend()
ax2.axvline(1.0, color='blue', linestyle=':', alpha=0.7, linewidth=2)

plt.tight_layout()

output_file = OUTPUT_DIR / "synii_vs_wt_amplitude_comparison_improved.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

#### G.7.b High-Amplitude Bouton Mapping

The previously identified high-amplitude WT boutons are highlighted within PCA space to see whether they define a coherent region or scatter across clusters. Their locations inform hypotheses about the mechanisms supporting exceptionally strong synapses.


In [ ]:
# Show PCA locations of high-amplitude WT traces identified in the previous cell (ignoring target type)
if wt_high_amplitude_traces:
    high_amp_ids = [bouton_id for bouton_id, _, _ in wt_high_amplitude_traces]
    high_amp_mask = PCA_Data_WT_Pooled_clustered['ID'].isin(high_amp_ids)
    high_amp_coordinates = pca_coordinates[high_amp_mask]   

    plt.figure(figsize=(8, 6)) 
# Plot WT pooled data with cluster colors
    plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1],
           c=get_cluster_colors(cluster_assignments), alpha=0.4, s=30, label='WT pooled (2.5mM Ca)')
    
# Plot high-amplitude WT traces
    plt.scatter(high_amp_coordinates[:, 0], high_amp_coordinates[:, 1], 
               marker='p', c='red', edgecolors='black', linewidths=0.6, s=70, label=f'High-Amplitude WT (n={len(high_amp_coordinates)})')
    
# Format plot
    plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
    plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
    plt.title('PCA Locations of High-Amplitude WT Traces')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    output_file = OUTPUT_DIR / "pca_high_amplitude_wt_traces.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

### G.8 – SynII and Target Identity

We investigate how SynII bouton positions relate to Purkinje and interneuron territories within PCA space.


#### G.8.a SynII Confidence Ellipses

We construct ellipses around SynII boutons in PCA space to delineate the region occupied by the mutant population. This geometric boundary provides an interpretable measure of SynII variability relative to WT clusters.


In [ ]:
def compare_inside_outside_simplified(in_ids, in_inside_mask, pc_ids, pc_inside_mask):
    """Create clean 2-panel comparison of inside vs outside ellipse traces."""
    
    # Prepare IN data
    in_inside_ids     = [in_ids[i] for i in range(len(in_ids)) if in_inside_mask[i]]
    in_outside_ids    = [in_ids[i] for i in range(len(in_ids)) if not in_inside_mask[i]]
    
    in_inside_traces  = [resampled_trace_lookup[bid]['Avg'] for bid in in_inside_ids if bid in resampled_trace_lookup]
    in_outside_traces = [resampled_trace_lookup[bid]['Avg'] for bid in in_outside_ids if bid in resampled_trace_lookup]
    
    # Prepare PC data
    pc_inside_ids     = [pc_ids[i] for i in range(len(pc_ids)) if pc_inside_mask[i]]
    pc_outside_ids    = [pc_ids[i] for i in range(len(pc_ids)) if not pc_inside_mask[i]]
    
    pc_inside_traces  = [resampled_trace_lookup[bid]['Avg'] for bid in pc_inside_ids if bid in resampled_trace_lookup]
    pc_outside_traces = [resampled_trace_lookup[bid]['Avg'] for bid in pc_outside_ids if bid in resampled_trace_lookup]
    
    # Create 2-panel figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel 1: IN cells
    if in_inside_traces and in_outside_traces:
        in_inside_mean  = np.nanmean(in_inside_traces, axis=0)
        in_inside_sem   = np.nanstd(in_inside_traces, axis=0, ddof=1) / np.sqrt(len(in_inside_traces))
        
        in_outside_mean = np.nanmean(in_outside_traces, axis=0)
        in_outside_sem  = np.nanstd(in_outside_traces, axis=0, ddof=1) / np.sqrt(len(in_outside_traces))
        
        ax1.plot(COMMON_TIME, in_inside_mean, color='red', linewidth=1, 
                label=f'Inside ellipse (n={len(in_inside_traces)})')
        ax1.fill_between(COMMON_TIME, in_inside_mean-in_inside_sem, in_inside_mean+in_inside_sem, 
                        color='red', alpha=0.3)
        
        ax1.plot(COMMON_TIME, in_outside_mean, color='black', linewidth=1,
                label=f'Outside ellipse (n={len(in_outside_traces)})')
        ax1.fill_between(COMMON_TIME, in_outside_mean-in_outside_sem, in_outside_mean+in_outside_sem, 
                        color='black', alpha=0.3)
    
    ax1.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax1.axvline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax1.set_xlim(0.5, 2.0)
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('ΔF/F')
    ax1.set_title('IN Cells: Inside vs Outside SynII Ellipse')
    ax1.legend()
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    
    # Panel 2: PC cells
    if pc_inside_traces and pc_outside_traces:
        pc_inside_mean  = np.nanmean(pc_inside_traces, axis=0)
        pc_inside_sem   = np.nanstd(pc_inside_traces, axis=0, ddof=1) / np.sqrt(len(pc_inside_traces))
        
        pc_outside_mean = np.nanmean(pc_outside_traces, axis=0)
        pc_outside_sem  = np.nanstd(pc_outside_traces, axis=0, ddof=1) / np.sqrt(len(pc_outside_traces))
        
        ax2.plot(COMMON_TIME, pc_inside_mean, color='green', linewidth=1, 
                label=f'Inside ellipse (n={len(pc_inside_traces)})')
        ax2.fill_between(COMMON_TIME, pc_inside_mean-pc_inside_sem, pc_inside_mean+pc_inside_sem, 
                        color='green', alpha=0.3)
        
        ax2.plot(COMMON_TIME, pc_outside_mean, color='black', linewidth=1,
                label=f'Outside ellipse (n={len(pc_outside_traces)})')
        ax2.fill_between(COMMON_TIME, pc_outside_mean-pc_outside_sem, pc_outside_mean+pc_outside_sem, 
                        color='black', alpha=0.3)
    
    ax2.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax2.axvline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax2.set_xlim(0.5, 2.0)
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('ΔF/F')
    ax2.set_title('PC Cells: Inside vs Outside SynII Ellipse')
    ax2.legend()
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "ellipse_inside_outside_comparison_simplified.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ IN: Inside n={len(in_inside_traces)}, Outside n={len(in_outside_traces)}")
    print(f"✓ PC: Inside n={len(pc_inside_traces)}, Outside n={len(pc_outside_traces)}")

# Run simplified comparison
compare_inside_outside_simplified(in_ids, in_inside_mask, pc_ids, pc_inside_mask)

#### G.8.b Target Composition Within SynII Space

By examining which Purkinje and interneuron boutons fall inside or outside the SynII ellipse, we evaluate whether the mutant phenotype preferentially overlaps with specific target identities.


In [ ]:
# Four-panel comparison: IN/PC inside/outside SynII ellipse traces

def plot_group_traces(ax, trace_ids, group_name, color, show_individuals=True):
    """Plot individual traces + average for a group."""
    # Get traces for this group
    group_traces = [resampled_trace_lookup[bid]['Avg'] for bid in trace_ids if bid in resampled_trace_lookup]
    
    if not group_traces:
        ax.text(0.5, 0.5, f'No traces\navailable', ha='center', va='center', 
                transform=ax.transAxes, fontsize=12, color='gray')
        ax.set_title(f'{group_name}\n(n=0)')
        return
    
    # Plot individual traces
    if show_individuals:
        for trace in group_traces:
            ax.plot(COMMON_TIME, trace, color=color, alpha=0.2, linewidth=0.5)
    
    # Calculate and plot average
    group_mean = np.nanmean(group_traces, axis=0)
    group_sem  = np.nanstd(group_traces, axis=0, ddof=1) / np.sqrt(len(group_traces))
    
    ax.plot(COMMON_TIME, group_mean, color=color, linewidth=1, 
            label=f'{group_name} avg')
    ax.fill_between(COMMON_TIME, group_mean-group_sem, group_mean+group_sem, 
                    color=color, alpha=0.3)
    
    # Format subplot
    ax.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax.axvline(1.0, color='black', linestyle='--', alpha=0.5)
    ax.set_xlim(0.5, 2.0)
    ax.set_title(f'{group_name}\n(n={len(group_traces)})')
    ax.grid(True, alpha=0.3)
    
    return group_mean, group_sem

# Prepare trace groups based on ellipse classification
if 'in_inside_mask' in locals() and 'pc_inside_mask' in locals():
    # Get IDs for each group
    in_inside_ids  = [in_ids[i] for i in range(len(in_ids)) if in_inside_mask[i]]
    in_outside_ids = [in_ids[i] for i in range(len(in_ids)) if not in_inside_mask[i]]
    pc_inside_ids  = [pc_ids[i] for i in range(len(pc_ids)) if pc_inside_mask[i]]
    pc_outside_ids = [pc_ids[i] for i in range(len(pc_ids)) if not pc_inside_mask[i]]
    
    # Create 2x2 subplot layout
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
    
    # Plot each group
    plot_group_traces(axes[0,0], in_inside_ids, 'IN Inside Ellipse', 'red')
    plot_group_traces(axes[0,1], in_outside_ids, 'IN Outside Ellipse', 'darkred')
    plot_group_traces(axes[1,0], pc_inside_ids, 'PC Inside Ellipse', 'mediumseagreen')
    plot_group_traces(axes[1,1], pc_outside_ids, 'PC Outside Ellipse', 'darkgreen')
    
    # Add common labels
    for ax in axes[-1, :]:  # Bottom row
        ax.set_xlabel('Time (s)')
    for ax in axes[:, 0]:   # Left column
        ax.set_ylabel('ΔF/F')
    
    # Add overall title
    fig.suptitle(f'Trace Analysis: Inside vs Outside SynII {ELLIPSE_CONFIDENCE:.0%} Ellipse', 
                 fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "four_panel_ellipse_trace_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Summary statistics
    print(f"=== ELLIPSE TRACE ANALYSIS SUMMARY ===")
    print(f"Ellipse confidence level: {ELLIPSE_CONFIDENCE:.0%}")
    print(f"IN inside ellipse:  {len(in_inside_ids):2d}  traces ({len(in_inside_ids)/(len(in_inside_ids)+len(in_outside_ids))*100:.1f}%)")
    print(f"IN outside ellipse: {len(in_outside_ids):2d} traces ({len(in_outside_ids)/(len(in_inside_ids)+len(in_outside_ids))*100:.1f}%)")
    print(f"PC inside ellipse:  {len(pc_inside_ids):2d}  traces ({len(pc_inside_ids)/(len(pc_inside_ids)+len(pc_outside_ids))*100:.1f}%)")
    print(f"PC outside ellipse: {len(pc_outside_ids):2d} traces ({len(pc_outside_ids)/(len(pc_inside_ids)+len(pc_outside_ids))*100:.1f}%)")
    
    print(f"\n✓ Saved four-panel comparison to {output_file}")
    
else:
    print("Run ellipse analysis first to generate inside/outside classifications")

## Chapter H – Calcium Perturbations and Stability Experiments

Chapter H explores how extracellular calcium and longitudinal manipulations reshape bouton phenotypes within the PCA framework.


### H.1 Calcium Modulation in PCA Space

High- and low-calcium conditions are projected into the WT PCA embedding to observe how extracellular calcium reshapes bouton distributions. Visualizing these shifts indicates whether calcium availability drives distinct synaptic states.


In [ ]:
# Analyze calcium concentration effects on bouton properties in PCA space

def plot_calcium_trajectories():
    """Plot how calcium concentration changes affect PCA positioning."""
    
    plt.figure(figsize=(10, 8))
    
    # Background: WT pooled (2.5mM Ca standard condition)
    plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1], 
               c=get_cluster_colors(cluster_assignments), alpha=0.4, s=30, label='WT pooled (2.5mM Ca)')
    
    # Low calcium (1.5mM) - blue triangles pointing down
    low_ca_coords = pca_data['WT_1_5Ca']
    plt.scatter(low_ca_coords[:, 0], low_ca_coords[:, 1], 
               marker='v', s=60, c='blue', alpha=0.8, edgecolors='darkblue', linewidth=0.5,
               label=f'1.5mM Ca (n={len(low_ca_coords)})')
    
    # High calcium (4mM) - red triangles pointing up  
    high_ca_coords = pca_data['WT_4Ca']
    plt.scatter(high_ca_coords[:, 0], high_ca_coords[:, 1], 
               marker='^', s=60, c='red', alpha=0.8, edgecolors='darkred', linewidth=0.5,
               label=f'4mM Ca (n={len(high_ca_coords)})')
    
    # Calculate centroids
    center_pooled = np.mean(pca_coordinates, axis=0)
    center_low_ca = np.mean(low_ca_coords, axis=0)
    center_high_ca = np.mean(high_ca_coords, axis=0)
    
    # Plot centroids
    plt.scatter(center_low_ca[0], center_low_ca[1], marker='X', s=180, c='blue', 
               edgecolor='black', linewidth=2, label='1.5mM centroid')
    plt.scatter(center_high_ca[0], center_high_ca[1], marker='X', s=180, c='red', 
               edgecolor='black', linewidth=2, label='4mM centroid')
    
    # Draw arrows from pooled centroid to calcium condition centroids
    plt.arrow(center_pooled[0], center_pooled[1],
              center_low_ca[0] - center_pooled[0], center_low_ca[1] - center_pooled[1],
              color='blue', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.9)
    plt.arrow(center_pooled[0], center_pooled[1],
              center_high_ca[0] - center_pooled[0], center_high_ca[1] - center_pooled[1],
              color='red', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.9)
    
    # Annotate centroid coordinates
    plt.text(center_low_ca[0], center_low_ca[1], f'  1.5mM\n({center_low_ca[0]:.2f},{center_low_ca[1]:.2f})', 
             color='blue', fontsize=9, ha='left', va='center', fontweight='bold')
    plt.text(center_high_ca[0], center_high_ca[1], f'  4mM\n({center_high_ca[0]:.2f},{center_high_ca[1]:.2f})', 
             color='red', fontsize=9, ha='left', va='center', fontweight='bold')
    
    # Connect paired boutons between conditions (assuming matched order)
    n_pairs = min(len(low_ca_coords), len(high_ca_coords))
    if n_pairs > 0:
        for i in range(n_pairs):
            plt.plot([low_ca_coords[i, 0], high_ca_coords[i, 0]],
                     [low_ca_coords[i, 1], high_ca_coords[i, 1]],
                     color='gray', alpha=0.4, linewidth=1)
        print(f"Connected {n_pairs} bouton pairs between calcium conditions")
    
    # Format plot
    pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
    plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
    plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
    plt.title('Calcium Concentration Effects on Bouton Properties')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "calcium_concentration_pca_trajectories.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return center_pooled, center_low_ca, center_high_ca, n_pairs

def calculate_calcium_movements(center_pooled, center_low_ca, center_high_ca, n_pairs):
    """Calculate movement statistics for calcium concentration changes."""
    
    # Distances from standard condition to each calcium level
    dist_to_low  = np.linalg.norm(center_low_ca - center_pooled)
    dist_to_high = np.linalg.norm(center_high_ca - center_pooled)
    
    # Individual bouton movements
    low_ca_coords  = pca_data['WT_1_5Ca']
    high_ca_coords = pca_data['WT_4Ca']
    
    # Movement from standard to low calcium
    movements_to_low  = []
    n_low_comparisons = min(len(pca_coordinates), len(low_ca_coords))
    for i in range(n_low_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - low_ca_coords[i])
        movements_to_low.append(dist)
    
    # Movement from standard to high calcium
    movements_to_high  = []
    n_high_comparisons = min(len(pca_coordinates), len(high_ca_coords))
    for i in range(n_high_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - high_ca_coords[i])
        movements_to_high.append(dist)
    
    # Movement between calcium conditions (paired boutons)
    calcium_range_movements = []
    for i in range(n_pairs):
        dist = np.linalg.norm(low_ca_coords[i] - high_ca_coords[i])
        calcium_range_movements.append(dist)
    
    return {
        'centroid_distances': {'low': dist_to_low, 'high': dist_to_high},
        'individual_movements': {
            'to_low': movements_to_low,
            'to_high': movements_to_high,
            'between_ca': calcium_range_movements
        }
    }

# Run analysis
center_pooled, center_low_ca, center_high_ca, n_pairs = plot_calcium_trajectories()
movement_stats = calculate_calcium_movements(center_pooled, center_low_ca, center_high_ca, n_pairs)

# Display results
print(f"\n=== CALCIUM CONCENTRATION ANALYSIS ===")
print(f"Centroid coordinates:")
print(f"  WT pooled (2.5mM): ({center_pooled[0]:.3f}, {center_pooled[1]:.3f})")
print(f"  1.5mM Ca:          ({center_low_ca[0]:.3f}, {center_low_ca[1]:.3f}) - distance: {movement_stats['centroid_distances']['low']:.3f}")
print(f"  4mM Ca:            ({center_high_ca[0]:.3f}, {center_high_ca[1]:.3f}) - distance: {movement_stats['centroid_distances']['high']:.3f}")

print(f"\nIndividual bouton movements in PCA space:")
if movement_stats['individual_movements']['to_low']:
    low_moves = movement_stats['individual_movements']['to_low']
    print(f"2.5mM → 1.5mM Ca (n={len(low_moves)}): {np.mean(low_moves):.3f} ± {np.std(low_moves):.3f}")

if movement_stats['individual_movements']['to_high']:
    high_moves = movement_stats['individual_movements']['to_high']
    print(f"2.5mM → 4mM Ca (n={len(high_moves)}): {np.mean(high_moves):.3f} ± {np.std(high_moves):.3f}")

if movement_stats['individual_movements']['between_ca']:
    range_moves = movement_stats['individual_movements']['between_ca']
    print(f"1.5mM ↔ 4mM Ca (n={len(range_moves)}): {np.mean(range_moves):.3f} ± {np.std(range_moves):.3f}")

print(f"\n✓ Calcium trajectory analysis complete")

### H.2 Calcium-Dependent Amp1 Distributions

Amplitude distributions for the first stimulus are contrasted between calcium conditions to test how release probability responds to extracellular calcium changes.


In [ ]:
# Compare AMP1 distributions between calcium concentrations

def plot_calcium_amp1_comparison():
    """Compare AMP1 distributions between 2.5mM and 1.5mM calcium."""
    
    # Get AMP1 data for both conditions
    amp1_standard = PCA_Data_WT_Pooled['AMP1'].dropna()
    amp1_low_ca   = PCA_Data_WT_Low_Ca['AMP1'].dropna()
    
    # Calculate common bins for fair comparison
    all_amp1_values = pd.concat([amp1_standard, amp1_low_ca])
    bin_edges       = np.linspace(all_amp1_values.min(), all_amp1_values.max(), 61)
    
    # Create figure
    plt.figure(figsize=(10, 6))
    
    # Calculate weights for percentage display
    weights_standard = np.ones(len(amp1_standard)) * (100.0 / len(amp1_standard))
    weights_low_ca   = np.ones(len(amp1_low_ca)) * (100.0 / len(amp1_low_ca))
    
    # Plot histograms
    plt.hist(amp1_standard, bins=bin_edges, alpha=0.7, color='gray', 
             weights=weights_standard, edgecolor='black', linewidth=0.5,
             label=f'WT 2.5mM Ca (n={len(amp1_standard)})')
    plt.hist(amp1_low_ca, bins=bin_edges, alpha=0.7, color='blue', 
             weights=weights_low_ca, edgecolor='darkblue', linewidth=0.5,
             label=f'WT 1.5mM Ca (n={len(amp1_low_ca)})')
    
    # Add vertical lines for means
    mean_standard = amp1_standard.mean()
    mean_low_ca = amp1_low_ca.mean()
    
    plt.axvline(mean_standard, color='black', linestyle='--', linewidth=2, alpha=0.8,
                label=f'Mean 2.5mM: {mean_standard:.3f}')
    plt.axvline(mean_low_ca, color='blue', linestyle='--', linewidth=2, alpha=0.8,
                label=f'Mean 1.5mM: {mean_low_ca:.3f}')
    
    # Format plot
    plt.xlabel('AMP1 (Amplitude)')
    plt.ylabel('Proportion (%)')
    plt.title('AMP1 Distribution: 2.5mM vs 1.5mM Calcium')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "amp1_histogram_calcium_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return amp1_standard, amp1_low_ca, output_file

# Run analysis
amp1_standard, amp1_low_ca, output_file = plot_calcium_amp1_comparison()

# Statistical comparison
from scipy.stats import mannwhitneyu, ttest_ind

# Perform statistical tests
mw_stat, mw_p = mannwhitneyu(amp1_standard, amp1_low_ca, alternative='two-sided')
t_stat, t_p   = ttest_ind(amp1_standard, amp1_low_ca)

# Summary statistics
print(f"=== AMP1 CALCIUM COMPARISON ===")
print(f"2.5mM Ca (standard): {amp1_standard.mean():.3f} ± {amp1_standard.std():.3f} (n={len(amp1_standard)})")
print(f"1.5mM Ca (low):      {amp1_low_ca.mean():.3f} ± {amp1_low_ca.std():.3f} (n={len(amp1_low_ca)})")

print(f"\nStatistical tests:")
print(f"Mann-Whitney U test: U={mw_stat:.1f}, p={mw_p:.4g}")
print(f"T-test: t={t_stat:.3f}, p={t_p:.4g}")

# Effect size (Cohen's d)
pooled_std = np.sqrt(((len(amp1_standard)-1)*amp1_standard.var() + (len(amp1_low_ca)-1)*amp1_low_ca.var()) / 
                     (len(amp1_standard) + len(amp1_low_ca) - 2))
cohens_d   = (amp1_standard.mean() - amp1_low_ca.mean()) / pooled_std
print(f"Cohen's d (effect size): {cohens_d:.3f}")

# Percentage change
pct_change = ((amp1_low_ca.mean() - amp1_standard.mean()) / amp1_standard.mean()) * 100
print(f"Percentage change (1.5mM vs 2.5mM): {pct_change:+.1f}%")

print(f"\n✓ Saved comparison to {output_file}")

### H.3 Calcium-Dependent Failure Rates

Failure percentages are compared between calcium levels, revealing whether reduced calcium disproportionately increases synaptic failures.


In [ ]:
# Compare failure rates between calcium concentrations
fail1_standard = PCA_Data_WT_Pooled['%Fail1'].dropna()
fail1_low_ca   = PCA_Data_WT_Low_Ca['%Fail1'].dropna()

# Plot histograms
all_fail1 = pd.concat([fail1_standard, fail1_low_ca])
bins      = np.linspace(all_fail1.min(), all_fail1.max(), 31)

plt.figure(figsize=(10, 6))
weights_standard = np.ones(len(fail1_standard)) / len(fail1_standard) * 100
weights_low_ca   = np.ones(len(fail1_low_ca)) / len(fail1_low_ca) * 100

plt.hist(fail1_standard, bins=bins, alpha=0.7, color='gray', weights=weights_standard, 
         edgecolor='black', label=f'WT 2.5mM Ca (n={len(fail1_standard)})')
plt.hist(fail1_low_ca, bins=bins, alpha=0.7, color='blue', weights=weights_low_ca, 
         edgecolor='darkblue', label=f'WT 1.5mM Ca (n={len(fail1_low_ca)})')

plt.axvline(fail1_standard.mean(), color='black', linestyle='--', linewidth=2)
plt.axvline(fail1_low_ca.mean(), color='blue', linestyle='--', linewidth=2)

plt.xlabel('%Fail1')
plt.ylabel('Proportion (%)')
plt.title('Failure Rate: 2.5mM vs 1.5mM Calcium')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

output_file = OUTPUT_DIR / "fail1_calcium_comparison.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Stats
from scipy.stats import mannwhitneyu
_, p_value = mannwhitneyu(fail1_standard, fail1_low_ca)
print(f"2.5mM Ca: {fail1_standard.mean():.1f}% ± {fail1_standard.std():.1f}%")
print(f"1.5mM Ca: {fail1_low_ca.mean():.1f}% ± {fail1_low_ca.std():.1f}%")
print(f"Mann-Whitney p = {p_value:.4g}")

### H.4 Calcium Impact on Summary Metrics

Non-parametric tests and paired boxplots quantify how calcium concentration affects key amplitudes and plasticity measures, providing statistical backing for observed shifts.


In [ ]:
# Direct comparison between low and high calcium conditions
from scipy.stats import mannwhitneyu

# Prepare data for plotting
amp1_comparison = pd.DataFrame({
    'AMP1': pd.concat([PCA_Data_WT_Low_Ca['AMP1'], PCA_Data_WT_High_Ca['AMP1']], ignore_index=True),
    'Condition': ['1.5mM Ca'] * len(PCA_Data_WT_Low_Ca) + ['4mM Ca'] * len(PCA_Data_WT_High_Ca)
})

fail1_comparison = pd.DataFrame({
    '%Fail1': pd.concat([PCA_Data_WT_Low_Ca['%Fail1'], PCA_Data_WT_High_Ca['%Fail1']], ignore_index=True),
    'Condition': ['1.5mM Ca'] * len(PCA_Data_WT_Low_Ca) + ['4mM Ca'] * len(PCA_Data_WT_High_Ca)
})

# Create side-by-side boxplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# AMP1 comparison
sns.boxplot(data=amp1_comparison, x='Condition', y='AMP1', ax=ax1, 
           palette=['blue', 'red'], showcaps=True, fliersize=0)
sns.stripplot(data=amp1_comparison, x='Condition', y='AMP1', ax=ax1,
             color='black', size=3, alpha=0.6)
ax1.set_title('AMP1: Low vs High Calcium')

# %Fail1 comparison  
sns.boxplot(data=fail1_comparison, x='Condition', y='%Fail1', ax=ax2,
           palette=['blue', 'red'], showcaps=True, fliersize=0)
sns.stripplot(data=fail1_comparison, x='Condition', y='%Fail1', ax=ax2,
             color='black', size=3, alpha=0.6)
ax2.set_title('%Fail1: Low vs High Calcium')

plt.tight_layout()

output_file = OUTPUT_DIR / "calcium_direct_comparison_boxplots.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Statistical tests
amp1_u, amp1_p = mannwhitneyu(PCA_Data_WT_Low_Ca['AMP1'], PCA_Data_WT_High_Ca['AMP1'])
fail1_u, fail1_p = mannwhitneyu(PCA_Data_WT_Low_Ca['%Fail1'], PCA_Data_WT_High_Ca['%Fail1'])

# Results
print("1.5mM vs 4mM Calcium Comparison:")
print("-" * 40)
print(f"AMP1:")
print(f"  1.5mM: {PCA_Data_WT_Low_Ca['AMP1'].mean():.3f} ± {PCA_Data_WT_Low_Ca['AMP1'].std():.3f}")
print(f"  4mM:   {PCA_Data_WT_High_Ca['AMP1'].mean():.3f} ± {PCA_Data_WT_High_Ca['AMP1'].std():.3f}")
print(f"  p = {amp1_p:.4g}")

print(f"%Fail1:")
print(f"  1.5mM: {PCA_Data_WT_Low_Ca['%Fail1'].mean():.1f}% ± {PCA_Data_WT_Low_Ca['%Fail1'].std():.1f}%")
print(f"  4mM:   {PCA_Data_WT_High_Ca['%Fail1'].mean():.1f}% ± {PCA_Data_WT_High_Ca['%Fail1'].std():.1f}%")
print(f"  p = {fail1_p:.4g}")

# Save stats
stats_file = OUTPUT_DIR / "calcium_comparison_statistics.txt"
with open(stats_file, 'w') as f:
    f.write("1.5mM vs 4mM Calcium Statistical Comparison\n")
    f.write("=" * 45 + "\n\n")
    f.write(f"AMP1: Mann-Whitney U={amp1_u:.1f}, p={amp1_p:.6g}\n")
    f.write(f"%Fail1: Mann-Whitney U={fail1_u:.1f}, p={fail1_p:.6g}\n")

print(f"✓ Saved to {output_file} and {stats_file}")

### H.5 Calcium PPR 

Average paired-pulse profiles are contrasted across calcium conditions to determine whether facilitation dynamics are calcium-sensitive.


In [ ]:
# Compare PPR profiles across calcium concentrations
ppr_cols      = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Low_Ca.columns]
pulse_numbers = list(range(1, len(ppr_cols) + 2))

# Calculate PPR profiles for each condition
def get_ppr_profile(data, condition_name):
    means = [1.0] + data[ppr_cols].mean().tolist()
    sems  = [0.0] + data[ppr_cols].sem().tolist()
    return means, sems

# Get profiles for each calcium condition
means_low_ca, sems_low_ca     = get_ppr_profile(PCA_Data_WT_Low_Ca, '1.5mM Ca')
means_high_ca, sems_high_ca   = get_ppr_profile(PCA_Data_WT_High_Ca, '4mM Ca') 
means_standard, sems_standard = get_ppr_profile(PCA_Data_WT_Pooled, '2.5mM Ca')

# Plot PPR profiles
plt.figure(figsize=(8, 5))

# Low calcium (blue)
plt.plot(pulse_numbers, means_low_ca, marker='o', color='blue', linewidth=2,
         label=f'1.5mM Ca (n={len(PCA_Data_WT_Low_Ca)})')
plt.fill_between(pulse_numbers, np.array(means_low_ca) - np.array(sems_low_ca),
                 np.array(means_low_ca) + np.array(sems_low_ca), color='blue', alpha=0.2)

# High calcium (red)
plt.plot(pulse_numbers, means_high_ca, marker='s', color='red', linewidth=2,
         label=f'4mM Ca (n={len(PCA_Data_WT_High_Ca)})')
plt.fill_between(pulse_numbers, np.array(means_high_ca) - np.array(sems_high_ca),
                 np.array(means_high_ca) + np.array(sems_high_ca), color='red', alpha=0.2)

# Standard calcium (black)
plt.plot(pulse_numbers, means_standard, marker='D', color='black', linewidth=2,
         label=f'2.5mM Ca (n={len(PCA_Data_WT_Pooled)})')
plt.fill_between(pulse_numbers, np.array(means_standard) - np.array(sems_standard),
                 np.array(means_standard) + np.array(sems_standard), color='black', alpha=0.15)

# Format plot
plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
plt.xlabel('Pulse Number')
plt.ylabel('PPR (A_n/A_1)')
plt.title('PPR Profiles: Calcium Concentration Effects')
plt.xticks(pulse_numbers)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save figure and data
output_fig = OUTPUT_DIR / "ppr_profiles_calcium_comparison.pdf"
plt.savefig(output_fig, dpi=300, bbox_inches='tight')
plt.show()

# Save numerical data
output_data = OUTPUT_DIR / "ppr_profiles_calcium_data.txt"
with open(output_data, "w") as f:
    f.write("Pulse\t1.5mM_Mean\t1.5mM_SEM\t4mM_Mean\t4mM_SEM\t2.5mM_Mean\t2.5mM_SEM\n")
    for i, pulse in enumerate(pulse_numbers):
        f.write(f"{pulse}\t{means_low_ca[i]:.4f}\t{sems_low_ca[i]:.4f}\t"
                f"{means_high_ca[i]:.4f}\t{sems_high_ca[i]:.4f}\t"
                f"{means_standard[i]:.4f}\t{sems_standard[i]:.4f}\n")

print(f"✓ Saved PPR profiles to {output_fig}")
print(f"✓ Saved numerical data to {output_data}")

### H.6 Calcium Trace Morphology

Mean traces from high- and low-calcium experiments are compared over the response window, highlighting kinetic differences attributable to calcium availability.


In [ ]:
# Compare mean traces between calcium concentrations (0.5-2.0s window)

# Extract traces for calcium conditions
low_ca_traces  = []
high_ca_traces = []

for _, row in NORM_TRACES_DATAFRAME.iterrows():
    if row['Condition'] == 'Theo_1_5Ca':
        low_ca_traces.append(row['Avg'])
    elif row['Condition'] == 'Theo_4Ca':
        high_ca_traces.append(row['Avg'])

if not low_ca_traces or not high_ca_traces:
    print("Calcium trace conditions not found in resampled data")
    available_conditions = NORM_TRACES_DATAFRAME['Condition'].unique()
    print(f"Available conditions: {list(available_conditions)}")
else:
    # Calculate means and SEMs
    low_ca_mean = np.nanmean(low_ca_traces, axis=0)
    low_ca_sem = np.nanstd(low_ca_traces, axis=0, ddof=1) / np.sqrt(len(low_ca_traces))
    
    high_ca_mean = np.nanmean(high_ca_traces, axis=0)
    high_ca_sem = np.nanstd(high_ca_traces, axis=0, ddof=1) / np.sqrt(len(high_ca_traces))
    
    # Plot comparison (0.5-2.0s window)
    plt.figure(figsize=(10, 5))
    
    plt.plot(COMMON_TIME, low_ca_mean, color='blue', linewidth=2, 
             label=f'1.5mM Ca (n={len(low_ca_traces)})')
    plt.fill_between(COMMON_TIME, low_ca_mean - low_ca_sem, low_ca_mean + low_ca_sem, 
                     color='blue', alpha=0.25)
    
    plt.plot(COMMON_TIME, high_ca_mean, color='red', linewidth=2,
             label=f'4mM Ca (n={len(high_ca_traces)})')
    plt.fill_between(COMMON_TIME, high_ca_mean - high_ca_sem, high_ca_mean + high_ca_sem, 
                     color='red', alpha=0.25)

    # Add stimulus markers (every 100ms from 1.0s)
    stim_times = [1.0 + 0.1*i for i in range(10)]
    for stim_time in stim_times:
        if stim_time <= 2.0:
            plt.axvline(stim_time, color='gray', linestyle='--', alpha=0.4, linewidth=1)
    
    plt.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    plt.xlim(0.5, 2.0)
    plt.xlabel('Time (s)')
    plt.ylabel('ΔF/F')
    plt.title('Mean Traces: 1.5mM vs 4mM Calcium')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "calcium_mean_traces_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Calcium trace comparison: 1.5mM (n={len(low_ca_traces)}) vs 4mM (n={len(high_ca_traces)})")
    print(f"✓ Saved to {output_file}")

### H.7 Calcium Correlation Structure

We correlate PPR ratios with amplitude and failure metrics under each calcium condition to see how release probability and synaptic reliability interact with plasticity when calcium is limiting.


In [ ]:
# Correlation analysis: AMP1 and %Fail1 vs PPR2/1 across calcium conditions
from scipy.stats import pearsonr, t

def calcium_scatter_analysis(x_param, y_param='PPR2/1'):
    """Create scatter plot with regression analysis for calcium conditions."""
    
    # Extract data for both conditions
    low_ca_data  = PCA_Data_WT_Low_Ca[[x_param, y_param]].dropna()
    high_ca_data = PCA_Data_WT_High_Ca[[x_param, y_param]].dropna()
    
    x_low, y_low   = low_ca_data[x_param].values, low_ca_data[y_param].values
    x_high, y_high = high_ca_data[x_param].values, high_ca_data[y_param].values
    
    # Calculate separate correlations
    r_low, p_low   = pearsonr(x_low, y_low) if len(x_low) > 1 else (float('nan'), float('nan'))
    r_high, p_high = pearsonr(x_high, y_high) if len(x_high) > 1 else (float('nan'), float('nan'))
    
    # Pooled analysis
    x_pool = np.concatenate([x_low, x_high])
    y_pool = np.concatenate([y_low, y_high])
    
    if len(x_pool) > 2:
        slope, intercept = np.polyfit(x_pool, y_pool, 1)
        r_pool, p_pool   = pearsonr(x_pool, y_pool)
        
        # Calculate confidence intervals
        x_grid = np.linspace(x_pool.min(), x_pool.max(), 100)
        y_fit  = intercept + slope * x_grid
        
        # Simplified CI calculation
        residuals = y_pool - (intercept + slope * x_pool)
        mse       = np.sum(residuals**2) / (len(x_pool) - 2)
        se        = np.sqrt(mse)
        
        t_crit = t.ppf(0.975, len(x_pool) - 2)
        margin = t_crit * se
        
    else:
        r_pool = p_pool = float('nan')
        x_grid = y_fit = margin = None
    
    # Create plot
    plt.figure(figsize=(7, 5))
    plt.scatter(x_low, y_low, c='blue', alpha=0.7, edgecolor='black', s=60, 
               label=f'1.5mM Ca (n={len(x_low)})')
    plt.scatter(x_high, y_high, c='red', alpha=0.7, edgecolor='black', s=60,
               label=f'4mM Ca (n={len(x_high)})')
    
    # Add regression line and confidence band
    if x_grid is not None:
        plt.plot(x_grid, y_fit, color='black', linewidth=2, label='Pooled regression')
        plt.fill_between(x_grid, y_fit - margin, y_fit + margin, 
                        color='black', alpha=0.15, label='95% CI')
    
    # Format plot
    plt.xlabel(x_param)
    plt.ylabel(y_param)
    plt.title(f'{x_param} vs {y_param}: Calcium Comparison')
    plt.grid(True, alpha=0.3)
    
    # Set reasonable axis limits
    if x_param == 'AMP1':
        plt.xlim(0, max(3, x_pool.max() * 1.1))
    plt.ylim(0, max(3, y_pool.max() * 1.1))
    
    # Add correlation statistics
    stats_text = (f"1.5mM: r={r_low:.2f}, p={p_low:.2g}\n"
                  f"4mM: r={r_high:.2f}, p={p_high:.2g}\n"
                  f"Pooled: r={r_pool:.2f}, p={p_pool:.2g}")
    plt.text(0.02, 0.98, stats_text, transform=plt.gca().transAxes, 
             va='top', fontsize=9, 
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.legend()
    plt.tight_layout()
    
    # Save results
    safe_param = x_param.replace('%', 'pct').replace('/', '_')
    output_file = OUTPUT_DIR / f"scatter_{safe_param}_vs_ppr2_1_calcium.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Save statistics
    stats_file = OUTPUT_DIR / f"correlation_{safe_param}_vs_ppr2_1_stats.txt"
    with open(stats_file, 'w') as f:
        f.write(f"Correlation: {x_param} vs {y_param}\n")
        f.write(f"1.5mM Ca: r={r_low:.4f}, p={p_low:.6g}, n={len(x_low)}\n")
        f.write(f"4mM Ca: r={r_high:.4f}, p={p_high:.6g}, n={len(x_high)}\n")
        f.write(f"Pooled: r={r_pool:.4f}, p={p_pool:.6g}, n={len(x_pool)}\n")
    
    print(f"✓ {x_param} vs {y_param}: {stats_text.replace(chr(10), ' | ')}")
    return output_file

# Run both analyses
amp1_output = calcium_scatter_analysis('AMP1')
fail1_output = calcium_scatter_analysis('%Fail1')

print(f"\n✓ Scatter analyses complete:")
print(f"  AMP1 vs PPR2/1: {amp1_output}")
print(f"  %Fail1 vs PPR2/1: {fail1_output}")

### H.8 Trial-Level Amplitude Distributions

Per-trial amplitude histograms are modeled to dissect how success and failure amplitudes diverge under different calcium concentrations, offering a granular view of release variability.


In [ ]:
# Analyze trial-level amplitude distributions using existing trials data
from scipy.stats import norm

def analyze_trial_amplitudes(fit_gaussian=False):
    """Analyze trial amplitude distributions from PPR_TRIALS_FILENAME."""
    
    # Load trials data using existing path structure
    trials_file = BASE_DIR / PPR_TRIALS_FILENAME
    
    try:
        trials = pd.read_excel(trials_file)
    except FileNotFoundError:
        print(f"Trials file not found: {trials_file}")
        return
    
    # Set column names based on the structure you provided
    trials.columns = ['AMP1', 'status', 'file', 'folder', 'trial']
    
    # Clean data
    trials['AMP1']   = pd.to_numeric(trials['AMP1'], errors='coerce')
    trials['status'] = trials['status'].astype(str).str.lower().str.strip()
    trials['folder'] = trials['folder'].astype(str).str.strip()
    
    # Filter for calcium conditions
    calcium_conditions = {'Theo_4Ca', 'Theo_1_5Ca'}
    trials_filtered = trials[
        trials['folder'].isin(calcium_conditions) &
        trials['status'].isin(['success', 'failure'])
    ].dropna(subset=['AMP1']).copy()
    
    if len(trials_filtered) == 0:
        print("No calcium trial data found")
        available_conditions = trials['folder'].unique()
        print(f"Available conditions: {list(available_conditions)}")
        return
    
    # Create categories
    def categorize_trial(row):
        if row['status'] == 'failure':
            return 'Failures (both Ca)'
        return f"Success {row['folder']}"
    
    trials_filtered['Category'] = trials_filtered.apply(categorize_trial, axis=1)
    
    # Set up plotting
    categories = ['Success Theo_4Ca', 'Success Theo_1_5Ca', 'Failures (both Ca)']
    colors = {'Success Theo_4Ca': 'red', 'Success Theo_1_5Ca': 'blue', 'Failures (both Ca)': 'green'}
    
    # Create bins
    amp_range = trials_filtered['AMP1']
    bins      = np.linspace(amp_range.min(), amp_range.max(), 81)
    bin_width = bins[1] - bins[0]
    
    plt.figure(figsize=(8, 6))
    fit_results = []
    
    for category in categories:
        subset = trials_filtered[trials_filtered['Category'] == category]
        if len(subset) == 0:
            continue
        
        amplitudes = subset['AMP1'].values
        weights = np.ones(len(amplitudes)) * (100.0 / len(amplitudes))
        
        # Plot histogram
        plt.hist(amplitudes, bins=bins, weights=weights, alpha=0.6, 
                color=colors[category], label=f"{category} (n={len(amplitudes)})",
                edgecolor='black', linewidth=0.3)
        
        # Add Gaussian fit if requested
        if fit_gaussian and len(amplitudes) > 1:
            mu, sigma   = norm.fit(amplitudes)
            bin_centers = (bins[:-1] + bins[1:]) / 2
            pdf_scaled  = norm.pdf(bin_centers, mu, sigma) * (bin_width * 100)
            plt.plot(bin_centers, pdf_scaled, color=colors[category], linewidth=2)
            
            # Mark peak
            y_peak = norm.pdf(mu, mu, sigma) * (bin_width * 100)
            plt.text(mu, y_peak * 1.02, f"{mu:.2f}", color='black',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
            
            fit_results.append({
                'Category': category,
                'mu': mu,
                'sigma': sigma,
                'n': len(amplitudes)
            })
    
    # Format plot
    plt.xlabel('AMP1 (Trial Amplitude)')
    plt.ylabel('Proportion (%)')
    title = 'Trial Amplitude Distributions: Calcium Conditions'
    if fit_gaussian:
        title += ' + Gaussian Fits'
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    suffix = "_gaussian_fits" if fit_gaussian else ""
    output_file = OUTPUT_DIR / f"trial_amplitudes_calcium{suffix}.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Display results
    print(f"Trial amplitude analysis:")
    for category in categories:
        subset = trials_filtered[trials_filtered['Category'] == category]
        if len(subset) > 0:
            mean_amp = subset['AMP1'].mean()
            print(f"  {category}: {len(subset)} trials (mean: {mean_amp:.3f})")
    
    if fit_gaussian and fit_results:
        print("\nGaussian fit parameters:")
        for result in fit_results:
            print(f"  {result['Category']}: μ={result['mu']:.3f}, σ={result['sigma']:.3f}")
    
    print(f"✓ Saved plot to {output_file}")
    return trials_filtered

# Run analyses
trials_basic = analyze_trial_amplitudes(fit_gaussian=False)
trials_fitted = analyze_trial_amplitudes(fit_gaussian=True)

## Chapter I - Stability

Chapter I examines the temporal stability of synaptic properties by comparing boutons imaged before and after a 8min interval.

### I.1 Stability Analysis Configuration

Parameter knobs and helper structures are established to analyze before-versus-after stability experiments. These controls govern how strictly clusters are defined in subsequent comparisons.


In [ ]:
# ==== Cell 0 — Config + tiny helpers (set your "edge" controls here) ====

ELLIPSE_ALPHA   = 0.95     # ellipse containment level
ALPHA_EXPANSION = 0.50     # alpha-shape expansion factor (0.0 → off)
KNN_K           = None     # k-NN neighbors (None → auto √N)

import numpy as np
from scipy.stats import chi2

def build_ellipse_models(X, y, n_clusters, alpha=0.95, ridge=1e-6):
    """Mean/cov/inv and chi2 threshold for each cluster."""
    thr = chi2.ppf(alpha, df=2)
    models = []
    for k in range(1, n_clusters+1):
        pts = X[y==k]
        if len(pts) > 2:
            mu  = pts.mean(axis=0)
            cov = np.cov(pts.T) + np.eye(2)*ridge
            inv = np.linalg.pinv(cov)
            models.append({'cluster': k, 'center': mu, 'cov': cov, 'inv_cov': inv})
    return models, thr

def mahalanobis_sq(x, mu, inv):
    d = x - mu
    return float(d.T @ inv @ d)

def ellipses_containing(x, models, thr):
    return {m['cluster'] for m in models if mahalanobis_sq(x, m['center'], m['inv_cov']) <= thr}

def nearest_ellipse_edge(x, models, thr):
    """Return (cluster_id, euclid_dist_to_edge)."""
    best = (None, np.inf)
    for m in models:
        md2 = mahalanobis_sq(x, m['center'], m['inv_cov'])
        if md2 <= thr:
            return m['cluster'], 0.0
        s = np.sqrt(thr/md2)
        x_proj = m['center'] + s*(x - m['center'])
        dist = float(np.linalg.norm(x - x_proj))
        if dist < best[1]:
            best = (m['cluster'], dist)
    return best


### I.2 Stability Trajectories

This visualization tracks how individual boutons move through PCA space from the baseline to the post-manipulation state, summarizing trajectory lengths and directionality.


In [ ]:
# ==== Cell 1 — Before/After trajectories + summary ====

import matplotlib.pyplot as plt

# Data
before_coords = np.asarray(pca_data['stab_before'])
after_coords  = np.asarray(pca_data['stab_after'])
n_pairs       = int(min(len(before_coords), len(after_coords)))
assert n_pairs > 0, "No paired before/after points."

# Plot
plt.figure(figsize=(8,6))
plt.scatter(pca_coordinates[:,0], pca_coordinates[:,1], c=get_cluster_colors(cluster_assignments), s=20, alpha=0.4, label='WT background')
plt.scatter(before_coords[:,0], before_coords[:,1], c='orange', s=60, edgecolors='darkorange', label=f'Before (n={len(before_coords)})')
plt.scatter(after_coords[:,0],  after_coords[:,1],  c='brown',  s=60, edgecolors='darkred',   label=f'After  (n={len(after_coords)})')

# Pair links + mean arrow
moves = []
for i in range(n_pairs):
    plt.plot([before_coords[i,0], after_coords[i,0]],
             [before_coords[i,1], after_coords[i,1]], color='gray', alpha=0.6, lw=1)
    moves.append(float(np.linalg.norm(after_coords[i] - before_coords[i])))

diffs    = after_coords[:n_pairs] - before_coords[:n_pairs]
mean_vec = diffs.mean(axis=0)
center   = np.vstack([before_coords[:n_pairs], after_coords[:n_pairs]]).mean(axis=0)
plt.arrow(center[0], center[1], mean_vec[0], mean_vec[1], color='black',
          width=0.05, head_width=0.25, head_length=0.25, length_includes_head=True, label='Mean trajectory')

pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%})'); plt.ylabel(f'PC2 ({pc2_variance:.1%})')
plt.title('Stability: Before vs After'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
out = OUTPUT_DIR / "stability_before_after_trajectories.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

print("=== STABILITY TRAJECTORY ANALYSIS ===")
print(f"Pairs: {n_pairs}")
print(f"Movement (mean±SD): {np.mean(moves):.3f} ± {np.std(moves):.3f}")
print(f"Mean trajectory |mag|: {np.linalg.norm(mean_vec):.3f}  dir=({mean_vec[0]:.3f}, {mean_vec[1]:.3f})")
print(f"✓ Saved {out}")


### I.3 Stability Movement Histogram

We compile a histogram of bouton displacements to quantify how much synaptic properties drift between the before and after conditions.


In [ ]:
# ==== Cell 2 — Movement distance histogram (auto bins) ====

# Freedman–Diaconis binning with fallback
md      = np.linalg.norm(diffs, axis=1).astype(float)
md      = md[np.isfinite(md)]
q25,q75 = np.percentile(md,[25,75]); iqr=float(q75-q25); n=len(md)
bw      = (2*iqr)/(n**(1/3)) if iqr>0 else 0.0
bins    = max(5, int(np.ceil((md.max()-md.min())/bw))) if bw>0 else max(5, int(np.ceil(np.sqrt(n))))

plt.figure(figsize=(6,4))
plt.hist(md, bins=bins, edgecolor='black', alpha=0.85)
plt.axvline(md.mean(), ls='--', lw=2, label=f'Mean = {md.mean():.2f}')
plt.xlabel('Distance in PCA space'); plt.ylabel('Count'); plt.title('Before→After distances'); plt.legend(); plt.tight_layout()
out = OUTPUT_DIR / "stability_movement_distances.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

print(f"n={n}  mean={md.mean():.3f}  sd={md.std(ddof=1):.3f}  median={np.median(md):.3f}  min={md.min():.3f}  max={md.max():.3f}  bins={bins}")
print(f"✓ Saved {out}")


### I.4 Stability Trace Evolution

Mean traces with confidence intervals are plotted for before and after recordings, revealing how waveform kinetics change with the stability manipulation.


In [ ]:
# ==== Cell 3 — Mean traces (±SEM) for Stability_Before/_05 vs Stability_After/_05 ====

assert 'NORM_TRACES_DATAFRAME' in locals() and 'COMMON_TIME' in locals()

import pandas as pd

def stack_traces(df, cond):
    rows = df[df['Condition']==cond]
    X    = [np.asarray(r['Avg'], float) for _,r in rows.iterrows()]
    X    = [t for t in X if np.isfinite(t).all() and len(t)==len(COMMON_TIME)]
    return (np.vstack(X) if len(X)>0 else np.empty((0,len(COMMON_TIME)))), len(X)

def mean_sem(X):
    if X.size==0: 
        z = np.zeros(len(COMMON_TIME)); return z,z
    m = np.nanmean(X, axis=0)
    s = np.nanstd(X, axis=0, ddof=1)/np.sqrt(max(1,X.shape[0]))
    return m,s

conds   = ["Stability_Before","Stability_Before_05","Stability_After","Stability_After_05"]
stacked = {c: stack_traces(NORM_TRACES_DATAFRAME,c) for c in conds}
stats   = {c: mean_sem(stacked[c][0]) for c in conds}
counts  = {c: stacked[c][1] for c in conds}

colors = {"Stability_Before":"#1f77b4","Stability_Before_05":"#1f77b4","Stability_After":"#d62728","Stability_After_05":"#d62728"}
styles = {"Stability_Before":('-',2.0),"Stability_Before_05":('--',1.8),"Stability_After":('-',2.0),"Stability_After_05":('--',1.8)}

plt.figure(figsize=(8.5,5.0))
for c in conds:
    mean,sem = stats[c]; ls,lw = styles[c]
    plt.plot(COMMON_TIME, mean, color=colors[c], ls=ls, lw=lw, label=f"{c} (n={counts[c]})")
    plt.fill_between(COMMON_TIME, mean-sem, mean+sem, color=colors[c], alpha=0.15, lw=0)
plt.axhline(0, color='gray', ls=':', lw=1.0)
plt.axvspan(0.5, 2.0, color='gray', alpha=0.08, label='0.5–2.0 s')
plt.xlabel('Time (s)'); plt.ylabel('ΔF/F'); plt.title('Stability conditions: mean traces (±SEM)')
plt.legend(ncol=2, fontsize=9, frameon=True); plt.tight_layout()
out = OUTPUT_DIR / "stability_mean_traces_before_after.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
print("Counts:", {c:counts[c] for c in conds}); print(f"✓ Saved {out}")


### I.5 Stability Ellipse Boundaries

Elliptical decision boundaries are applied in PCA space to evaluate which boutons remain within the WT tolerance zone after the manipulation, providing a geometric perspective on stability.


In [ ]:
# ===== Cell 4 — Ellipses: PCA with hard edge + tolerance zone, and pie =====
# knobs
ELLIPSE_ALPHA = 0.8        # hard edge level
ELLIPSE_TOL   = 1.0       # inflate ellipse threshold by (1 + ELLIPSE_TOL)

import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# --- build models ---
def _ellipse_models(X, y, n_clusters, ridge=1e-6):
    models=[]
    for k in range(1, n_clusters+1):
        pts = X[y==k]
        if len(pts)>2:
            mu=pts.mean(axis=0); cov=np.cov(pts.T)+np.eye(2)*ridge; inv=np.linalg.pinv(cov)
            models.append({'cluster':k,'center':mu,'cov':cov,'inv_cov':inv})
    return models
def _md2(x, m): 
    d=x-m['center']; return float(d.T @ m['inv_cov'] @ d)

ellipse_models = _ellipse_models(pca_coordinates, cluster_assignments, N_CLUSTERS)
thr_hard       = chi2.ppf(ELLIPSE_ALPHA, df=2)
thr_tol        = thr_hard*(1.0 + ELLIPSE_TOL)
print(f"Thresholds - Hard: {thr_hard:.2f}, Tolerance: {thr_tol:.2f}")

# --- plot PCA with hard + tolerance ---
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(pca_coordinates[:,0], pca_coordinates[:,1], c=get_cluster_colors(cluster_assignments), s=20, alpha=0.35, label='WT')
uniq = np.unique(cluster_assignments)

def _draw_ellipse(ax, mean, cov, thr, color, fa, lw, ls='-'):
    U,s,_=np.linalg.svd(cov); ang=np.degrees(np.arctan2(U[1,0],U[0,0])); w,h=2*np.sqrt(thr*s)
    ax.add_patch(Ellipse(mean, w, h, angle=ang, facecolor=color, edgecolor=color, alpha=fa, lw=lw, ls=ls))

for cid in uniq:
    pts = pca_coordinates[cluster_assignments==cid]
    if len(pts)<3: continue
    m     = [mm for mm in ellipse_models if mm['cluster']==cid][0]
    color = get_cluster_color(cid)
    # tolerance ring: draw tol (filled light), then hard (outline)
    _draw_ellipse(ax, m['center'], m['cov'], thr_tol,  color, 0.10, 0.0)    # tol zone
    _draw_ellipse(ax, m['center'], m['cov'], thr_hard, color, 0.00, 2.0)    # hard edge

# overlay pairs
n_pairs = int(min(len(pca_data['stab_before']), len(pca_data['stab_after'])))
before  = np.asarray(pca_data['stab_before'])[:n_pairs]
after   = np.asarray(pca_data['stab_after'])[:n_pairs]

# Calculate stability for line colors
def _label_all_for_plot(x, thr):
    """Return all clusters that contain point x (for plotting)"""
    inside = [m['cluster'] for m in ellipse_models if _md2(x, m) <= thr]
    if inside:
        return set(inside)
    # If not inside any ellipse, assign to nearest-edge cluster
    d       = [(m['cluster'], np.sqrt(_md2(x, m)/thr)-1.0) for m in ellipse_models]
    nearest = min(d, key=lambda t: t[1])[0]
    return {nearest}

# Draw connecting lines with appropriate colors
for i in range(n_pairs):
    b_clusters = _label_all_for_plot(before[i], thr_tol)  
    a_clusters = _label_all_for_plot(after[i], thr_tol)   
    is_stable  = len(b_clusters & a_clusters) > 0
    
    line_color = 'gray' if is_stable else 'red'
    line_alpha = 0.6 if is_stable else 0.8
    line_width = 1 if is_stable else 1.2
    
    ax.plot([before[i,0], after[i,0]], [before[i,1], after[i,1]], 
            color=line_color, alpha=line_alpha, lw=line_width)

ax.scatter(before[:,0], before[:,1], c='orange', s=60, edgecolors='k', lw=0.5, label='Before')
ax.scatter(after[:,0],  after[:,1],  c='brown',  s=60, marker='s', edgecolors='k', lw=0.5, label='After')

# Add legend entries for line colors
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='gray', alpha=0.6, lw=1, label='Stable pairs'),
    Line2D([0], [0], color='red', alpha=0.8, lw=1.2, label='Unstable pairs')
]

pc1,pc2 = PCA_RESULTS['explained_variance']; ax.set_xlabel(f'PC1 ({pc1:.1%})'); ax.set_ylabel(f'PC2 ({pc2:.1%})')
ax.set_title(f'Ellipses: hard α={ELLIPSE_ALPHA:.2f}, tol +{ELLIPSE_TOL*100:.0f}% (Conservative)')

# Combine existing legend with line color legend
handles, labels = ax.get_legend_handles_labels()
handles.extend(legend_elements)
ax.legend(handles=handles); ax.grid(alpha=0.3); plt.tight_layout()
out = OUTPUT_DIR / "ellipses_pca_hard_tol.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

# --- Conservative stability approach (hard-edge only) ---
def _inside_any(x, thr): 
    return any(_md2(x,m) <= thr for m in ellipse_models)

def _label_all(x, thr):
    """Return all clusters that contain point x (conservative approach)"""
    inside = [m['cluster'] for m in ellipse_models if _md2(x, m) <= thr]
    if inside:
        return set(inside)
    # If not inside any ellipse, assign to nearest-edge cluster
    d       = [(m['cluster'], np.sqrt(_md2(x, m)/thr)-1.0) for m in ellipse_models]
    nearest = min(d, key=lambda t: t[1])[0]
    return {nearest}

# Get all cluster memberships for each point
b_labs = [_label_all(before[i], thr_tol) for i in range(n_pairs)]
a_labs = [_label_all(after[i], thr_tol) for i in range(n_pairs)]

# Conservative stability: stable if any overlap between before and after cluster sets
stable   = sum(1 for i in range(n_pairs) if len(b_labs[i] & a_labs[i]) > 0)
unstable = n_pairs - stable

plt.figure(figsize=(6.3,5.8))
plt.pie([stable,unstable], labels=[f"Stable ({stable})", f"Unstable ({unstable})"],
        colors=['#4CAF50','#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
plt.title('Conservative Ellipse Stability (hard-edge)'); plt.tight_layout()
out = OUTPUT_DIR / "ellipses_stability_pie.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
print(f"[Ellipses] Conservative: hard α={ELLIPSE_ALPHA:.2f}  stable={stable}/{n_pairs} ({100*stable/n_pairs:.1f}%)  ✓ {out}")

# Optional: Print details about overlapping cases for verification
overlap_cases = []
for i in range(n_pairs):
    if len(b_labs[i] & a_labs[i]) > 0 and (len(b_labs[i]) > 1 or len(a_labs[i]) > 1):
        overlap_cases.append((i, b_labs[i], a_labs[i], b_labs[i] & a_labs[i]))

if overlap_cases:
    print(f"Found {len(overlap_cases)} cases with ellipse overlaps:")
    for i, before_clusters, after_clusters, shared in overlap_cases[:5]:  # Show first 5
        print(f"  Pair {i}: Before={before_clusters}, After={after_clusters}, Shared={shared}")
    if len(overlap_cases) > 5:
        print(f"  ... and {len(overlap_cases)-5} more cases")


### I.6 Stability Alpha Shapes

Alpha-shape contours supply a non-parametric boundary around WT clusters, allowing us to test whether post-manipulation boutons exit the original manifold.


In [ ]:
# ===== Cell — Alpha-shapes: PCA with hard polygon + tolerance zone, and pie =====
# knobs
ALPHA_EXPANSION = 2      # tolerance zone: expanded by sqrt(1+exp)
ALPHA_KNN_Q     = 0.2       # alpha heuristic quantile (0.8 for very small n)

import numpy as np, matplotlib.pyplot as plt, alphashape
from shapely.affinity import scale as shp_scale
from shapely.geometry import MultiPoint, Polygon as ShapelyPolygon, MultiPolygon, Point

def _alpha_for(pts):
    n=len(pts); d2=np.sum((pts[:,None,:]-pts[None,:,:])**2, axis=2); np.fill_diagonal(d2, np.inf)
    kth=np.partition(d2,1,axis=1)[:,1]; base=np.sqrt(kth)
    return float(1.5*np.quantile(base, ALPHA_KNN_Q if n>=10 else 0.8))

def _make_alpha_shapes(X,y,expansion):
    s = float(np.sqrt(1.0+expansion)) if expansion>0 else 1.0
    res={}
    for cid in np.unique(y):
        pts = X[y==cid]
        if len(pts)<3: res[cid]=None; continue
        a=_alpha_for(pts)
        poly = alphashape.alphashape([tuple(r) for r in pts], a)
        if poly is None or getattr(poly,'is_empty',True): poly = MultiPoint([tuple(r) for r in pts]).convex_hull
        res[cid]={'hard':poly, 'tol': (shp_scale(poly, xfact=s, yfact=s, origin='centroid') if expansion>0 else poly)}
    return res

shapes = _make_alpha_shapes(pca_coordinates, cluster_assignments, ALPHA_EXPANSION)
# --- PCA: draw tol (light fill) + hard (outline) ---
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(pca_coordinates[:,0], pca_coordinates[:,1], c=get_cluster_colors(cluster_assignments), s=20, alpha=0.35, label='WT')
for cid,sh in shapes.items():
    if not sh: continue
    for tag, (fa, ls, lw) in dict(tol=(0.10,'-',0.0), hard=(0.00,'-',2.0)).items():
        g = sh[tag]
        geoms=[g] if g.geom_type=='Polygon' else (list(g.geoms) if g.geom_type=='MultiPolygon' else [])
        for gg in geoms:
            X,Y = np.array(gg.exterior.coords).T
            if fa>0: ax.fill(X,Y,color=get_cluster_color(cid),alpha=fa)
            ax.plot(X,Y,color=get_cluster_color(cid),ls=ls,lw=lw)

# pairs with conservative coloring
n_pairs = int(min(len(pca_data['stab_before']), len(pca_data['stab_after'])))
before = np.asarray(pca_data['stab_before'])[:n_pairs]
after  = np.asarray(pca_data['stab_after'])[:n_pairs]

# Conservative approach: find ALL clusters that contain each point
def _in_shape_all(x, use_tol=False, paired_point=None, paired_clusters=None):
    """Return set of all clusters that contain point x (conservative approach)"""
    p = Point(float(x[0]), float(x[1]))
    clusters = set()
    
    # Check containment in all shapes
    for cid, sh in shapes.items():
        if sh is None: continue
        g = sh['tol'] if use_tol else sh['hard']
        if g.geom_type == 'Polygon':
            if g.contains(p):
                clusters.add(cid)
        elif g.geom_type == 'MultiPolygon':
            if any(gg.contains(p) for gg in g.geoms):
                clusters.add(cid)
    
    # If not inside any shape, assign to nearest boundary
    if not clusters:
        distances = []
        for cid, sh in shapes.items():
            if sh is None: continue
            g = sh['tol'] if use_tol else sh['hard']
            dist = p.distance(g)
            distances.append((cid, dist))
        
        if distances:
            # If paired point is inside shapes, prefer those clusters when distances are close
            if paired_clusters:
                # Find distances to paired clusters
                paired_dists = [(cid, d) for cid, d in distances if cid in paired_clusters]
                if paired_dists:
                    min_paired_dist = min(paired_dists, key=lambda t: t[1])[1]
                    overall_min_dist = min(distances, key=lambda t: t[1])[1]
                    # If paired cluster is within 20% of nearest, prefer it
                    if min_paired_dist <= overall_min_dist * 1.2:
                        nearest_cid = min(paired_dists, key=lambda t: t[1])[0]
                        clusters = {nearest_cid}
                        return clusters
            
            # Otherwise use nearest
            nearest_cid = min(distances, key=lambda t: t[1])[0]
            clusters = {nearest_cid}
    
    return clusters

# Draw connecting lines with appropriate colors
for i in range(n_pairs):
    b_clusters = _in_shape_all(before[i], use_tol=True)
    a_clusters = _in_shape_all(after[i], use_tol=True)
    
    # Check stability
    is_stable = len(b_clusters & a_clusters) > 0
    
    line_color = 'gray' if is_stable else 'red'
    line_alpha = 0.6 if is_stable else 0.8
    line_width = 1 if is_stable else 1.2
    
    ax.plot([before[i,0], after[i,0]], [before[i,1], after[i,1]], 
            color=line_color, alpha=line_alpha, lw=line_width)

ax.scatter(before[:,0], before[:,1], c='orange', s=60, edgecolors='k', lw=0.5, label='Before')
ax.scatter(after[:,0],  after[:,1],  c='brown',  s=60, marker='s', edgecolors='k', lw=0.5, label='After')

# Add legend entries for line colors
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='gray', alpha=0.6, lw=1, label='Stable pairs'),
    Line2D([0], [0], color='red', alpha=0.8, lw=1.2, label='Unstable pairs')
]

pc1,pc2 = PCA_RESULTS['explained_variance']; ax.set_xlabel(f'PC1 ({pc1:.1%})'); ax.set_ylabel(f'PC2 ({pc2:.1%})')
ax.set_title(f'Alpha-shapes: hard + tol (exp={ALPHA_EXPANSION:.2f}) (Conservative)')

# Combine existing legend with line color legend
handles, labels = ax.get_legend_handles_labels()
handles.extend(legend_elements)
ax.legend(handles=handles); ax.grid(alpha=0.3); plt.tight_layout()

out = OUTPUT_DIR / "alphashapes_pca_hard_tol.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

# --- Conservative stability (tolerance boundary) ---
b_clusters_all = [_in_shape_all(before[i], use_tol=True) for i in range(n_pairs)]
a_clusters_all = [_in_shape_all(after[i], use_tol=True) for i in range(n_pairs)]

# Conservative stability: stable if any overlap between before and after cluster sets
stable = sum(1 for i in range(n_pairs) if len(b_clusters_all[i] & a_clusters_all[i]) > 0)
unstable = n_pairs - stable

plt.figure(figsize=(6.3,5.8))
plt.pie([stable,unstable], labels=[f"Stable ({stable})", f"Unstable ({unstable})"],
        colors=['#4CAF50','#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
plt.title(f'Conservative Alpha-shape Stability (exp={ALPHA_EXPANSION:.2f})'); plt.tight_layout()
out = OUTPUT_DIR / "alphashapes_stability_pie.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
print(f"[Alpha] Conservative: exp={ALPHA_EXPANSION:.2f}  stable={stable}/{n_pairs} ({100*stable/n_pairs:.1f}%)  ✓ {out}")

# Optional: Print details about overlapping cases for verification
overlap_cases = []
for i in range(n_pairs):
    if len(b_clusters_all[i] & a_clusters_all[i]) > 0 and (len(b_clusters_all[i]) > 1 or len(a_clusters_all[i]) > 1):
        overlap_cases.append((i, b_clusters_all[i], a_clusters_all[i], b_clusters_all[i] & a_clusters_all[i]))

# After the stability calculation, add this diagnostic
print("\nDiagnostic for UNSTABLE pairs (red lines):")
unstable_indices = [i for i in range(n_pairs) if len(b_clusters_all[i] & a_clusters_all[i]) == 0]

for idx in unstable_indices[:10]:  # Show first 10 unstable pairs
    b_pos = before[idx]
    a_pos = after[idx]
    b_clusters = b_clusters_all[idx]
    a_clusters = a_clusters_all[idx]
    
    print(f"\nPair {idx}: UNSTABLE")
    print(f"  Before {b_pos}: assigned to clusters={b_clusters}")
    print(f"  After  {a_pos}: assigned to clusters={a_clusters}")
    
    # Show distances to all shape boundaries
    for cid, sh in shapes.items():
        if sh is None: continue
        g = sh['tol']
        p_before = Point(float(b_pos[0]), float(b_pos[1]))
        p_after = Point(float(a_pos[0]), float(a_pos[1]))
        b_dist = p_before.distance(g)
        a_dist = p_after.distance(g)
        print(f"    Cluster {cid}: Before dist={b_dist:.3f}, After dist={a_dist:.3f}")

# After creating shapes, verify tolerance expansion
print("\nVerifying tolerance zone sizes:")
for cid, sh in shapes.items():
    if sh is None: continue
    hard_area = sh['hard'].area
    tol_area = sh['tol'].area
    expansion_ratio = np.sqrt(tol_area / hard_area)
    print(f"Cluster {cid}: area ratio = {expansion_ratio:.3f} (expected: {np.sqrt(1+ALPHA_EXPANSION):.3f})")
if overlap_cases:
    print(f"Found {len(overlap_cases)} cases with alpha-shape overlaps:")
    for i, before_clusters, after_clusters, shared in overlap_cases[:5]:  # Show first 5
        print(f"  Pair {i}: Before={before_clusters}, After={after_clusters}, Shared={shared}")
    if len(overlap_cases) > 5:
        print(f"  ... and {len(overlap_cases)-5} more cases")



### I.7 Stability Bootstrap Analysis

Bootstrap resampling compares observed bouton shifts to random expectations and visualizes representative random pairs, strengthening conclusions about genuine remodeling.


In [ ]:
# ==== Cell 7 — Random bootstrap + (NEW) distance comparison + PCA overlay + random-pair mean traces ====
# knobs
BOOT_ITERS       = 2000     # bootstrap iterations
RANDOM_SEED      = 41       # RNG seed
SHOW_PAIRS_PLOT  = 50       # max random pairs to draw on PCA

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2

# --- ellipse models (reuse from earlier cells or rebuild with Cell 0 helper) ---
def _ensure_ellipse_models():
    if 'ellipse_models' in locals() and 'chi2_thr' in locals():
        return ellipse_models, chi2_thr
    return build_ellipse_models(pca_coordinates, cluster_assignments, N_CLUSTERS, alpha=ELLIPSE_ALPHA)

ellipse_models, chi2_thr = _ensure_ellipse_models()

def _ellipses_containing(x):
    # mahalanobis containment check
    hit = set()
    for m in ellipse_models:
        d = x - m['center']
        if float(d.T @ m['inv_cov'] @ d) <= chi2_thr:
            hit.add(m['cluster'])
    return hit

def _bootstrap_random_pairs(n_pairs_in, iters=2000, seed=42):
    rng = np.random.default_rng(seed)
    X = np.asarray(pca_coordinates, float); N=len(X)
    n_eff = int(min(n_pairs_in, N//2))
    if n_eff <= 0: raise RuntimeError("[bootstrap] Not enough WT points to form random pairs.")
    props = np.empty(iters, float)
    # keep one pairing for visualization & distance comparison
    perm = rng.permutation(N)
    idx1_vis = perm[:n_eff]; idx2_vis = perm[n_eff:2*n_eff]
    for t in range(iters):
        perm = rng.permutation(N)
        idx1 = perm[:n_eff]; idx2 = perm[n_eff:2*n_eff]
        shared = 0
        for i1, i2 in zip(idx1, idx2):
            if _ellipses_containing(X[i1]).intersection(_ellipses_containing(X[i2])):
                shared += 1
        props[t] = shared / n_eff
    return props, n_eff, (idx1_vis, idx2_vis)

# observed stability (ellipses, hard)
before_coords = np.asarray(pca_data['stab_before'])
after_coords  = np.asarray(pca_data['stab_after'])
n_pairs = int(min(len(before_coords), len(after_coords)))

def _hard_label(x):
    inside=[(m['cluster'], float((x-m['center']).T @ m['inv_cov'] @ (x-m['center'])))
            for m in ellipse_models if float((x-m['center']).T @ m['inv_cov'] @ (x-m['center'])) <= chi2_thr]
    if inside: return min(inside, key=lambda t:t[1])[0]
    # nearest-edge proxy by boundary gap
    gaps=[(m['cluster'], np.sqrt(float((x-m['center']).T @ m['inv_cov'] @ (x-m['center']))/chi2_thr)-1.0)
          for m in ellipse_models]
    return min(gaps, key=lambda t:t[1])[0]

b_h = np.array([_hard_label(before_coords[i]) for i in range(n_pairs)])
a_h = np.array([_hard_label(after_coords[i])  for i in range(n_pairs)])
observed_prop = float(np.sum(b_h==a_h)) / n_pairs

# --- run bootstrap ---
props, n_eff, (idx1_vis, idx2_vis) = _bootstrap_random_pairs(n_pairs, iters=BOOT_ITERS, seed=RANDOM_SEED)
p_ge  = float(np.mean(props >= observed_prop))
p_two = float(np.mean(np.abs(props - props.mean()) >= abs(observed_prop - props.mean())))

# (A0) NEW — Distance comparison: observed before→after vs random-pair distances
obs_dists = np.linalg.norm(after_coords[:n_pairs] - before_coords[:n_pairs], axis=1).astype(float)
rand_dists = np.linalg.norm(pca_coordinates[idx2_vis[:n_eff]] - pca_coordinates[idx1_vis[:n_eff]], axis=1).astype(float)

# Freedman–Diaconis bins over combined set
combined = np.concatenate([obs_dists, rand_dists])
q25,q75 = np.percentile(combined,[25,75]); iqr = float(q75-q25); N = len(combined)
bw = (2*iqr)/(N**(1/3)) if iqr>0 else 0.0
n_bins = max(8, int(np.ceil((combined.max()-combined.min())/bw))) if bw>0 else max(8, int(np.ceil(np.sqrt(N))))
bins = np.linspace(combined.min(), combined.max(), n_bins + 1)

plt.figure(figsize=(8.4,5.6))
plt.hist(rand_dists, bins=bins, alpha=0.45, edgecolor='black', label=f'Random pairs (n={len(rand_dists)})')
plt.hist(obs_dists,  bins=bins, alpha=0.45, edgecolor='black', label=f'Observed before→after (n={len(obs_dists)})')
plt.axvline(rand_dists.mean(), color='C0', ls='--', lw=2, label=f'Random mean={rand_dists.mean():.2f}')
plt.axvline(obs_dists.mean(),  color='C1', ls='--', lw=2, label=f'Observed mean={obs_dists.mean():.2f}')
plt.xlabel('Distance in PCA space'); plt.ylabel('Count'); plt.title('Distance distributions: Random vs Observed')
plt.legend(); plt.tight_layout()
out_dist = OUTPUT_DIR / "random_vs_observed_distances.pdf"
plt.savefig(out_dist, dpi=300, bbox_inches='tight'); plt.show()

# (A) PCA overlay of a subset of random pairs
max_show = min(SHOW_PAIRS_PLOT, n_eff)
x = pca_coordinates[:,0]; y = pca_coordinates[:,1]
plt.figure(figsize=(8,6))
plt.scatter(x, y, c=get_cluster_colors(cluster_assignments), s=20, alpha=0.35, label='WT background')
for i1, i2 in zip(idx1_vis[:max_show], idx2_vis[:max_show]):
    plt.plot([x[i1], x[i2]], [y[i1], y[i2]], color='tab:blue', alpha=0.6, lw=1.2)
plt.scatter(x[idx1_vis[:max_show]], y[idx1_vis[:max_show]], c='tab:blue', s=35, edgecolors='k', lw=0.3, label='Random pair A')
plt.scatter(x[idx2_vis[:max_show]], y[idx2_vis[:max_show]], c='tab:cyan',  s=35, edgecolors='k', lw=0.3, label='Random pair B', marker='s')
pc1, pc2 = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1:.1%})'); plt.ylabel(f'PC2 ({pc2:.1%})')
plt.title(f'Random WT pairs overlay (showing {max_show} of {n_eff})')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
out_pca = OUTPUT_DIR / "random_pairs_pca_overlay.pdf"
plt.savefig(out_pca, dpi=300, bbox_inches='tight'); plt.show()

# (B) Random-pair mean traces (if index→trace mapping is available)
def _get_trace_by_index(idx):
    if 'WT_IDS' in locals() and 'resampled_trace_lookup' in locals():
        _id = WT_IDS[idx]
        if _id in resampled_trace_lookup and 'Avg' in resampled_trace_lookup[_id]:
            tr = np.asarray(resampled_trace_lookup[_id]['Avg'], float)
            return tr if np.isfinite(tr).all() and len(tr)==len(COMMON_TIME) else None
    if 'WT_TRACES_ARRAY' in locals() and len(WT_TRACES_ARRAY) == len(pca_coordinates):
        tr = np.asarray(WT_TRACES_ARRAY[idx], float)
        return tr if np.isfinite(tr).all() and len(tr)==len(COMMON_TIME) else None
    return None

pair_means = []
for i1, i2 in zip(idx1_vis[:n_eff], idx2_vis[:n_eff]):
    t1 = _get_trace_by_index(i1); t2 = _get_trace_by_index(i2)
    if t1 is not None and t2 is not None:
        pair_means.append(0.5*(t1+t2))
pair_means = np.vstack(pair_means) if len(pair_means)>0 else np.empty((0, len(COMMON_TIME)))

if pair_means.size > 0:
    m = np.nanmean(pair_means, axis=0)
    s = np.nanstd(pair_means, axis=0, ddof=1)/np.sqrt(max(1, pair_means.shape[0]))
    plt.figure(figsize=(8.5,5.0))
    plt.plot(COMMON_TIME, m, lw=2.0, label=f'Random-pair mean (n={pair_means.shape[0]})')
    plt.fill_between(COMMON_TIME, m-s, m+s, alpha=0.2, lw=0)
    if 'stats' in locals():  # overlay observed means from Cell 3
        for c, col, ls in [("Stability_Before","#1f77b4","-"), ("Stability_After","#d62728","-")]:
            if c in stats:
                mm, ss = stats[c]; plt.plot(COMMON_TIME, mm, color=col, ls=ls, lw=1.5, alpha=0.9, label=f'{c} mean')
    plt.axhline(0, color='gray', ls=':', lw=1.0)
    plt.xlabel('Time (s)'); plt.ylabel('ΔF/F'); plt.title('Random-pair mean traces (±SEM)')
    plt.legend(); plt.tight_layout()
    out_tr = OUTPUT_DIR / "random_pairs_mean_traces.pdf"
    plt.savefig(out_tr, dpi=300, bbox_inches='tight'); plt.show()
else:
    print("Random-pair traces: skipped (no index→trace mapping).")

# (C) Box vs observed (assignment sharing)
plt.figure(figsize=(8.2,6.0))
plt.boxplot(props, positions=[1], patch_artist=True,
            boxprops=dict(facecolor='lightblue', alpha=0.7),
            medianprops=dict(color='navy', linewidth=2))
plt.scatter([2], [observed_prop], color='red', s=110, zorder=5, label='Observed stability')
plt.axhline(props.mean(), color='blue', linestyle='--', alpha=0.7, label='Random mean')
plt.xticks([1,2], [f'Random pairs\n({len(props)} iters)', 'Observed'])
plt.ylabel('Proportion sharing ≥1 ellipse'); plt.title('Random vs Observed stability (ellipses)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
out_box = OUTPUT_DIR / "ellipse_random_vs_observed.pdf"
plt.savefig(out_box, dpi=300, bbox_inches='tight'); plt.show()

# --- summary ---
print(f"DISTANCES: random mean={rand_dists.mean():.3f} (n={len(rand_dists)}) | observed mean={obs_dists.mean():.3f} (n={len(obs_dists)})  ✓ {out_dist}")
print(f"ASSIGNMENTS: random mean={props.mean():.3f} ± {props.std(ddof=1):.3f}, observed={observed_prop:.3f}, p>=obs={p_ge:.4f}, p(two)={p_two:.4f}")
print(f"PLOTS: PCA overlay ✓ {out_pca} | Distances ✓ {out_dist} | Box ✓ {out_box}" )



### I.8 Stability Plasticity Profiles

Paired-pulse and amplitude profiles are recalculated for before and after datasets with statistical annotations to detect systematic shifts in short-term plasticity.


In [ ]:
# Compare PPR profiles before vs after treatment
from scipy.stats import ttest_rel

def plot_stability_ppr_profile():
    """Plot PPR profile with statistical testing."""
    
    # PPR analysis
    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_Stability_Before.columns]
    pulse_numbers = list(range(1, len(ppr_cols) + 2))
    
    # Calculate PPR profiles
    ppr_before = [1.0] + PCA_Data_Stability_Before[ppr_cols].mean().tolist()
    ppr_before_sem = [0.0] + PCA_Data_Stability_Before[ppr_cols].sem().tolist()
    ppr_after = [1.0] + PCA_Data_Stability_After[ppr_cols].mean().tolist()
    ppr_after_sem = [0.0] + PCA_Data_Stability_After[ppr_cols].sem().tolist()
    
    # Create plot
    plt.figure(figsize=(8, 6))
    
    # PPR plot
    plt.plot(pulse_numbers, ppr_before, marker='o', color='orange', linewidth=2, 
             label=f'Before (n={len(PCA_Data_Stability_Before)})')
    plt.fill_between(pulse_numbers, np.array(ppr_before) - np.array(ppr_before_sem),
                     np.array(ppr_before) + np.array(ppr_before_sem), color='orange', alpha=0.2)
    plt.plot(pulse_numbers, ppr_after, marker='s', color='brown', linewidth=2, 
             label=f'After (n={len(PCA_Data_Stability_After)})')
    plt.fill_between(pulse_numbers, np.array(ppr_after) - np.array(ppr_after_sem),
                     np.array(ppr_after) + np.array(ppr_after_sem), color='brown', alpha=0.2)
    plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
    plt.xlabel('Pulse Number')
    plt.ylabel('PPR (A_n/A_1)')
    plt.title('PPR Profile: Before vs After')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(pulse_numbers)
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "stability_ppr_profile.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Statistical testing for PPR
    print("=== PPR STATISTICAL ANALYSIS ===")
    for i, col in enumerate(ppr_cols, start=2):
        t_stat, p_val = ttest_rel(PCA_Data_Stability_Before[col], PCA_Data_Stability_After[col], nan_policy='omit')
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
        print(f"Pulse {i}: t={t_stat:.3f}, p={p_val:.3e} ({sig})")
    
    return output_file

plot_output = plot_stability_ppr_profile()
print(f"✓ Saved PPR profile to {plot_output}")


### I.9 Stability Parameter Comparisons

Detailed paired statistical tests are run for amplitudes, failure rates, and key ratios, with significance markers highlighting which parameters change reliably.


In [ ]:
# Detailed statistical comparisons for key parameters
def plot_stability_comparisons():
    """Create paired boxplots with statistical tests."""
    
    def add_significance_bar(ax, p_value, positions=[0, 1]):
        """Add significance bar above boxplot."""
        y_max = ax.get_ylim()[1]
        y_min = ax.get_ylim()[0]
        y_sig = y_max + 0.05 * (y_max - y_min)
        
        # Significance stars
        if p_value < 0.001:
            sig_text = '***'
        elif p_value < 0.01:
            sig_text = '**'
        elif p_value < 0.05:
            sig_text = '*'
        else:
            sig_text = 'ns'
        
        # Draw bar and text
        ax.plot(positions, [y_sig, y_sig], color='black', linewidth=1.5)
        ax.text(np.mean(positions), y_sig + 0.01 * (y_max - y_min), sig_text,
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # AMP1 comparison
    amp1_data = pd.DataFrame({
        'AMP1': pd.concat([PCA_Data_Stability_Before['AMP1'], PCA_Data_Stability_After['AMP1']]),
        'Condition': ['Before'] * len(PCA_Data_Stability_Before) + ['After'] * len(PCA_Data_Stability_After)
    })
    
    sns.boxplot(data=amp1_data, x='Condition', y='AMP1', ax=axes[0], palette=['orange', 'brown'])
    sns.stripplot(data=amp1_data, x='Condition', y='AMP1', ax=axes[0], color='black', size=3, alpha=0.6)
    
    # Add connecting lines for paired data
    n_pairs = min(len(PCA_Data_Stability_Before), len(PCA_Data_Stability_After))
    for i in range(n_pairs):
        axes[0].plot([0, 1], [PCA_Data_Stability_Before['AMP1'].iloc[i], PCA_Data_Stability_After['AMP1'].iloc[i]],
                     color='gray', alpha=0.4, linewidth=1)
    
    t_stat_amp1, p_val_amp1 = ttest_rel(PCA_Data_Stability_Before['AMP1'], PCA_Data_Stability_After['AMP1'])
    add_significance_bar(axes[0], p_val_amp1)
    axes[0].set_title('AMP1: Before vs After')
    
    # PPR2/1 comparison
    ppr_data = pd.DataFrame({
        'PPR2/1': pd.concat([PCA_Data_Stability_Before['PPR2/1'], PCA_Data_Stability_After['PPR2/1']]),
        'Condition': ['Before'] * len(PCA_Data_Stability_Before) + ['After'] * len(PCA_Data_Stability_After)
    })
    
    sns.boxplot(data=ppr_data, x='Condition', y='PPR2/1', ax=axes[1], palette=['orange', 'brown'])
    sns.stripplot(data=ppr_data, x='Condition', y='PPR2/1', ax=axes[1], color='black', size=3, alpha=0.6)
    
    for i in range(n_pairs):
        axes[1].plot([0, 1], [PCA_Data_Stability_Before['PPR2/1'].iloc[i], PCA_Data_Stability_After['PPR2/1'].iloc[i]],
                     color='gray', alpha=0.4, linewidth=1)
    
    t_stat_ppr, p_val_ppr = ttest_rel(PCA_Data_Stability_Before['PPR2/1'], PCA_Data_Stability_After['PPR2/1'])
    add_significance_bar(axes[1], p_val_ppr)
    axes[1].set_title('PPR2/1: Before vs After')
    
    # Baseline fluorescence (F0) analysis
    if 'stab_before_traces' in locals() and 'stab_after_traces' in locals():
        # Calculate baseline from first 0.5s of traces
        baseline_before = [np.mean(trace[:int(0.5 * len(COMMON_TIME))]) for trace in stab_before_traces]
        baseline_after = [np.mean(trace[:int(0.5 * len(COMMON_TIME))]) for trace in stab_after_traces]
        
        f0_data = pd.DataFrame({
            'F0': baseline_before + baseline_after,
            'Condition': ['Before'] * len(baseline_before) + ['After'] * len(baseline_after)
        })
        
        sns.boxplot(data=f0_data, x='Condition', y='F0', ax=axes[2], palette=['orange', 'brown'])
        sns.stripplot(data=f0_data, x='Condition', y='F0', ax=axes[2], color='black', size=3, alpha=0.6)
        
        # Connect paired points
        for i in range(min(len(baseline_before), len(baseline_after))):
            axes[2].plot([0, 1], [baseline_before[i], baseline_after[i]],
                         color='gray', alpha=0.4, linewidth=1)
        
        t_stat_f0, p_val_f0 = ttest_rel(baseline_before, baseline_after)
        add_significance_bar(axes[2], p_val_f0)
        axes[2].set_title('Baseline F0: Before vs After')
        axes[2].set_ylabel('F0 (ΔF/F)')
    else:
        axes[2].text(0.5, 0.5, 'F0 data\nnot available', ha='center', va='center', transform=axes[2].transAxes)
        axes[2].set_title('Baseline F0')
    
    plt.tight_layout()
    output_file = OUTPUT_DIR / "stability_statistical_comparisons.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print results
    print("=== STABILITY STATISTICAL RESULTS ===")
    print(f"AMP1: t={t_stat_amp1:.3f}, p={p_val_amp1:.4g}")
    print(f"PPR2/1: t={t_stat_ppr:.3f}, p={p_val_ppr:.4g}")
    
    return output_file

comparison_output = plot_stability_comparisons()
print(f"✓ Saved comparisons to {comparison_output}")

#### F Focus – Stability Amplitude and PPR Comparisons

Histograms and paired statistics highlight how Amp1 and PPR2/1 values evolve between the stability experiment's before and after states.


##### F Detail – Histogram Perspective

Viewing the paired distributions clarifies whether shifts arise from global scaling or from specific subpopulations of boutons.


### I.10 Stability Synthesis

A textual summary consolidates sample sizes, significant metrics, and interpretation from the stability analysis, ensuring the narrative captures the most impactful findings.


In [ ]:
# Final stability analysis summary
def stability_summary():
    """Generate comprehensive stability analysis summary."""
    
    print("=" * 60)
    print("COMPREHENSIVE STABILITY ANALYSIS SUMMARY")
    print("=" * 60)
    
    # Sample sizes
    n_before = len(PCA_Data_Stability_Before)
    n_after = len(PCA_Data_Stability_After)
    n_pairs = min(n_before, n_after)
    
    print(f"\nSample sizes:")
    print(f"  Before: {n_before} boutons")
    print(f"  After: {n_after} boutons")
    print(f"  Paired: {n_pairs} boutons")
    
    # Key parameter changes
    print(f"\nKey parameter changes (Before → After):")
    
    # AMP1
    amp1_before_mean = PCA_Data_Stability_Before['AMP1'].mean()
    amp1_after_mean = PCA_Data_Stability_After['AMP1'].mean()
    amp1_change = ((amp1_after_mean - amp1_before_mean) / amp1_before_mean) * 100
    print(f"  AMP1: {amp1_before_mean:.3f} → {amp1_after_mean:.3f} ({amp1_change:+.1f}%)")
    
    # PPR2/1
    ppr_before_mean = PCA_Data_Stability_Before['PPR2/1'].mean()
    ppr_after_mean = PCA_Data_Stability_After['PPR2/1'].mean()
    ppr_change = ((ppr_after_mean - ppr_before_mean) / ppr_before_mean) * 100
    print(f"  PPR2/1: {ppr_before_mean:.3f} → {ppr_after_mean:.3f} ({ppr_change:+.1f}%)")
    
    # PCA movement analysis (if available)
    if 'movement_distances' in locals():
        print(f"\nPCA space movement:")
        print(f"  Mean distance: {np.mean(movement_distances):.3f} ± {np.std(movement_distances):.3f}")
        print(f"  Max distance: {np.max(movement_distances):.3f}")
        print(f"  Boutons with large movement (>mean): {np.sum(movement_distances > np.mean(movement_distances))}/{len(movement_distances)}")
    
    # Cluster stability (if available)
    if 'stable_pairs' in locals() and 'unstable_pairs' in locals():
        total_analyzed = stable_pairs + unstable_pairs
        stability_pct = (stable_pairs / total_analyzed) * 100
        print(f"\nCluster assignment stability:")
        print(f"  Stable pairs: {stable_pairs}/{total_analyzed} ({stability_pct:.1f}%)")
        print(f"  Unstable pairs: {unstable_pairs}/{total_analyzed} ({100-stability_pct:.1f}%)")
    
    print(f"\n" + "=" * 60)
    print("Analysis complete - all figures saved to OUTPUT_DIR")
    print("=" * 60)

# Run summary
stability_summary()

## Chapter J - 50Hz experiments with low and high calcium

### J.1 50Hz Trajectories

In [ ]:
# Analyze theo concentration effects on bouton properties in PCA space

def plot_theo_50Hz_trajectories():
    """Plot how theo concentration changes affect PCA positioning at 50Hz."""
    
    plt.figure(figsize=(12, 8))
    
    # Background: WT pooled (standard condition)
    plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1], 
               c=get_cluster_colors(cluster_assignments), alpha=0.3, s=30, label='WT pooled')
    
    # Get projected data from pca_data dictionary
    theo_1_5_coords = pca_data['50Hz_1_5Ca']
    theo_2_5_coords = pca_data['50Hz_2_5Ca']
    theo_4_coords   = pca_data['50Hz_4Ca']

    # theo 1.5mM @ 50Hz - blue triangles pointing down
    plt.scatter(theo_1_5_coords[:, 0], theo_1_5_coords[:, 1], 
               marker='v', s=60, c='blue', alpha=0.8, edgecolors='darkblue', linewidth=0.5,
               label=f'Theo 1.5mM 50Hz (n={len(theo_1_5_coords)})')
    
    # theo 2.5mM @ 50Hz - orange diamonds
    plt.scatter(theo_2_5_coords[:, 0], theo_2_5_coords[:, 1], 
               marker='D', s=60, c='orange', alpha=0.8, edgecolors='darkorange', linewidth=0.5,
               label=f'Theo 2.5mM 50Hz (n={len(theo_2_5_coords)})')
    
    # theo 4mM @ 50Hz - red triangles pointing up  
    plt.scatter(theo_4_coords[:, 0], theo_4_coords[:, 1], 
               marker='^', s=60, c='red', alpha=0.8, edgecolors='darkred', linewidth=0.5,
               label=f'Theo 4mM 50Hz (n={len(theo_4_coords)})')
    
    # Calculate centroids
    center_pooled   = np.mean(pca_coordinates, axis=0)
    center_theo_1_5 = np.mean(theo_1_5_coords, axis=0)
    center_theo_2_5 = np.mean(theo_2_5_coords, axis=0)
    center_theo_4   = np.mean(theo_4_coords, axis=0)
    
    # Plot centroids
    plt.scatter(center_theo_1_5[0], center_theo_1_5[1], marker='X', s=200, c='blue', 
               edgecolor='black', linewidth=2, label='1.5mM centroid', zorder=5)
    plt.scatter(center_theo_2_5[0], center_theo_2_5[1], marker='X', s=200, c='orange', 
               edgecolor='black', linewidth=2, label='2.5mM centroid', zorder=5)
    plt.scatter(center_theo_4[0], center_theo_4[1], marker='X', s=200, c='red', 
               edgecolor='black', linewidth=2, label='4mM centroid', zorder=5)
    
    # Draw arrows from pooled centroid to theo condition centroids
    plt.arrow(center_pooled[0], center_pooled[1],
              center_theo_1_5[0] - center_pooled[0], center_theo_1_5[1] - center_pooled[1],
              color='blue', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.7)
    plt.arrow(center_pooled[0], center_pooled[1],
              center_theo_2_5[0] - center_pooled[0], center_theo_2_5[1] - center_pooled[1],
              color='orange', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.7)
    plt.arrow(center_pooled[0], center_pooled[1],
              center_theo_4[0] - center_pooled[0], center_theo_4[1] - center_pooled[1],
              color='red', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.7)
    
    # Annotate centroid coordinates
    plt.text(center_theo_1_5[0], center_theo_1_5[1]-0.3, 
             f'1.5mM\n({center_theo_1_5[0]:.2f},{center_theo_1_5[1]:.2f})', 
             color='blue', fontsize=9, ha='center', va='top', fontweight='bold')
    plt.text(center_theo_2_5[0], center_theo_2_5[1]+0.3, 
             f'2.5mM\n({center_theo_2_5[0]:.2f},{center_theo_2_5[1]:.2f})', 
             color='darkorange', fontsize=9, ha='center', va='bottom', fontweight='bold')
    plt.text(center_theo_4[0], center_theo_4[1]+0.3, 
             f'4mM\n({center_theo_4[0]:.2f},{center_theo_4[1]:.2f})', 
             color='red', fontsize=9, ha='center', va='bottom', fontweight='bold')
    
    # Connect paired boutons across concentrations (assuming matched order)
    n_pairs = min(len(theo_1_5_coords), len(theo_2_5_coords), len(theo_4_coords))
    if n_pairs > 0:
        for i in range(n_pairs):
            # Connect 1.5 → 2.5 → 4 mM
            plt.plot([theo_1_5_coords[i, 0], theo_2_5_coords[i, 0], theo_4_coords[i, 0]],
                     [theo_1_5_coords[i, 1], theo_2_5_coords[i, 1], theo_4_coords[i, 1]],
                     color='gray', alpha=0.3, linewidth=0.8, linestyle='--')
        print(f"Connected {n_pairs} bouton triplets across theo concentrations")
    
    # Format plot
    pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
    plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
    plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
    plt.title('theo Concentration Effects on Bouton Properties (50Hz)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "theo_50Hz_concentration_pca_trajectories.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return center_pooled, center_theo_1_5, center_theo_2_5, center_theo_4, n_pairs

def calculate_theo_50Hz_movements(center_pooled, center_theo_1_5, center_theo_2_5, center_theo_4, n_pairs):
    """Calculate movement statistics for theo concentration changes at 50Hz."""
    
    # Get coordinates
    theo_1_5_coords = pca_data['50Hz_1_5Ca']
    theo_2_5_coords = pca_data['50Hz_2_5Ca']
    theo_4_coords   = pca_data['50Hz_4Ca']

    # Distances from standard condition to each theo level
    dist_to_1_5 = np.linalg.norm(center_theo_1_5 - center_pooled)
    dist_to_2_5 = np.linalg.norm(center_theo_2_5 - center_pooled)
    dist_to_4   = np.linalg.norm(center_theo_4 - center_pooled)
    
    # Movement from standard to each theo concentration
    movements_to_1_5  = []
    n_1_5_comparisons = min(len(pca_coordinates), len(theo_1_5_coords))
    for i in range(n_1_5_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - theo_1_5_coords[i])
        movements_to_1_5.append(dist)
    
    movements_to_2_5  = []
    n_2_5_comparisons = min(len(pca_coordinates), len(theo_2_5_coords))
    for i in range(n_2_5_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - theo_2_5_coords[i])
        movements_to_2_5.append(dist)
    
    movements_to_4  = []
    n_4_comparisons = min(len(pca_coordinates), len(theo_4_coords))
    for i in range(n_4_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - theo_4_coords[i])
        movements_to_4.append(dist)
    
    # Movement between theo concentrations (paired boutons)
    movements_1_5_to_2_5 = []
    movements_2_5_to_4   = []
    movements_1_5_to_4   = []
    
    for i in range(n_pairs):
        dist_1_5_to_2_5 = np.linalg.norm(theo_1_5_coords[i] - theo_2_5_coords[i])
        dist_2_5_to_4   = np.linalg.norm(theo_2_5_coords[i] - theo_4_coords[i])
        dist_1_5_to_4   = np.linalg.norm(theo_1_5_coords[i] - theo_4_coords[i])
        
        movements_1_5_to_2_5.append(dist_1_5_to_2_5)
        movements_2_5_to_4.append(dist_2_5_to_4)
        movements_1_5_to_4.append(dist_1_5_to_4)
    
    return {
        'centroid_distances': {
            '1.5mM': dist_to_1_5,
            '2.5mM': dist_to_2_5,
            '4mM': dist_to_4
        },
        'individual_movements': {
            'to_1.5': movements_to_1_5,
            'to_2.5': movements_to_2_5,
            'to_4': movements_to_4,
            '1.5_to_2.5': movements_1_5_to_2_5,
            '2.5_to_4': movements_2_5_to_4,
            '1.5_to_4': movements_1_5_to_4
        }
    }

# Run analysis
center_pooled, center_theo_1_5, center_theo_2_5, center_theo_4, n_pairs = plot_theo_50Hz_trajectories()
movement_stats = calculate_theo_50Hz_movements(center_pooled, center_theo_1_5, center_theo_2_5, center_theo_4, n_pairs)

# Display results
print(f"\n=== theo CONCENTRATION ANALYSIS (50Hz) ===")
print(f"Centroid coordinates:")
print(f"  WT pooled:      ({center_pooled[0]:.3f}, {center_pooled[1]:.3f})")
print(f"  Theo 1.5mM:     ({center_theo_1_5[0]:.3f}, {center_theo_1_5[1]:.3f}) - distance: {movement_stats['centroid_distances']['1.5mM']:.3f}")
print(f"  Theo 2.5mM:     ({center_theo_2_5[0]:.3f}, {center_theo_2_5[1]:.3f}) - distance: {movement_stats['centroid_distances']['2.5mM']:.3f}")
print(f"  Theo 4mM:       ({center_theo_4[0]:.3f}, {center_theo_4[1]:.3f}) - distance: {movement_stats['centroid_distances']['4mM']:.3f}")

print(f"\nIndividual bouton movements from WT pooled:")
if movement_stats['individual_movements']['to_1.5']:
    moves = movement_stats['individual_movements']['to_1.5']
    print(f"WT → Theo 1.5mM (n={len(moves)}): {np.mean(moves):.3f} ± {np.std(moves):.3f}")

if movement_stats['individual_movements']['to_2.5']:
    moves = movement_stats['individual_movements']['to_2.5']
    print(f"WT → Theo 2.5mM (n={len(moves)}): {np.mean(moves):.3f} ± {np.std(moves):.3f}")

if movement_stats['individual_movements']['to_4']:
    moves = movement_stats['individual_movements']['to_4']
    print(f"WT → Theo 4mM (n={len(moves)}): {np.mean(moves):.3f} ± {np.std(moves):.3f}")

print(f"\nPairwise movements between theo concentrations:")
if movement_stats['individual_movements']['1.5_to_2.5']:
    moves = movement_stats['individual_movements']['1.5_to_2.5']
    print(f"1.5mM → 2.5mM (n={len(moves)}): {np.mean(moves):.3f} ± {np.std(moves):.3f}")

if movement_stats['individual_movements']['2.5_to_4']:
    moves = movement_stats['individual_movements']['2.5_to_4']
    print(f"2.5mM → 4mM (n={len(moves)}): {np.mean(moves):.3f} ± {np.std(moves):.3f}")

if movement_stats['individual_movements']['1.5_to_4']:
    moves = movement_stats['individual_movements']['1.5_to_4']
    print(f"1.5mM ↔ 4mM (n={len(moves)}): {np.mean(moves):.3f} ± {np.std(moves):.3f}")

print(f"\n✓ theo 50Hz trajectory analysis complete")
print(f"✓ Saved to {OUTPUT_DIR / 'theo_50Hz_concentration_pca_trajectories.pdf'}")

### J.2 Cluster Composition Comparison

In [ ]:
# Get WT cluster counts
wt_cluster_counts = pd.Series(cluster_assignments).value_counts().sort_index()

# Get 50Hz cluster assignments using kNN classifier
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn.fit(pca_coordinates, cluster_assignments)

# Project 50Hz conditions and assign to clusters
hz50_1_5_assignments = knn.predict(pca_data['50Hz_1_5Ca'])
hz50_2_5_assignments = knn.predict(pca_data['50Hz_2_5Ca'])
hz50_4_assignments   = knn.predict(pca_data['50Hz_4Ca'])

hz50_1_5_counts = pd.Series(hz50_1_5_assignments).value_counts()
hz50_2_5_counts = pd.Series(hz50_2_5_assignments).value_counts()
hz50_4_counts   = pd.Series(hz50_4_assignments).value_counts()

# Ensure all clusters represented
all_clusters = sorted(wt_cluster_counts.index)
hz50_1_5_complete = pd.Series([hz50_1_5_counts.get(c, 0) for c in all_clusters], index=all_clusters)
hz50_2_5_complete = pd.Series([hz50_2_5_counts.get(c, 0) for c in all_clusters], index=all_clusters)
hz50_4_complete  = pd.Series([hz50_4_counts.get(c, 0) for c in all_clusters], index=all_clusters)

# Calculate percentages
wt_percentages = 100 * wt_cluster_counts / len(cluster_assignments)
hz50_1_5_percentages = 100 * hz50_1_5_complete / len(hz50_1_5_assignments)
hz50_2_5_percentages = 100 * hz50_2_5_complete / len(hz50_2_5_assignments)
hz50_4_percentages   = 100 * hz50_4_complete / len(hz50_4_assignments)

# Stacked bar plot
fig, ax   = plt.subplots(figsize=(10, 6))
bar_width = 0.6

def get_text_color(rgb):
    r, g, b = rgb[:3]
    return 'white' if 0.2126*r + 0.7152*g + 0.0722*b < 0.55 else 'black'

# Prepare data for all conditions
conditions = [
    (0, wt_percentages, len(cluster_assignments), 'WT'),
    (1, hz50_1_5_percentages, len(hz50_1_5_assignments), '50Hz\n1.5mM Ca'),
    (2, hz50_2_5_percentages, len(hz50_2_5_assignments), '50Hz\n2.5mM Ca'),
    (3, hz50_4_percentages, len(hz50_4_assignments), '50Hz\n4mM Ca')
]

# Plot stacked bars for each condition
for pos, percentages, n_samples, label in conditions:
    bottom = 0
    for cluster_id in all_clusters:
        color = get_cluster_color(cluster_id)
        pct = percentages.iloc[cluster_id - 1]
        
        # Draw bar segment
        ax.bar(pos, pct, bar_width, bottom=bottom, color=color,
               alpha=0.7 if pos > 0 else 1.0,  # Make 50Hz conditions slightly transparent
               label=f'Cluster {cluster_id}' if pos == 0 else None)
        
        # Add percentage text if segment is large enough
        if pct > 3:
            ax.text(pos, bottom + pct/2, f"{pct:.1f}%", ha='center', va='center',
                    color=get_text_color(color), fontsize=8, fontweight='bold')
        
        bottom += pct
    
    # Add sample count above each bar
    ax.text(pos, 102, f"n={n_samples}", ha='center', va='bottom', fontweight='bold', fontsize=9)

# Format plot
ax.set_ylabel('Percentage (%)', fontsize=11)
ax.set_title('Cluster Distribution: WT vs 50Hz Conditions', fontsize=12, fontweight='bold')
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels([label for _, _, _, label in conditions])
ax.set_ylim(0, 115)
ax.grid(axis='y', alpha=0.3)

# Legend
handles = [plt.Rectangle((0,0),1,1, color=get_cluster_color(c)) for c in all_clusters]
ax.legend(handles, [f'Cluster {c}' for c in all_clusters],
          bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

plt.tight_layout()
output_file = OUTPUT_DIR / "cluster_distribution_all_conditions.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("=== CLUSTER DISTRIBUTION SUMMARY ===")
print(f"\n{'Condition':<15} {'n':<6} {'Top Cluster':<12} {'Distribution'}")
print("-" * 70)
for pos, percentages, n_samples, label in conditions:
    top_cluster = percentages.idxmax()
    top_pct = percentages.max()
    distribution = " ".join([f"C{i}:{percentages.iloc[i-1]:.1f}%" for i in all_clusters if percentages.iloc[i-1] > 5])
    print(f"{label.replace(chr(10), ' '):<15} {n_samples:<6} C{top_cluster} ({top_pct:.1f}%)   {distribution}")

print(f"\n✓ Saved cluster distribution to {output_file}")


### J.3 PPR profiles

In [ ]:
# Compare PPR profiles across WT and 50Hz conditions at different calcium concentrations
ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled.columns]
pulse_numbers = list(range(1, len(ppr_cols) + 2))

# Calculate PPR profiles for each condition
def get_ppr_profile(data, condition_name):
    means = [1.0] + data[ppr_cols].mean().tolist()
    sems = [0.0] + data[ppr_cols].sem().tolist()
    return means, sems

# Get profiles for each condition
means_wt, sems_wt = get_ppr_profile(PCA_Data_WT_Pooled, 'WT 2.5mM Ca')
means_50hz_1_5, sems_50hz_1_5 = get_ppr_profile(PCA_Data_50Hz_1_5_Ca, '50Hz 1.5mM Ca')  # Fixed: added underscore
means_50hz_2_5, sems_50hz_2_5 = get_ppr_profile(PCA_Data_50Hz_2_5_Ca, '50Hz 2.5mM Ca')  # Fixed: added underscore
means_50hz_4, sems_50hz_4 = get_ppr_profile(PCA_Data_50Hz_4_Ca, '50Hz 4mM Ca')  # Fixed: added underscore


# Plot PPR profiles
plt.figure(figsize=(10, 6))

# WT pooled (black - reference)
plt.plot(pulse_numbers, means_wt, marker='o', color='black', linewidth=2,
         label=f'WT 2.5mM Ca (n={len(PCA_Data_WT_Pooled)})')
plt.fill_between(pulse_numbers, np.array(means_wt) - np.array(sems_wt),
                 np.array(means_wt) + np.array(sems_wt), color='black', alpha=0.15)

# 50Hz 1.5mM Ca (blue)
plt.plot(pulse_numbers, means_50hz_1_5, marker='v', color='blue', linewidth=2,
         label=f'50Hz 1.5mM Ca (n={len(PCA_Data_50Hz_1_5_Ca)})')
plt.fill_between(pulse_numbers, np.array(means_50hz_1_5) - np.array(sems_50hz_1_5),
                 np.array(means_50hz_1_5) + np.array(sems_50hz_1_5), color='blue', alpha=0.2)

# 50Hz 2.5mM Ca (orange)
plt.plot(pulse_numbers, means_50hz_2_5, marker='D', color='orange', linewidth=2,
         label=f'50Hz 2.5mM Ca (n={len(PCA_Data_50Hz_2_5_Ca)})')
plt.fill_between(pulse_numbers, np.array(means_50hz_2_5) - np.array(sems_50hz_2_5),
                 np.array(means_50hz_2_5) + np.array(sems_50hz_2_5), color='orange', alpha=0.2)

# 50Hz 4mM Ca (red)
plt.plot(pulse_numbers, means_50hz_4, marker='^', color='red', linewidth=2,
         label=f'50Hz 4mM Ca (n={len(PCA_Data_50Hz_4_Ca)})')
plt.fill_between(pulse_numbers, np.array(means_50hz_4) - np.array(sems_50hz_4),
                 np.array(means_50hz_4) + np.array(sems_50hz_4), color='red', alpha=0.2)

# Format plot
plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
plt.xlabel('Pulse Number', fontsize=11)
plt.ylim(0.9,3.5)
plt.ylabel('PPR (A_n/A_1)', fontsize=11)
plt.title('PPR Profiles: WT vs 50Hz Conditions at Different Calcium Concentrations', fontsize=12, fontweight='bold')
plt.xticks(pulse_numbers)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save figure and data
output_fig = OUTPUT_DIR / "ppr_profiles_wt_vs_50hz_all_calcium.pdf"
plt.savefig(output_fig, dpi=300, bbox_inches='tight')
plt.show()

# Save numerical data
output_data = OUTPUT_DIR / "ppr_profiles_wt_vs_50hz_data.txt"
with open(output_data, "w") as f:
    f.write("Pulse\tWT_Mean\tWT_SEM\t50Hz_1.5mM_Mean\t50Hz_1.5mM_SEM\t50Hz_2.5mM_Mean\t50Hz_2.5mM_SEM\t50Hz_4mM_Mean\t50Hz_4mM_SEM\n")
    for i, pulse in enumerate(pulse_numbers):
        f.write(f"{pulse}\t{means_wt[i]:.4f}\t{sems_wt[i]:.4f}\t"
                f"{means_50hz_1_5[i]:.4f}\t{sems_50hz_1_5[i]:.4f}\t"
                f"{means_50hz_2_5[i]:.4f}\t{sems_50hz_2_5[i]:.4f}\t"
                f"{means_50hz_4[i]:.4f}\t{sems_50hz_4[i]:.4f}\n")

# Print summary statistics
print("=== PPR PROFILE COMPARISON: WT vs 50Hz CONDITIONS ===")
print(f"\nWT 2.5mM Ca (n={len(PCA_Data_WT_Pooled)}):")
print(f"  PPR2/1: {means_wt[1]:.3f} ± {sems_wt[1]:.3f}")
print(f"  PPR10/1: {means_wt[-1]:.3f} ± {sems_wt[-1]:.3f}")

print(f"\n50Hz 1.5mM Ca (n={len(PCA_Data_50Hz_1_5_Ca)}):")
print(f"  PPR2/1: {means_50hz_1_5[1]:.3f} ± {sems_50hz_1_5[1]:.3f}")
print(f"  PPR10/1: {means_50hz_1_5[-1]:.3f} ± {sems_50hz_1_5[-1]:.3f}")

print(f"\n50Hz 2.5mM Ca (n={len(PCA_Data_50Hz_2_5_Ca)}):")
print(f"  PPR2/1: {means_50hz_2_5[1]:.3f} ± {sems_50hz_2_5[1]:.3f}")
print(f"  PPR10/1: {means_50hz_2_5[-1]:.3f} ± {sems_50hz_2_5[-1]:.3f}")

print(f"\n50Hz 4mM Ca (n={len(PCA_Data_50Hz_4_Ca)}):")
print(f"  PPR2/1: {means_50hz_4[1]:.3f} ± {sems_50hz_4[1]:.3f}")
print(f"  PPR10/1: {means_50hz_4[-1]:.3f} ± {sems_50hz_4[-1]:.3f}")

print(f"\n✓ Saved PPR profiles to {output_fig}")
print(f"✓ Saved numerical data to {output_data}")

### J.4 50Hz traces

In [ ]:
# Plot mean traces for all three 50Hz conditions on the same graph

# Extract traces for each 50Hz condition
hz50_1_5_traces = []
hz50_2_5_traces = []
hz50_4_traces = []

for _, row in NORM_TRACES_DATAFRAME.iterrows():
    if row['Condition'] == 'Theo_1_5_50Hz':
        hz50_1_5_traces.append(row['Avg'])
    elif row['Condition'] == 'Theo_2_5_50Hz':
        hz50_2_5_traces.append(row['Avg'])
    elif row['Condition'] == 'Theo_4_50Hz':
        hz50_4_traces.append(row['Avg'])

# Check if we have traces
if not (hz50_1_5_traces and hz50_2_5_traces and hz50_4_traces):
    print("50Hz trace conditions not found in resampled data")
    available_conditions = NORM_TRACES_DATAFRAME['Condition'].unique()
    print(f"Available conditions: {list(available_conditions)}")
else:
    # Calculate means and SEMs for each condition
    hz50_1_5_mean = np.nanmean(hz50_1_5_traces, axis=0)
    hz50_1_5_sem = np.nanstd(hz50_1_5_traces, axis=0, ddof=1) / np.sqrt(len(hz50_1_5_traces))
    
    hz50_2_5_mean = np.nanmean(hz50_2_5_traces, axis=0)
    hz50_2_5_sem = np.nanstd(hz50_2_5_traces, axis=0, ddof=1) / np.sqrt(len(hz50_2_5_traces))
    
    hz50_4_mean = np.nanmean(hz50_4_traces, axis=0)
    hz50_4_sem = np.nanstd(hz50_4_traces, axis=0, ddof=1) / np.sqrt(len(hz50_4_traces))
    
    # Create plot with all three conditions
    plt.figure(figsize=(12, 6))
    
    # 1.5mM Ca (blue)
    plt.plot(COMMON_TIME, hz50_1_5_mean, color='blue', linewidth=2, 
             label=f'50Hz 1.5mM Ca (n={len(hz50_1_5_traces)})')
    plt.fill_between(COMMON_TIME, hz50_1_5_mean - hz50_1_5_sem, 
                     hz50_1_5_mean + hz50_1_5_sem, color='blue', alpha=0.25)
    
    # 2.5mM Ca (orange)
    plt.plot(COMMON_TIME, hz50_2_5_mean, color='orange', linewidth=2,
             label=f'50Hz 2.5mM Ca (n={len(hz50_2_5_traces)})')
    plt.fill_between(COMMON_TIME, hz50_2_5_mean - hz50_2_5_sem, 
                     hz50_2_5_mean + hz50_2_5_sem, color='orange', alpha=0.25)
    
    # 4mM Ca (red)
    plt.plot(COMMON_TIME, hz50_4_mean, color='red', linewidth=2,
             label=f'50Hz 4mM Ca (n={len(hz50_4_traces)})')
    plt.fill_between(COMMON_TIME, hz50_4_mean - hz50_4_sem, 
                     hz50_4_mean + hz50_4_sem, color='red', alpha=0.25)
    
    # Add stimulus markers (10 pulses at 50Hz = every 20ms starting at 1.0s)
    stim_times = [1.0 + 0.02*i for i in range(10)]  # 50Hz = 20ms intervals
    for stim_time in stim_times:
        if stim_time <= 2.0:
            plt.axvline(stim_time, color='gray', linestyle='--', alpha=0.4, linewidth=1)
    
    # Formatting
    plt.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    plt.xlim(0.5, 2.0)
    plt.xlabel('Time (s)', fontsize=11)
    plt.ylabel('ΔF/F', fontsize=11)
    plt.xlim(0.998,1.4)
    plt.title('Mean Traces: 50Hz Stimulation at Different Calcium Concentrations', 
              fontsize=12, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "50hz_mean_traces_all_calcium.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print summary statistics
    print(f"=== 50Hz MEAN TRACE COMPARISON ===")
    print(f"1.5mM Ca: n={len(hz50_1_5_traces)} traces")
    print(f"  Peak response: {hz50_1_5_mean.max():.3f} ΔF/F at t={COMMON_TIME[np.argmax(hz50_1_5_mean)]:.2f}s")
    
    print(f"\n2.5mM Ca: n={len(hz50_2_5_traces)} traces")
    print(f"  Peak response: {hz50_2_5_mean.max():.3f} ΔF/F at t={COMMON_TIME[np.argmax(hz50_2_5_mean)]:.2f}s")
    
    print(f"\n4mM Ca: n={len(hz50_4_traces)} traces")
    print(f"  Peak response: {hz50_4_mean.max():.3f} ΔF/F at t={COMMON_TIME[np.argmax(hz50_4_mean)]:.2f}s")
    
    print(f"\n✓ Saved mean traces to {output_file}")